# DA3METRIC-LARGE — **Final Self-Supervised Fine-Tuning** on KITTI Eigen-Split  ·  6 selected recipes

> **Derived from** `DA3_SelfSupervised_FT_Ablation_HPT_v6_2_1_stationary_fix.ipynb` — the *only* implementation source of truth.
> This is a **training** notebook, not an ablation / HPT notebook: the ablation groups, LR / architecture / augmentation / optimizer / weight sweeps, the stationary-object losses, the hybrid-LiDAR recipes and the supervised control are **removed**. Every retained function is **copied from the source** (extracted programmatically, not retyped); every deliberate deviation is listed in the *Source-fidelity ledger* below.

## Scope

Six independent self-supervised fine-tunings of the pretrained DA3METRIC-LARGE:

| Group | Recipes | Pose | Notes |
|---|---|---|---|
| **A** — OXTS-pose SSL | `A1_gt_pose_photo`, `A3_gt_pose_photo_smooth_temporal` | OXTS (GT camera-frame relative pose) | *self-supervised w.r.t. depth*: OXTS supplies camera motion only |
| **B** — OXTS-pose SSL | `B1_gt_pose_rigid_flow` | OXTS | idem |
| **C** — predicted-pose pure SSL | `C1_rigid_flow`, `C3_photo_l1_flow_smooth`, `C5_photo_l1_feature_metric` | PoseNet (learned) | no OXTS, no LiDAR |

**No LiDAR depth loss, sparse-depth anchor or GT-depth target is used in any recipe.**
*Validated in the ablation:* A1, B1, C1. *New combinations proposed from the ablation findings (not directly validated):* A3, C3, C5.

## Section map

`1` Configuration · `2` Imports + provenance hash · `3` Reused utilities (SAM3 / dynamic mask, metrics, test evaluation, training / validation loops) · `4` Dataset · `5` OXTS utilities + canonical sample set + loaders · `6` Model, PoseNet, fp32 warping, pristine DA3 state · `7` Losses · `8` Final recipe registry · `9` Run-level helpers, PoseNet warm-up, per-recipe trial · `10` Pre-flight · `11` Runner + per-recipe training · `12` Validation · `13` Test evaluation · `14` Final table · `15` Plots · `16` Summary.

## Why the v6 fixes matter (retained from the source — not to be reverted)

| Topic | Fix that is preserved here |
|---|---|
| Silent supervised fallback | Every SSL loss used to fall back to `|pred − gt|` when pose was missing. Now missing input ⇒ `SSLInputError`; pure-SSL losses are called with `gt=None, mask=None`. |
| Zero depth-gradient at identity pose | PoseNet is not zero-initialised (output scale 0.1), pre-trained with depth frozen (`POSE_WARMUP_EPOCHS`), with its own LR. |
| Scheduler length | Sized by optimizer updates, `ceil(len(loader)/ACCUM_STEPS)`, not by batches; the final partial accumulation window is flushed and divided by its true size. |
| fp16 geometry overflow | Backprojection / projection / warping / all loss maths in fp32 (`K·X ≈ 1e5 > 65 504` overflowed in fp16 → NaN → every update skipped). Non-finite updates are counted; a run that applies zero updates in epoch 1 aborts. |
| OXTS frame | `T_cam = T_ic·inv(W_j)·W_i·inv(T_ic)` (conjugation). The IMU-axes product cancelled `T_ic`. Missing calibration ⇒ sample dropped. |
| Horizontal flip | `cx → W−1−cx`, pose `T → F·T·F`, SAM3 detections mirrored; drawn once per sample and applied to all frames. |
| Dynamic-object mask | SAM3 class prior of the **current** frame only (the t+1 mask lives in other coordinates). Used by A3's temporal term. |
| Identical populations | One canonical sample list (complete OXTS pose + calibration) and one train/val split shared by all six recipes. |

**Scale caveat.** Monocular photometric self-supervision constrains depth and translation only jointly, so predicted-pose recipes (C) can drift in metric scale even from a metric initialisation; OXTS recipes (A/B) remove that ambiguity through metric translation. For this reason **unscaled** test metrics are primary; median / mean / affine-aligned metrics are diagnostics.

## Isolation & test hygiene

* Output goes to a **new** directory (`final_self_supervised_FT`); an assertion rejects any path containing `ablation_study`. Checkpoints, caches and summary rows carry an experiment hash and are only resumed when hash **and** recipe id match.
* Every recipe re-loads the same pretrained state (never `A1 → A3 → …`) and has its own checkpoint directory.
* Model selection / early stopping use the **validation** split only; the test set is evaluated once per recipe after training. (Validation RMSE is measured against sparse LiDAR as *evaluation* — never inside a loss.)

## Run order
Set the **Configuration** cell (paths, `FINAL_EPOCHS`, recipe switches / `SELECTED_RECIPES`) → run top to bottom. Section 10 (pre-flight) must pass before Section 11 trains anything.

## Source-fidelity ledger

**Copied verbatim** (byte-identical text slices of the source): imports, AMP dtype block, experiment provenance block, SAM3 loading + trackers, dynamic filter + self-test, metrics, test evaluation, dataset, OXTS utilities + OXTS dataset, model wrapper / PoseNet / warp, device + DA3 load + pristine state + load_da3_model_fast, loss classes (19), _silog (referenced only by the unreachable anchor branch), _GTCombo (Group 5b), LossSpec, _unified_forward, check_gt_pose_convention (0a), pose_scale_check, plot palette / _save / _grp_color.
*(also verbatim: the SAM3 loader / trackers (cell 11), the dynamic filter (13), the model / PoseNet / warp cell (15), metrics (27), test evaluation (31), dataset (7), OXTS (9), training / validation loops (29, one token renamed), the runner helpers of cell 33, and the data-loader cell 35 with the edits below).*

**Deliberate deviations from the source** (each was applied through an assertion that the replaced text occurs exactly the expected number of times):

| Where | Change | Why |
|---|---|---|
| 2b provenance | `"MAX_TEST", "STRICT_")` -> `"MAX_TEST", "STRICT_", "SELECTED", "PREFLIGHT")` | recipe subset selection / pre-flight switches must not change the experiment hash |
| 2b provenance | `EXPERIMENT_CODE_VERSION = "v6.2.1-stationary-fix"       # bumped: track_and_filter_pair co` -> `EXPERIMENT_CODE_VERSION = "final-v1 (derived from v6.2.1-stationary-fix)"   # differs from` | new code-version tag => new EXPERIMENT_HASH (through the source's unchanged hashing mechanism) |
| 3.3 batch masks | cell truncated before `# v6.2.1 — Group 1c REAL-DATA stationary-pair coverage diagnostic` | dropped `stationary_pair_coverage_report`, `print_stationary_coverage_report`, `compare_filter_modes`, `_self_test_coordinate_fix` (stationary-object ablation only) |
| 3.6 build_scheduler | `ABLATION_EPOCHS` -> `FINAL_EPOCHS` | the old ablation budget no longer exists; the scheduler is sized by FINAL_EPOCHS (all three occurrences, inside build_scheduler) |
| 5b loaders | `if CANONICAL_SAMPLE_SET and (RUN_ORACLE_POSE_ABLATION or RUN_SANITY_EXPERIMENTS):` -> `if CANONICAL_SAMPLE_SET:` | the ablation-only switches no longer exist; the canonical set is always probed so that a subset run uses the same population as a full run |
| 5b loaders | `from TEST_ROOT, never seen during HPT)` -> `from TEST_ROOT, never used for training / selection)` | log wording only |
| 5b loaders | removed `get_completed_ablations()` block (ablation SUMMARY_CSV resume) | replaced by `get_completed_recipes()` (Section 11) which reads the FINAL summary CSV |
| 7c spec_for | `LOSS_SPECS[loss_name]` -> `FINAL_LOSS_SPECS[loss_name]` | the ablation `LOSS_SPECS` table does not exist here; the final table is `FINAL_LOSS_SPECS` (Section 8) |
| 9 trial | `def _train_one_hpt_trial(` -> `def _train_one_final_run(` | renamed (no HPT sweep exists in this notebook) |
| 9 trial | `custom_criterion: Optional[nn.Module] = None,     spec:           Optional["LossSpec"] = N` -> `custom_criterion: Optional[nn.Module] = None,     spec:           Optional["LossSpec"] = N` | per-recipe checkpoint directory + recipe metadata are passed in |
| 9 trial | `criterion = copy.deepcopy(LOSS_REGISTRY[loss_name]).to(device)` -> `criterion = FINAL_RECIPES[loss_name].build_criterion().to(device)` | ablation LOSS_REGISTRY does not exist; (branch is unused: a custom_criterion is always supplied) |
| 9 trial | `ckpt = _ckpt_path(ablation_label, tag, thash)     latest_ckpt = ckpt.replace("temp_best_",` -> `assert ckpt_dir is not None, "each recipe needs its own isolated checkpoint directory"    ` | isolated per-recipe checkpoint files instead of shared temp_* files |
| 9 trial | `if checkpoint.get("trial_hash") != thash or checkpoint.get("experiment_hash") != EXPERIMEN` -> `if (checkpoint.get("trial_hash") != thash or checkpoint.get("experiment_hash") != EXPERIME` | a checkpoint of another recipe is never resumed (trial hash already differs; explicit check added) |
| 9 trial | `for epoch in range(start_epoch, ABLATION_EPOCHS + 1):` -> `for epoch in range(start_epoch, FINAL_EPOCHS + 1):` | ABLATION_EPOCHS -> FINAL_EPOCHS |
| 9 trial | `epoch, ABLATION_EPOCHS, lr,` -> `epoch, FINAL_EPOCHS, lr,` | ABLATION_EPOCHS -> FINAL_EPOCHS (log line) |
| 9 trial | `[HPT trial] ablation=%s` -> `[final run] recipe=%s` | log wording only |
| 9 trial | `rec = {"config": ablation_label, "hpt_tag": tag, "epoch": epoch, "lr": lr,` -> `rec = {"config": ablation_label, "hpt_tag": tag, "epoch": epoch, "lr": lr,                ` | additive history columns (end-of-epoch scheduled LR, epoch wall-time) for the LR curve / training-time report |
| 9 trial | `"loss_name": loss_name, "arch_mode": arch_mode}` -> `"loss_name": loss_name, "arch_mode": arch_mode, **(ckpt_meta or {})}` | checkpoint carries recipe / weights / pose-source / LR / aug / hash metadata |
| 9 trial | `torch.save(_blob(False), ckpt.replace("temp_best_", f"temp_ep{epoch:03d}_"))` -> `torch.save(_blob(False), os.path.join(ckpt_dir, f"epoch_{epoch:03d}.pt"))` | with the new file naming the old `.replace('temp_best_', ...)` would have overwritten best_model.pt |
| 10 run_sanity_experiment | `spec = LOSS_SPECS[loss_name]` -> `spec = FINAL_LOSS_SPECS[loss_name]` | look the recipe up in the FINAL table (loss_name == recipe_id) |
| 10 run_sanity_experiment | `crit = copy.deepcopy(LOSS_REGISTRY[loss_name]).to(device)` -> `crit = FINAL_RECIPES[loss_name].build_criterion().to(device)` | build the recipe's criterion from the FINAL registry |
| 15 plot_rmse_all_scales | `SSL Ablation Configurations` -> `Final SSL Fine-Tuning (unscaled = primary)` | title wording only (function body unchanged) |
| 15 plot_error_metrics | `Error Metrics (Unscaled) — SSL Ablation` -> `Error Metrics (Unscaled) — Final SSL Fine-Tuning` | title wording only (function body unchanged) |
| 15 plot_delta_accuracy | `Threshold Accuracy (Unscaled) — SSL Ablation` -> `Threshold Accuracy (Unscaled) — Final SSL Fine-Tuning` | title wording only (function body unchanged) |
| 15 plot_bias | `Depth Bias Across Scale Variants — SSL Ablation` -> `Depth Bias Across Scale Variants — Final SSL Fine-Tuning` | title wording only (function body unchanged) |

**New code** (none of it re-implements source functionality): the configuration cell, `_SSLCombo` (C3 / C5 composer, built only from source losses), `FinalRecipe` + registry + safety assertions, the final pre-flight wrappers (`final_gradient_smoke_test`, `check_recipe_compositions`, `run_final_preflight`), `get_completed_recipes`, `run_final_recipe`, and the reporting / plotting cells (12–16).

## 1 · Configuration

Everything that is meant to be changed lives in this one cell. **No training function has to be edited.**

| Block | What you can change |
|---|---|
| Paths | data roots + a **new** output directory (never an ablation directory) |
| Recipes | `RUN_A1 … RUN_C5` switches, or `SELECTED_RECIPES = [...]` (overrides the switches) |
| Budget | `FINAL_EPOCHS` (default **50**), `BATCH_SIZE`, `ACCUM_STEPS`, `MAX_TRAIN` (0 = all samples) |
| Fixed by the ablation | `LEARNING_RATE=1e-6`, `WEIGHT_DECAY=1e-4`, AdamW, cosine warm-up, `decoder_only`, `base` augmentation — **not swept here** |
| Early stopping | `EARLY_STOP_PATIENCE` (default 10) — deliberately **not** tied to the old 5-epoch ablation budget |
| Inherited source constants | dataset / SAM3 / PoseNet / geometry constants that the copied source code reads — leave unchanged |

`RECIPE_WEIGHTS` is shown for transparency. The registry (Section 8) **asserts** that it equals the recipe definitions, so an accidental edit fails loudly instead of silently changing a recipe.

In [1]:
# =============================================================================
# FINAL DA3METRIC-LARGE SELF-SUPERVISED FINE-TUNING — CONFIGURATION
# (pure Python: no torch import here, so CUDA_VISIBLE_DEVICES can still be set)
# =============================================================================
import os

# Source default was CUDA_VISIBLE_DEVICES="1" (hard-coded in the source's first cell).  Set None to leave untouched.
GPU_INDEX = "1"
if GPU_INDEX is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX

# ─── PATHS ───────────────────────────────────────────────────────────────────
KITTI_ROOT      = "/home/arka/Downloads/MMDE/datasets/KITTI/complete_oxts"
TRAIN_ROOT      = os.path.join(KITTI_ROOT, "train")
TEST_ROOT       = os.path.join(KITTI_ROOT, "test")
CACHE_DIR       = "/home/arka/Downloads/MMDE/cache"

# NEW dedicated directory for the final fine-tuning.  Never an ablation directory: nothing here can resume /
# overwrite an ablation checkpoint (and every checkpoint / cache also carries an experiment hash, see 2b).
OUTPUT_BASE     = "/home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/final_self_supervised_FT"
assert "ablation_study" not in OUTPUT_BASE, "OUTPUT_BASE must be a NEW directory, not an ablation-study directory"
CHECKPOINT_DIR  = os.path.join(OUTPUT_BASE, "checkpoints")       # <CHECKPOINT_DIR>/<recipe_id>/{best_model.pt, latest_resume.pt}
RESULTS_DIR     = os.path.join(OUTPUT_BASE, "results")
PLOT_DIR        = os.path.join(OUTPUT_BASE, "plots")
ANALYSIS_DIR    = os.path.join(OUTPUT_BASE, "analysis")
SUMMARY_CSV     = os.path.join(RESULTS_DIR, "final_summary.csv")
TEMP_CKPT_DIR   = os.path.join(OUTPUT_BASE, "temp_ckpts")        # warm-up / baseline caches (hash-namespaced in 2b)
for d in [OUTPUT_BASE, CHECKPOINT_DIR, RESULTS_DIR, PLOT_DIR, ANALYSIS_DIR, TEMP_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── MODEL (semantics identical to the source) ────────────────────────────────
MODEL_ID        = "depth-anything/DA3METRIC-LARGE"
ORTHO_NORM      = 300.0
FOCAL_LENGTH_X  = 721.5377         # KITTI default: avg(fx, fy) ≈ 721.5 px  (forward_batch uses avg(fx, fy) from K)
USE_MODEL_FOCAL = False            # DA3METRIC-LARGE doesn't have intrinsic prediction

# ─── WHICH RECIPES TO RUN ─────────────────────────────────────────────────────
#  A = OXTS/GT-pose self-supervised (photometric family)   B = OXTS/GT-pose rigid flow   C = predicted-pose (PoseNet) pure SSL
#  All six use NO LiDAR / GT depth in any loss.  OXTS supplies only the camera motion.
RUN_A1 = False     # A1_gt_pose_photo                       (validated in ablation: Group 5b "H")
RUN_A3 = False     # A3_gt_pose_photo_smooth_temporal       (NEW combination, pure SSL — no LiDAR anchor)
RUN_B1 = True     # B1_gt_pose_rigid_flow                  (validated in ablation: Group 5b "K")
RUN_C1 = False     # C1_rigid_flow                          (validated in ablation: Group 1a `rigid_flow_consistency`)
RUN_C3 = False     # C3_photo_l1_flow_smooth                (NEW combination)
RUN_C5 = False     # C5_photo_l1_feature_metric             (NEW combination)
SELECTED_RECIPES = None   # e.g. ["A1_gt_pose_photo", "C1_rigid_flow"]  -> runs ONLY these (overrides the RUN_* switches)

# ─── FINAL TRAINING BUDGET ────────────────────────────────────────────────────
FINAL_EPOCHS    = 50               # <- change to 60 / 80 / 100 ... freely (scheduler + early-stop logic follow it)
BATCH_SIZE      = 8                # from 8
ACCUM_STEPS     = 4                # from 4 # effective batch = BATCH_SIZE * ACCUM_STEPS = 32 # from 16
MAX_TRAIN       = 0                # 0 = ALL available samples (source convention)
MAX_TEST        = 0                # 0 = all test frames

# ─── FIXED FINAL HYPER-PARAMETERS (decided after the ablation / HPT stage — NOT swept here) ──────────
LEARNING_RATE   = 1e-6             # depth LR
WEIGHT_DECAY    = 1e-4
ARCH_MODE       = "decoder_only"   # fixed architecture
AUG_MODE        = "base"           # fixed augmentation (flip + colour jitter; K / pose mirrored on flip)
OPTIM_MODE      = "adamw"
SCHED_MODE      = "cosine_warmup"
WARMUP_EPOCHS   = 1                # cosine-warm-up length, in epochs (scheduler is stepped per OPTIMIZER UPDATE)
MAX_GRAD_NORM   = 1.0
MIXED_PREC      = True
NUM_WORKERS     = 8
gpu_duty_cycle  = 1.0              # 1.0 = no throttling
"""
    gpu_duty_cycle: fraction of wall-time the GPU is actively computing (0.5 -> ~50 % utilisation).
"""

# ─── EARLY STOPPING / VALIDATION / CHECKPOINTS ────────────────────────────────
EARLY_STOP_PATIENCE  = 10          # NOT tied to the old ablation budget (source: ABLATION_EPOCHS - 1) # from 10
VAL_SPLIT_FRAC       = 0.10
VAL_SPLIT_SEED       = 42
SAVE_EVERY_N_EPOCHS  = 2           # from 0 = only best_model.pt + latest_resume.pt per recipe

# ─── DATASET (inherited) ──────────────────────────────────────────────────────
TARGET_H        = 378              # multiple of 14
TARGET_W        = 1246             # multiple of 14
MIN_DEPTH       = 1e-3
MAX_DEPTH       = 80.0
SSL_MIN_DEPTH   = 1e-3
SSL_MAX_DEPTH   = 80.0
SOURCE_OFFSETS  = (-1, +1)         # 3-frame training; +1 MUST be present (temporal / flow recipes use t+1)
CANONICAL_SAMPLE_SET = True        # ALL six recipes use exactly the same samples (those with complete OXTS pose + calibration)

# ─── SSL-SPECIFIC HYPER-PARAMETERS (inherited; read by the copied loss code) ──────────────────────
EPS                             = 1e-6
SSIM_ALPHA                      = 0.85
SMOOTHNESS_DECAY                = 1.0
MIN_REPROJ_AUTO_MASK_THRESHOLD  = 0.0       # 0 -> strict auto-mask (Monodepth2 default)

# ─── FINAL RECIPE WEIGHTS (part of the DEFINITION of each recipe — asserted in Section 8, do not tune) ──────
RECIPE_WEIGHTS = {
    "A1_gt_pose_photo":                 {"gt_photo": 1.0},
    "A3_gt_pose_photo_smooth_temporal": {"gt_photo": 1.0, "smooth": 1e-3, "gt_temporal": 0.1},
    "B1_gt_pose_rigid_flow":            {"gt_flow": 1.0},
    "C1_rigid_flow":                    {"rigid_flow_consistency": 1.0},
    "C3_photo_l1_flow_smooth":          {"photo_l1": 1.0, "flow_smooth": 0.1},
    "C5_photo_l1_feature_metric":       {"photo_l1": 1.0, "feature_metric": 0.1},
}
W_ANCHOR = 0.0     # LiDAR / SILog anchor weight: ALWAYS 0 here (Group 5b J's 0.5 anchor is deliberately NOT inherited)

# ─── POSE NETWORK (used by C1 / C3 / C5 only; inherited exactly) ──────────────────────────────────
POSE_LR                 = 1e-4
POSE_OUT_SCALE          = 0.1
POSE_WARMUP_EPOCHS      = 1          # PoseNet pre-training with depth FROZEN (photometric loss only, no GT depth)
POSE_WARMUP_MAX_BATCHES = 0          # 0 = whole epoch
POSE_WARMUP_CODE_VERSION = "v6.2"    # kept from the source (part of the warm-up cache key)

# ─── MIXED PRECISION (geometry + all loss maths ALWAYS run in fp32 — see Section 4) ───────────────
AMP_DTYPE_MODE = "fp16"              # "fp16" (GradScaler) | "bf16" | "auto"

# ─── PRE-FLIGHT (Section 9) ───────────────────────────────────────────────────
RUN_SANITY_EXPERIMENTS   = True        # True      # compact final pre-flight; training refuses to start if it fails
REQUIRE_SANITY_PASS      = True        # True
RUN_PREFLIGHT_SHORT_TRAINING = True    # True  # short real training run per selected recipe (proves the depth net is updated)
PREFLIGHT_AUG_MODE       = "base"       # same loaders as the final training -> shares the PoseNet warm-up cache
SANITY_TRAIN_BATCHES     = 120          # => 120 / ACCUM_STEPS optimizer updates per recipe
SANITY_VAL_BATCHES       = 16
SANITY_MAX_SKIP_FRAC     = 0.25
SANITY_LR                = LEARNING_RATE   # the pre-flight probes the SAME depth LR the final run uses

# ─── VALIDITY SWITCHES (inherited) ────────────────────────────────────────────
USE_DYNAMIC_MASK         = True      # SAM3 class-prior mask of the CURRENT frame excludes potentially-moving objects from
                                     # the temporal depth-consistency term (only A3 uses it)
DYN_MASK_DILATE_PX       = 3
STRICT_ARCH_CHECK        = True
EXPECTED_TRAINABLE_FRACTION = {"decoder_only": (1e-3, 0.60),
                               "last2_blocks": (1e-4, 0.60),
                               "full":         (0.999, 1.0)}

# ─── SAM3 / TRACKING / DYNAMIC-FILTER CONSTANTS (inherited: read by the copied SAM3 code) ─────────
# None of the six final recipes uses tracking or the stationary-object losses (spec.needs_tracking is False for all).
ENABLE_TRACKING        = False
ENABLE_GEOMETRIC_FILTER = False      # stationary-object machinery is NOT part of the final recipes; must stay False
GEOMETRIC_FILTER_WARMUP_EPOCHS = 2
TRACKER_TYPE           = "hungarian"
SAM3_MASK_DIR          = "/home/arka/Downloads/MMDE/outputs/inst_seg_output"
DYNAMIC_FILTER_MODE    = "both"
FLOW_RESIDUAL_PX       = 2.0
TRACK_IOU_MATCH_THRESH = 0.15
TRACK_MAX_AGE          = 5
TRACK_MIN_HITS         = 1
BYTETRACK_HIGH_THRESH  = 0.5
BYTETRACK_LOW_THRESH   = 0.1

# ─── quick echo (the full banner is printed by Section 8 once the recipe registry exists) ─────────
_ALL_IDS = list(RECIPE_WEIGHTS)
_FLAGS   = dict(zip(_ALL_IDS, [RUN_A1, RUN_A3, RUN_B1, RUN_C1, RUN_C3, RUN_C5]))
_sel     = [r for r in _ALL_IDS if _FLAGS[r]] if SELECTED_RECIPES is None else list(SELECTED_RECIPES)
print("FINAL DA3METRIC-LARGE SELF-SUPERVISED FINE-TUNING")
print("-" * 49)
print(f"Model: {MODEL_ID.split('/')[-1]}   Architecture: {ARCH_MODE}   Learning rate: {LEARNING_RATE:g}   Weight decay: {WEIGHT_DECAY:g}")
print(f"Optimizer: {OPTIM_MODE}   Scheduler: {SCHED_MODE}   Augmentation: {AUG_MODE}")
print(f"Epochs: {FINAL_EPOCHS}   Batch size: {BATCH_SIZE}   Accumulation: {ACCUM_STEPS}   Effective batch size: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Early-stop patience: {EARLY_STOP_PATIENCE}   MAX_TRAIN: {MAX_TRAIN} (0 = all)")
print(f"Output: {OUTPUT_BASE}")
print("Selected recipes:", _sel)

FINAL DA3METRIC-LARGE SELF-SUPERVISED FINE-TUNING
-------------------------------------------------
Model: DA3METRIC-LARGE   Architecture: decoder_only   Learning rate: 1e-06   Weight decay: 0.0001
Optimizer: adamw   Scheduler: cosine_warmup   Augmentation: base
Epochs: 50   Batch size: 8   Accumulation: 4   Effective batch size: 32
Early-stop patience: 10   MAX_TRAIN: 0 (0 = all)
Output: /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/final_self_supervised_FT
Selected recipes: ['B1_gt_pose_rigid_flow']


## 2 · Environment / Imports

In [2]:
from __future__ import annotations
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import copy
import csv
import glob
import json
import logging
import math
import shutil
import sys
import time
import warnings
from itertools import combinations
from collections import Counter
from dataclasses import dataclass, field
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset
import torch.backends.cudnn as cudnn
from contextlib import nullcontext
from functools import lru_cache         # unsure about the mask size
from concurrent.futures import ThreadPoolExecutor
executor = ThreadPoolExecutor(max_workers=2)
cudnn.benchmark = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ssl_ablation")
logging.getLogger("transformers").setLevel(logging.ERROR)

print("Imports OK  |  CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")

Imports OK  |  CUDA: True
  device: Tesla V100-PCIE-32GB


### 2b · Derived configuration, mixed-precision mode and experiment provenance

Copied from the source's configuration cell (AMP dtype block + provenance hash). Every checkpoint, warm-up cache and summary row is namespaced by `EXPERIMENT_HASH` (a hash of all training-relevant constants), so a checkpoint produced under a different configuration or code version can never be resumed by accident. The hash exclusion list additionally ignores `SELECTED_RECIPES` and the pre-flight switches, so running a *subset* of recipes does not change the hash of the others.

In [3]:
# (3) Mixed precision.  Geometry (backprojection / projection / warping) and all
#     loss maths ALWAYS run in fp32 in v6 (fp16 overflowed: K·X reaches ~1e5 > 65504).
#     "fp16" (default, uses GradScaler) | "bf16" | "auto"
AMP_DTYPE_MODE = "fp16"
if AMP_DTYPE_MODE == "bf16" or (AMP_DTYPE_MODE == "auto" and torch.cuda.is_available()
                                and torch.cuda.is_bf16_supported()):
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16
USE_GRAD_SCALER = bool(MIXED_PREC and torch.cuda.is_available() and AMP_DTYPE == torch.float16)



# =============================================================================
# EXPERIMENT PROVENANCE (v6.2)
# =============================================================================
# Every temp checkpoint, warm-up state and summary row is namespaced by a hash of ALL training-relevant
# constants above (paths, RUN_* switches, HPT grids, worker counts etc. are excluded), so a checkpoint
# produced under a different configuration / code version can never be resumed by accident.
# On top of this, each trial adds its own hash (loss definition + weights, arch, aug, optimizer, scheduler,
# routing spec, HPT config) — see `_trial_hash` in the runner.
import hashlib as _hashlib, json as _json
EXPERIMENT_CODE_VERSION = "final-v1 (derived from v6.2.1-stationary-fix)"   # differs from every ablation build -> ablation checkpoints can never be resumed
_HASH_EXCLUDE = ("DIR", "ROOT", "CSV", "PATH", "OUTPUT", "CACHE", "RUN_", "NUM_WORKERS", "SANITY",
                 "REQUIRE_", "LOG", "EXPERIMENT", "CUDA", "GPU", "_GRID", "MAX_TEST", "STRICT_", "SELECTED", "PREFLIGHT")
EXPERIMENT_CONFIG = {
    k: (list(v) if isinstance(v, tuple) else v)
    for k, v in sorted(globals().items())
    if k.isupper() and not k.startswith("_") and isinstance(v, (int, float, str, bool, tuple, list, dict))
    and not any(x in k for x in _HASH_EXCLUDE)
}
EXPERIMENT_CONFIG["TRAIN_ROOT"] = str(TRAIN_ROOT)          # data source + SAM3 masks change the experiment
EXPERIMENT_CONFIG["SAM3_MASK_DIR"] = str(SAM3_MASK_DIR)
EXPERIMENT_CONFIG["AMP_DTYPE"] = str(AMP_DTYPE)
EXPERIMENT_CONFIG["USE_GRAD_SCALER"] = bool(USE_GRAD_SCALER)
EXPERIMENT_CONFIG["CODE_VERSION"] = EXPERIMENT_CODE_VERSION
EXPERIMENT_HASH = _hashlib.md5(_json.dumps(EXPERIMENT_CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:10]
TEMP_CKPT_DIR = os.path.join(TEMP_CKPT_DIR, EXPERIMENT_HASH)          # hash-namespaced checkpoint directory
os.makedirs(TEMP_CKPT_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, f"experiment_config_{EXPERIMENT_HASH}.json"), "w") as _f:
    _json.dump(EXPERIMENT_CONFIG, _f, indent=1, sort_keys=True, default=str)
print(f"  experiment hash {EXPERIMENT_HASH}  (checkpoints: {TEMP_CKPT_DIR})")


# ── final-notebook guards (NEW) ──────────────────────────────────────────────
assert ARCH_MODE == "decoder_only", "the final notebook trains the fixed decoder_only architecture"
assert AUG_MODE == "base" and OPTIM_MODE == "adamw" and SCHED_MODE == "cosine_warmup"
assert +1 in SOURCE_OFFSETS, "temporal / flow recipes need the t+1 frame"
assert CANONICAL_SAMPLE_SET, "all recipes must share one canonical sample set (A/B need OXTS; C must use the SAME images)"
assert not ENABLE_GEOMETRIC_FILTER, "stationary-object geometric filter is not part of the final recipes"
assert W_ANCHOR == 0.0, "no LiDAR / SILog anchor in any final recipe"
assert FINAL_EPOCHS >= 1 and EARLY_STOP_PATIENCE >= 1

  experiment hash b093e0aeb3  (checkpoints: /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/final_self_supervised_FT/temp_ckpts/b093e0aeb3)


## 3 · Reused Source-Notebook Utilities  *(copied verbatim)*

### 3.1 · SAM3 masks & tracking data structures (source Part 3)
Required by the dynamic-object mask used by **A3**'s temporal term (`load_sam3_masks_for_frame`, `MaskDetection`). The three tracker classes are kept **unchanged** because the copied training loop (`_iter_batches`) still references the tracking path; **no final recipe activates it** (`needs_tracking=False`, asserted in Section 8).

In [4]:
# =============================================================================
# SAM3 mask loading + Detection/Track data structures (Part 3)
# =============================================================================

@dataclass
class MaskDetection:
    """One SAM3 instance mask, resolved to the geometric quantities every
    tracker needs (bbox / centroid / area are cached at construction time so
    trackers never recompute them from the raw mask on every comparison)."""
    mask: np.ndarray                              # bool [H, W]
    bbox: Tuple[float, float, float, float]        # (x1, y1, x2, y2)
    centroid: Tuple[float, float]                  # (cx, cy) in pixel coords
    area: float                                    # mask pixel count
    class_name: str = "unknown"
    score: float = 1.0
    embedding: Optional[np.ndarray] = None         # optional appearance descriptor


@dataclass
class Track:
    """A tracked object, persisted across frames by all three trackers below."""
    track_id: int
    mask: np.ndarray
    bbox: Tuple[float, float, float, float]
    centroid: Tuple[float, float]
    area: float
    class_name: str
    score: float
    embedding: Optional[np.ndarray] = None
    age: int = 0                                   # frames since track creation
    hits: int = 1                                  # number of matched detections total
    time_since_update: int = 0                     # frames since last matched detection
    history: List[Tuple[float, float]] = field(default_factory=list)


def _bbox_iou(b1: Tuple[float, float, float, float],
              b2: Tuple[float, float, float, float]) -> float:
    """Cheap axis-aligned bbox IoU, used as a quick reject before the
    expensive full-resolution mask IoU below."""
    x1, y1, x2, y2 = b1
    X1, Y1, X2, Y2 = b2
    ix1, iy1 = max(x1, X1), max(y1, Y1)
    ix2, iy2 = min(x2, X2), min(y2, Y2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    a1 = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a2 = max(0.0, X2 - X1) * max(0.0, Y2 - Y1)
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0


def mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Exact pixel-wise IoU between two boolean masks."""
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def _color_histogram_embedding(rgb_uint8: np.ndarray, mask: np.ndarray,
                                bins: int = 16) -> np.ndarray:
    """Cheap appearance descriptor: normalized per-channel HSV histogram
    inside the mask. Serves as the 'optional appearance embedding' term in
    the Hungarian / ByteTrack cost functions when no learned Re-ID network
    is available (SAM3 does not expose one by default)."""
    hsv = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2HSV)
    pix = hsv[mask]
    if pix.shape[0] == 0:
        return np.zeros(bins * 3, dtype=np.float32)
    hist = [np.histogram(pix[:, c], bins=bins, range=(0, 256))[0].astype(np.float32)
            for c in range(3)]
    v = np.concatenate(hist)
    return v / (v.sum() + 1e-6)


def _mask_to_detection(mask: np.ndarray, class_name: str = "unknown",
                        score: float = 1.0,
                        rgb: Optional[np.ndarray] = None) -> Optional[MaskDetection]:
    """Builds a MaskDetection from a raw boolean mask. Returns None for an
    empty mask (can happen after resizing/cropping a SAM3 export)."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x1, x2 = float(xs.min()), float(xs.max())
    y1, y2 = float(ys.min()), float(ys.max())
    cx, cy = float(xs.mean()), float(ys.mean())
    area = float(mask.sum())
    emb = _color_histogram_embedding(rgb, mask) if rgb is not None else None
    return MaskDetection(mask=mask, bbox=(x1, y1, x2, y2), centroid=(cx, cy),
                         area=area, class_name=class_name, score=float(score),
                         embedding=emb)

# Mask helpers
def get_mask_dir(rgb_path: str, mask_root: str, kitti_root: str) -> str:
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), kitti_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)

@lru_cache(maxsize=256)          # from none from 4096 unsure about the size so not caching the masks
def load_masks_for_frame(mask_dir: str) -> List[Dict]:
    p = Path(mask_dir)
    if not p.exists(): return []
    stem = p.name; manifest = p / f"{stem}_manifest.json"
    instances = []
    if manifest.exists():
        for rec in json.loads(manifest.read_text()):
            mf = p / rec["mask_file"]
            if not mf.exists(): continue
            instances.append({
                "uid": rec["uid"], "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"], "sub_cat": rec["sub_cat"],
                "score": float(rec["score"]), "box": rec.get("box"),
                "mask": np.array(Image.open(mf).convert("L")) > 127, "path": str(mf),
            })
        return instances
    for mf in sorted(p.glob("*_mask.png")):
        s = mf.stem[:-5]
        if not s.startswith(stem+"_"): continue
        parts = s[len(stem)+1:].rsplit("_", 3)
        if len(parts) != 4: continue
        cat_str, idx_str, uid_str, score_str = parts
        cp = cat_str.split("_", 1)
        if len(cp) != 2: continue
        super_cat, sub_cat = cp
        try:
            local_idx = int(idx_str); score = float(score_str)
        except ValueError: continue
        instances.append({
            "uid": uid_str, "local_idx": local_idx, "super_cat": super_cat,
            "sub_cat": sub_cat, "score": score, "box": None,
            "mask": np.array(Image.open(mf).convert("L")) > 127, "path": str(mf),
        })
    instances.sort(key=lambda x: x["local_idx"]); return instances
# ─────────────────────────────────────────────────────────────────────────────

def load_sam3_masks_for_frame(
    rgb_path: str,
    rgb_np: Optional[np.ndarray] = None
) -> List[MaskDetection]:
    """
    Load pre-generated SAM3 instance masks for one KITTI RGB frame and
    convert them into the MaskDetection objects expected by the trackers.

    SAM3 export layout:

        <SAM3_MASK_DIR>/
            train/
                <date>/
                    <drive>/
                        image_02/
                            data/
                                <frame_id>/
                                    <frame_id>_manifest.json
                                    <frame_id>_overview.jpg
                                    <frame_id>_*_mask.png

    Example:

        ~/Downloads/MMDE/outputs/inst_seg_output/train/
            2011_09_26/
                2011_09_26_drive_0001_sync/
                    image_02/
                        data/
                            0000000005/
                                0000000005_manifest.json
                                0000000005_overview.jpg
                                0000000005_road_road_surface_0000_..._mask.png

    The actual on-disk format is handled by:

        get_mask_dir(...)
        load_masks_for_frame(...)

    This function only adapts those loaded instance dictionaries into
    MaskDetection objects used by the tracking code.

    Args:
        rgb_path:
            Path to the corresponding KITTI RGB frame.

        rgb_np:
            Optional RGB uint8 image with shape [H, W, 3].
            When supplied, a HSV color-histogram appearance embedding
            is calculated for each mask.

    Returns:
        List[MaskDetection]:
            One MaskDetection per valid SAM3 instance.
            Returns [] when the mask directory does not exist or no valid
            masks can be loaded.
    """

    try:
        # ------------------------------------------------------------------
        # 1. Resolve the frame-specific SAM3 mask directory.
        #
        # Example result:
        #
        # ~/Downloads/MMDE/outputs/inst_seg_output/train/
        #     2011_09_26/
        #     2011_09_26_drive_0001_sync/
        #     image_02/data/
        #     0000000005/
        # ------------------------------------------------------------------
        mask_root = str(Path(SAM3_MASK_DIR).expanduser())
        kitti_root = str(Path(KITTI_ROOT).expanduser())

        mask_dir = get_mask_dir(
            rgb_path=rgb_path,
            mask_root=mask_root,
            kitti_root=kitti_root,
        )

        # ------------------------------------------------------------------
        # 2. Load the masks using the actual SAM3 directory/manifest format.
        # ------------------------------------------------------------------
        instances = load_masks_for_frame(mask_dir)

        if not instances:
            return []

    except Exception as e:
        log.warning(
            "[SAM3] failed to load masks for frame %s: %s",
            rgb_path,
            e,
        )
        return []

    # ----------------------------------------------------------------------
    # 3. Convert every loaded SAM3 instance into a MaskDetection.
    # ----------------------------------------------------------------------
    detections: List[MaskDetection] = []

    for inst in instances:
        try:
            # Mask returned by load_masks_for_frame() is already boolean,
            # but np.asarray(..., dtype=bool) makes this explicit and safe.
            mask = np.asarray(inst["mask"], dtype=bool)
            if mask.shape[:2] != (TARGET_H, TARGET_W):              # added a check to match and correct the target height and target width of mask and image
                mask = cv2.resize(
                    mask.astype(np.uint8), (TARGET_W, TARGET_H),
                    interpolation=cv2.INTER_NEAREST,
                ).astype(bool)

            # --------------------------------------------------------------
            # Build the tracker-facing class label.
            #
            # Example:
            #   super_cat = "road"
            #   sub_cat   = "road_surface"
            #
            # becomes:
            #   "road/road_surface"
            # --------------------------------------------------------------
            super_cat = str(inst.get("super_cat", "")).strip()
            sub_cat = str(inst.get("sub_cat", "")).strip()

            if super_cat and sub_cat:
                class_name = f"{super_cat}/{sub_cat}"
            elif sub_cat:
                class_name = sub_cat
            elif super_cat:
                class_name = super_cat
            else:
                class_name = "unknown"

            score = float(inst.get("score", 1.0))

            # --------------------------------------------------------------
            # Convert raw mask -> MaskDetection.
            #
            # _mask_to_detection() calculates:
            #   bbox
            #   centroid
            #   area
            #   optional appearance embedding
            # --------------------------------------------------------------
            det = _mask_to_detection(
                mask=mask,
                class_name=class_name,
                score=score,
                rgb=rgb_np,
            )

            # Empty masks are ignored because _mask_to_detection()
            # returns None for them.
            if det is not None:
                detections.append(det)

        except Exception as e:
            log.warning(
                "[SAM3] failed to convert mask instance from %s: %s",
                mask_dir,
                e,
            )
            continue

    return detections

#__________________________________________________________________________________________________________

# =============================================================================
# Tracker 1 — Mask IoU Tracker (greedy)
# =============================================================================

class MaskIoUTracker:
    """
    Greedy frame-to-frame association using a weighted combination of:
      • mask IoU                     (primary geometric cue)
      • normalized centroid distance (handles small-IoU-but-same-object cases
                                       from motion / segmentation jitter)
      • area ratio                   (penalizes matches with drastically
                                       different scale, e.g. mis-segmentation)

    Matching is greedy (highest combined score first), not globally optimal —
    a faithful mask-based adaptation of the association step in SORT
    (Bewley et al., ICRA 2016). Cheapest of the three trackers.

    Limitation: greedy matching can commit to a locally-optimal but globally
    suboptimal assignment when many similar objects are close together
    (e.g. a row of parked cars); Tracker 2 resolves this with an optimal solve.
    """

    def __init__(self, iou_weight: float = 0.6, centroid_weight: float = 0.25,
                 area_weight: float = 0.15, iou_match_threshold: float = TRACK_IOU_MATCH_THRESH,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.iou_weight, self.centroid_weight, self.area_weight = iou_weight, centroid_weight, area_weight
        self.iou_match_threshold = iou_match_threshold
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._next_id = 1

    def _score(self, trk: Track, det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        cx1, cy1 = trk.centroid
        cx2, cy2 = det.centroid
        centroid_sim = max(0.0, 1.0 - math.hypot(cx1 - cx2, cy1 - cy2) / self.img_diag)
        area_ratio = min(trk.area, det.area) / (max(trk.area, det.area) + 1e-6)
        return (self.iou_weight * iou + self.centroid_weight * centroid_sim +
                self.area_weight * area_ratio)

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        matched_trk, matched_det = set(), set()

        if self.tracks and detections:
            pairs = [(self._score(trk, det), ti, di)
                     for ti, trk in enumerate(self.tracks)
                     for di, det in enumerate(detections)]
            pairs = [p for p in pairs if p[0] > 0]
            pairs.sort(key=lambda x: -x[0])
            for s, ti, di in pairs:
                if ti in matched_trk or di in matched_det or s < self.iou_match_threshold:
                    continue
                self._apply_match(ti, detections[di])
                matched_trk.add(ti)
                matched_det.add(di)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_trk:
                trk.age += 1
                trk.time_since_update += 1

        for di, det in enumerate(detections):
            if di not in matched_det:
                self._spawn_track(det)

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        self.tracks.append(Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox,
                                  centroid=det.centroid, area=det.area, class_name=det.class_name,
                                  score=det.score, embedding=det.embedding, history=[det.centroid]))
        self._next_id += 1


# =============================================================================
# Tracker 2 — Hungarian Assignment Tracker (optimal)
# =============================================================================

class HungarianMaskTracker:
    """
    Formulates frame-to-frame association as a linear assignment problem,
    solved optimally (Kuhn–Munkres) via scipy.optimize.linear_sum_assignment:

        minimize  sum_{i,j} C[i,j] * x[i,j]   s.t. each track <-> at most one detection

    Cost matrix combines four normalized terms in [0, 1]:

        C = w_iou * (1 - IoU) + w_centroid * (dist / img_diag)
            + w_area * (1 - area_ratio) + w_appearance * (1 - cosine_sim(embedding))

    The appearance term is skipped (weight renormalized across the other
    three) when either side has no embedding.

    Unlike Tracker 1's greedy matching, this always finds the assignment that
    minimizes TOTAL cost across all pairs simultaneously — important when
    several similar stationary objects sit next to each other (e.g. parked
    cars), where greedy IoU tracking can swap identities.
    """

    def __init__(self, w_iou: float = 0.5, w_centroid: float = 0.25, w_area: float = 0.15,
                 w_appearance: float = 0.10, max_cost: float = 0.85,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.w_iou, self.w_centroid, self.w_area, self.w_appearance = w_iou, w_centroid, w_area, w_appearance
        self.max_cost = max_cost
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._next_id = 1

    def _cost(self, trk: Track, det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        cx1, cy1 = trk.centroid
        cx2, cy2 = det.centroid
        dist_n = min(1.0, math.hypot(cx1 - cx2, cy1 - cy2) / self.img_diag)
        area_ratio = min(trk.area, det.area) / (max(trk.area, det.area) + 1e-6)

        w_iou, w_c, w_a = self.w_iou, self.w_centroid, self.w_area
        if trk.embedding is None or det.embedding is None:
            renorm = 1.0 / (w_iou + w_c + w_a + 1e-9)
            w_iou, w_c, w_a = w_iou * renorm, w_c * renorm, w_a * renorm
            appearance_term = 0.0
        else:
            num = float(np.dot(trk.embedding, det.embedding))
            den = float(np.linalg.norm(trk.embedding) * np.linalg.norm(det.embedding) + 1e-9)
            cos_sim = max(-1.0, min(1.0, num / den))
            appearance_term = self.w_appearance * (1.0 - cos_sim)

        return w_iou * (1.0 - iou) + w_c * dist_n + w_a * (1.0 - area_ratio) + appearance_term

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        n_trk, n_det = len(self.tracks), len(detections)
        matched_trk, matched_det = set(), set()

        if n_trk > 0 and n_det > 0:
            C = np.full((n_trk, n_det), 10.0, dtype=np.float64)
            for ti, trk in enumerate(self.tracks):
                for di, det in enumerate(detections):
                    C[ti, di] = self._cost(trk, det)
            row_idx, col_idx = linear_sum_assignment(C)
            for ti, di in zip(row_idx, col_idx):
                if C[ti, di] > self.max_cost:
                    continue
                self._apply_match(ti, detections[di])
                matched_trk.add(ti)
                matched_det.add(di)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_trk:
                trk.age += 1
                trk.time_since_update += 1

        for di, det in enumerate(detections):
            if di not in matched_det:
                self._spawn_track(det)

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        self.tracks.append(Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox,
                                  centroid=det.centroid, area=det.area, class_name=det.class_name,
                                  score=det.score, embedding=det.embedding, history=[det.centroid]))
        self._next_id += 1


# =============================================================================
# Tracker 3 — ByteTrack-compatible Tracker (masks + Kalman-predicted centroid)
# =============================================================================

class _ConstantVelocityKF:
    """Minimal constant-velocity Kalman filter over 2-D centroid (cx, cy, vx, vy).
    Plays the role of the bbox Kalman filter in the original SORT/ByteTrack
    formulation, adapted to mask centroids since masks have no natural
    fixed-size box parametrization to filter."""

    def __init__(self, cx: float, cy: float):
        self.x = np.array([cx, cy, 0.0, 0.0], dtype=np.float64)
        self.P = np.eye(4) * 10.0
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=np.float64)
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=np.float64)
        self.Q = np.eye(4) * 1.0
        self.R = np.eye(2) * 4.0

    def predict(self) -> Tuple[float, float]:
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return float(self.x[0]), float(self.x[1])

    def update(self, cx: float, cy: float) -> None:
        z = np.array([cx, cy], dtype=np.float64)
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P


class ByteTrackMaskTracker:
    """
    Adapts Zhang et al., "ByteTrack: Multi-Object Tracking by Associating
    Every Detection Box" (ECCV 2022), to instance segmentation masks. The
    core idea preserved from the original: don't discard low-confidence
    detections before matching — they are frequently partially-occluded
    TRUE objects, and dropping them (as Trackers 1-2 implicitly do by only
    seeing high-confidence SAM3 output) causes ID switches and fragmented
    tracks.

    Two-stage association per frame:
      Stage 1 (detections with score >= high_thresh):
          Hungarian-matched to ALL existing tracks, cost computed against
          each track's Kalman-PREDICTED centroid (not its last observation).
      Stage 2 (low_thresh <= score < high_thresh):
          Hungarian-matched only to tracks still unmatched after Stage 1 —
          recovers tracks through brief partial occlusion without ever
          letting a low-confidence detection spawn a new track.
      New tracks are created ONLY from unmatched Stage-1 detections.
      Unmatched tracks are kept ("lost") for `max_age` frames, Kalman-predicted
      forward with no measurement update, before deletion.

    Complexity: O(NM) per stage (two Hungarian solves), same order as Tracker 2.
    Limitation: still a nearest-neighbour tracker with no long-term Re-ID
    gallery; identity can be lost across occlusions longer than max_age.
    """

    def __init__(self, high_thresh: float = BYTETRACK_HIGH_THRESH, low_thresh: float = BYTETRACK_LOW_THRESH,
                 w_iou: float = 0.6, w_centroid: float = 0.4, max_cost: float = 0.9,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.high_thresh, self.low_thresh = high_thresh, low_thresh
        self.w_iou, self.w_centroid = w_iou, w_centroid
        self.max_cost = max_cost
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._kfs: Dict[int, _ConstantVelocityKF] = {}
        self._next_id = 1

    def _cost(self, trk: Track, pred_centroid: Tuple[float, float], det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        dist_n = min(1.0, math.hypot(pred_centroid[0] - det.centroid[0],
                                     pred_centroid[1] - det.centroid[1]) / self.img_diag)
        return self.w_iou * (1.0 - iou) + self.w_centroid * dist_n

    def _hungarian_match(self, track_idxs: List[int], dets: List[MaskDetection],
                          pred_centroids: Dict[int, Tuple[float, float]]):
        if not track_idxs or not dets:
            return [], list(range(len(dets))), list(track_idxs)
        C = np.full((len(track_idxs), len(dets)), 10.0)
        for r, ti in enumerate(track_idxs):
            for c, det in enumerate(dets):
                C[r, c] = self._cost(self.tracks[ti], pred_centroids[ti], det)
        rows, cols = linear_sum_assignment(C)
        matches: List[Tuple[int, int]] = []
        unmatched_dets, unmatched_trks = set(range(len(dets))), set(track_idxs)
        for r, c in zip(rows, cols):
            if C[r, c] <= self.max_cost:
                matches.append((track_idxs[r], c))
                unmatched_dets.discard(c)
                unmatched_trks.discard(track_idxs[r])
        return matches, sorted(unmatched_dets), sorted(unmatched_trks)

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        high = [d for d in detections if d.score >= self.high_thresh]
        low = [d for d in detections if self.low_thresh <= d.score < self.high_thresh]

        pred_centroids: Dict[int, Tuple[float, float]] = {}
        for ti, trk in enumerate(self.tracks):
            kf = self._kfs.get(trk.track_id)
            pred_centroids[ti] = kf.predict() if kf is not None else trk.centroid

        all_track_idx = list(range(len(self.tracks)))
        matched_this_frame = set()

        matches1, unmatched_high, unmatched_trk1 = self._hungarian_match(all_track_idx, high, pred_centroids)
        for ti, di in matches1:
            self._apply_match(ti, high[di])
            matched_this_frame.add(ti)

        matches2, unmatched_low, unmatched_trk2 = self._hungarian_match(unmatched_trk1, low, pred_centroids)
        for ti, di in matches2:
            self._apply_match(ti, low[di])
            matched_this_frame.add(ti)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_this_frame:
                trk.age += 1
                trk.time_since_update += 1

        for di in unmatched_high:
            self._spawn_track(high[di])

        alive = []
        for trk in self.tracks:
            if trk.time_since_update <= self.max_age:
                alive.append(trk)
            else:
                self._kfs.pop(trk.track_id, None)
        self.tracks = alive
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)
        kf = self._kfs.get(trk.track_id)
        if kf is not None:
            kf.update(*det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        new_trk = Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox, centroid=det.centroid,
                         area=det.area, class_name=det.class_name, score=det.score,
                         embedding=det.embedding, history=[det.centroid])
        self.tracks.append(new_trk)
        self._kfs[new_trk.track_id] = _ConstantVelocityKF(*det.centroid)
        self._next_id += 1


def build_tracker(tracker_type: str = TRACKER_TYPE, img_diag: float = 1000.0):
    """Factory selecting one of the three trackers by TRACKER_TYPE config."""
    if tracker_type == "iou":
        return MaskIoUTracker(img_diag=img_diag)
    if tracker_type == "hungarian":
        return HungarianMaskTracker(img_diag=img_diag)
    if tracker_type == "bytetrack":
        return ByteTrackMaskTracker(img_diag=img_diag)
    raise ValueError(f"Unknown TRACKER_TYPE: {tracker_type}")


print("SAM3 mask loading + 3 trackers ready.")
print("  load_sam3_masks_for_frame -> List[MaskDetection] per frame (adapter, edit path logic if needed)")
print("  MaskIoUTracker            -> Tracker 1 (greedy)")
print("  HungarianMaskTracker      -> Tracker 2 (optimal assignment)")
print("  ByteTrackMaskTracker      -> Tracker 3 (two-stage + Kalman-predicted centroid)")
print("  build_tracker(TRACKER_TYPE) -> convenience factory used by the training loop")

SAM3 mask loading + 3 trackers ready.
  load_sam3_masks_for_frame -> List[MaskDetection] per frame (adapter, edit path logic if needed)
  MaskIoUTracker            -> Tracker 1 (greedy)
  HungarianMaskTracker      -> Tracker 2 (optimal assignment)
  ByteTrackMaskTracker      -> Tracker 3 (two-stage + Kalman-predicted centroid)
  build_tracker(TRACKER_TYPE) -> convenience factory used by the training loop


### 3.2 · Moving-object class filter (source Part 4, incl. the v6.2.1 compound-label normalisation)
`is_class_dynamic` decides which SAM3 classes (car, pedestrian, cyclist, …) are treated as potentially moving. Its built-in self-test (which also covers the compound `"super_cat/sub_cat"` labels) runs on synthetic masks.

In [5]:
# =============================================================================
# Moving-object filter (Part 4)
# =============================================================================

DYNAMIC_CLASS_NAMES = {
    "car", "truck", "bus", "van", "person", "pedestrian", "rider", "riders",
    "bicycle", "bicyclist", "cyclist", "motorcycle", "motorbike", "train", "trailer",
}
STATIC_CLASS_NAMES = {
    "pole", "traffic sign", "traffic light", "building", "wall", "fence",
    "vegetation", "terrain", "sidewalk", "road", "sky", "guard rail",
    "parking", "static", "bridge", "tunnel",
}


def _normalize_class_token(token: str) -> str:
    """v6.2.1 (Bug 2): STATIC_CLASS_NAMES / DYNAMIC_CLASS_NAMES are flat, space-separated
    names (\"traffic sign\", \"guard rail\", \"pole\", \"car\", ...). The actual SAM3 loader
    (load_sam3_masks_for_frame, Part 3) emits class_name as \"super_cat/sub_cat\"
    (e.g. \"vehicle/car\", \"traffic/traffic_sign\", \"road/road_surface\"), with underscores
    inside multi-word sub-categories. An exact match of that compound string against either
    set almost never succeeds, so nearly everything fell through to the conservative
    "unknown -> dynamic" default regardless of its true class -- this is the most likely
    reason the real Group 1c data produced zero usable stationary objects. Normalizing
    underscores/hyphens to spaces is the ONLY change here; no new class names are added."""
    return " ".join(token.strip().lower().replace("_", " ").replace("-", " ").split())


def is_class_dynamic(class_name: str) -> bool:
    """Manual filtering rule (Part 4a). Unknown/unlisted classes default to
    'dynamic' (excluded) — see markdown above for why this default is
    deliberately conservative for this use case.

    v6.2.1: SAM3 emits compound labels ("super_cat/sub_cat", e.g. "vehicle/car",
    "traffic/traffic_sign", "road/road_surface"). We check the whole normalized label
    plus each "/"-separated part -- most specific (sub_cat) first, then super_cat --
    against the SAME two sets, with underscores/hyphens normalized to spaces. The
    STATIC_CLASS_NAMES / DYNAMIC_CLASS_NAMES sets and the "unknown -> dynamic" default
    are unchanged; only which strings get compared against them changed."""
    raw = class_name.strip().lower()
    parts = [p for p in raw.split("/") if p]
    candidates = [_normalize_class_token(p) for p in reversed(parts)]   # sub_cat before super_cat
    whole = _normalize_class_token(raw.replace("/", " "))
    if whole not in candidates:
        candidates.append(whole)
    for name in candidates:
        if name in STATIC_CLASS_NAMES:
            return False
        if name in DYNAMIC_CLASS_NAMES:
            return True
    return True  # unknown class -> conservatively treat as dynamic (exclude)


def compute_dense_flow(rgb_curr_np: np.ndarray, rgb_next_np: np.ndarray) -> np.ndarray:
    """Farneback dense optical flow between two uint8 RGB frames. Returns
    flow [H, W, 2] in pixels (Part 4b)."""
    g1 = cv2.cvtColor(rgb_curr_np, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(rgb_next_np, cv2.COLOR_RGB2GRAY)
    return cv2.calcOpticalFlowFarneback(
        g1, g2, None, pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)


def estimate_ego_flow(flow: np.ndarray, static_mask_union: Optional[np.ndarray] = None) -> np.ndarray:
    """Median flow over background pixels as a cheap proxy for the
    ego-motion-induced flow field. Prefers the union of class-static masks;
    falls back to the image's upper quarter (sky/distant background in
    KITTI's forward-facing camera) if no static masks are available."""
    if static_mask_union is not None and static_mask_union.sum() > 100:
        return np.median(flow[static_mask_union], axis=0)
    h = flow.shape[0]
    return np.median(flow[: h // 4, :, :].reshape(-1, 2), axis=0)


def is_object_stationary_by_flow(mask: np.ndarray, flow: np.ndarray,
                                  ego_flow: Optional[np.ndarray] = None,
                                  residual_threshold: float = FLOW_RESIDUAL_PX) -> bool:
    """
    Flags an object as stationary if its median optical flow, AFTER
    subtracting the scene's ego-motion-induced background flow, is below
    `residual_threshold` pixels. See markdown above for the reasoning.
    """
    if mask.sum() < 20:
        return False  # too small a region to trust a flow estimate; exclude conservatively
    obj_median = np.median(flow[mask], axis=0)
    if ego_flow is None:
        ego_flow = np.zeros(2, dtype=np.float32)
    residual = float(np.linalg.norm(obj_median - ego_flow))
    return residual <= residual_threshold


def filter_dynamic_objects(tracks: List[Track],
                            rgb_curr_np: Optional[np.ndarray] = None,
                            rgb_next_np: Optional[np.ndarray] = None,
                            mode: str = DYNAMIC_FILTER_MODE,
                            flow_residual_threshold: float = FLOW_RESIDUAL_PX) -> List[Track]:
    """
    Combined moving-object filter (Part 4). Returns only the STATIONARY
    subset of `tracks`, ready to feed into Part 5's stationary-object losses.

    mode:
      "class" -> manual class-name filtering only (fast, no RGB needed)
      "flow"  -> optical-flow residual filtering only
      "both"  -> object must pass BOTH checks (default; see markdown above)
    """
    if mode not in ("class", "flow", "both"):
        raise ValueError(f"Unknown DYNAMIC_FILTER_MODE: {mode}")

    class_static = [not is_class_dynamic(t.class_name) for t in tracks]

    if mode == "class":
        return [t for t, keep in zip(tracks, class_static) if keep]

    if rgb_curr_np is None or rgb_next_np is None:
        log.warning("[filter_dynamic_objects] mode=%s requires rgb frames; "
                     "falling back to class-only filtering for this pair.", mode)
        return [t for t, keep in zip(tracks, class_static) if keep]

    flow = compute_dense_flow(rgb_curr_np, rgb_next_np)
    static_union = None
    if any(class_static):
        static_union = np.zeros(flow.shape[:2], dtype=bool)
        for t, keep in zip(tracks, class_static):
            if keep and t.mask.shape == static_union.shape:
                static_union |= t.mask
    ego_flow = estimate_ego_flow(flow, static_union)

    flow_static = [is_object_stationary_by_flow(t.mask, flow, ego_flow, flow_residual_threshold)
                   for t in tracks]

    if mode == "flow":
        return [t for t, keep in zip(tracks, flow_static) if keep]

    return [t for t, c_ok, f_ok in zip(tracks, class_static, flow_static) if c_ok and f_ok]


# =============================================================================
# Self-test: verify all 3 trackers preserve identity across a synthetic
# 2-object sequence, and that the dynamic filter separates them correctly.
# =============================================================================

def _make_synthetic_mask(h: int, w: int, cx: float, cy: float, r: float) -> np.ndarray:
    yy, xx = np.mgrid[0:h, 0:w]
    return ((xx - cx) ** 2 + (yy - cy) ** 2) <= r * r


def _self_test_tracking_and_filter() -> None:
    h, w = 128, 256
    img_diag = math.hypot(h, w)

    frame0 = [
        _mask_to_detection(_make_synthetic_mask(h, w, 40, 60, 12), class_name="pole", score=0.95),
        _mask_to_detection(_make_synthetic_mask(h, w, 150, 70, 18), class_name="car", score=0.9),
    ]
    frame1 = [
        _mask_to_detection(_make_synthetic_mask(h, w, 42, 60, 12), class_name="pole", score=0.95),
        _mask_to_detection(_make_synthetic_mask(h, w, 165, 70, 18), class_name="car", score=0.9),
    ]

    for name, trk in [("MaskIoUTracker", MaskIoUTracker(img_diag=img_diag)),
                       ("HungarianMaskTracker", HungarianMaskTracker(img_diag=img_diag)),
                       ("ByteTrackMaskTracker", ByteTrackMaskTracker(img_diag=img_diag))]:
        t1 = trk.update(frame0)
        t2 = trk.update(frame1)
        ids1, ids2 = sorted(t.track_id for t in t1), sorted(t.track_id for t in t2)
        assert ids1 == ids2, f"{name}: track IDs changed across frames ({ids1} -> {ids2})"
        print(f"  {name}: OK  ({len(t2)} tracks, ids={ids2})")

    # Dynamic filter, class-mode only (no synthetic RGB needed)
    tracker = HungarianMaskTracker(img_diag=img_diag)
    tracks = tracker.update(frame0)
    stationary = filter_dynamic_objects(tracks, mode="class")
    assert len(stationary) == 1 and stationary[0].class_name == "pole", \
        f"class filter should keep only the pole, got {[t.class_name for t in stationary]}"
    print(f"  filter_dynamic_objects(mode='class'): OK (kept {[t.class_name for t in stationary]})")

    # v6.2.1 (Bug 2) regression checks: SAM3's actual compound "super_cat/sub_cat" labels
    # must be classified the same way their flat equivalents already are.
    compound_cases = [
        ("vehicle/car", True), ("traffic/traffic_sign", False), ("road/road_surface", False),
        ("structure/pole", False), ("human/pedestrian", True), ("nature/vegetation", False),
        ("Vehicle/Bicycle", True), ("some_never_seen_thing", True),   # still conservative: unknown -> dynamic
    ]
    for name, expect_dynamic in compound_cases:
        got = is_class_dynamic(name)
        assert got == expect_dynamic, f"is_class_dynamic({name!r}) = {got}, expected {expect_dynamic}"
    print(f"  is_class_dynamic() compound-label normalization: OK ({len(compound_cases)} cases)")


_self_test_tracking_and_filter()
print("Tracking + dynamic-filter self-test passed "
      "(synthetic 2-object sequence: all 3 trackers preserve IDs, filter isolates the stationary pole).")

  MaskIoUTracker: OK  (2 tracks, ids=[1, 2])
  HungarianMaskTracker: OK  (2 tracks, ids=[1, 2])
  ByteTrackMaskTracker: OK  (2 tracks, ids=[1, 2])
  filter_dynamic_objects(mode='class'): OK (kept ['pole'])
  is_class_dynamic() compound-label normalization: OK (8 cases)
Tracking + dynamic-filter self-test passed (synthetic 2-object sequence: all 3 trackers preserve IDs, filter isolates the stationary pole).


### 3.3 · Batch-level dynamic-object mask (source Part 7)
`get_batch_dynamic_masks` = SAM3 class prior of the **current frame only**, mirrored for flipped samples, dilated by `DYN_MASK_DILATE_PX`. `_flip_detection`, `track_and_filter_pair` and `get_batch_stationary_pairs` are copied unchanged (the last is referenced by the training loop but never activated). *Not copied:* the Group-1c real-data coverage diagnostics and their regression test (they only serve the stationary-object ablation).

In [6]:
# =============================================================================
# Batch-level stationary track-pair extraction (bridges Part 3/4 -> Part 5)
# =============================================================================

def _flip_detection(det: MaskDetection) -> MaskDetection:
    """Mirror a SAM3 detection horizontally (v6: the dataset flips images on the fly, but the
    SAM3 masks on disk belong to the UNFLIPPED frame — v5 silently mismatched them)."""
    Wn = det.mask.shape[1]
    x1, y1, x2, y2 = det.bbox
    return MaskDetection(mask=np.ascontiguousarray(det.mask[:, ::-1]),
                         bbox=(Wn - 1 - x2, y1, Wn - 1 - x1, y2),
                         centroid=(Wn - 1 - det.centroid[0], det.centroid[1]),
                         area=det.area, class_name=det.class_name, score=det.score,
                         embedding=det.embedding)


def track_and_filter_pair(rgb_curr_np: np.ndarray, rgb_next_np: np.ndarray,
                          rgb_curr_path: str, rgb_next_path: str,
                          tracker_type: str = TRACKER_TYPE,
                          filter_mode: str = DYNAMIC_FILTER_MODE,
                          flipped: bool = False,
                          return_diag: bool = False
                          ):
    """
    Runs SAM3 mask loading + tracking + dynamic filtering for ONE (frame_t,
    frame_{t+1}) training pair. Returns matched (detection_t, detection_{t+1})
    tuples for objects that are BOTH successfully tracked across the pair AND
    judged stationary. This is the single entry point Part 5's losses use —
    they never touch the tracker or filter APIs directly.

    return_diag=True (v6.2.1, Bug 3/4 -- additive, default False, no change to any
    existing caller) additionally returns a small diagnostic dict so the real-data
    coverage report below can distinguish "no SAM3 detections", "no matched tracks"
    and "tracks matched but none stationary" without duplicating this function's logic.
    """
    diag = {"n_dets_curr": 0, "n_dets_next": 0, "class_hist_curr": Counter(),
           "n_matched_pairs": 0, "n_stationary_pairs": 0}
    dets_curr = load_sam3_masks_for_frame(rgb_curr_path, rgb_curr_np)
    dets_next = load_sam3_masks_for_frame(rgb_next_path, rgb_next_np)
    if flipped:
        dets_curr = [_flip_detection(d) for d in dets_curr]
        dets_next = [_flip_detection(d) for d in dets_next]
    diag["n_dets_curr"] = len(dets_curr)
    diag["n_dets_next"] = len(dets_next)
    diag["class_hist_curr"] = Counter(d.class_name for d in dets_curr)
    if not dets_curr or not dets_next:
        return ([], diag) if return_diag else []

    img_diag = math.hypot(rgb_curr_np.shape[0], rgb_curr_np.shape[1])
    tracker = build_tracker(tracker_type, img_diag=img_diag)

    tracker.update(dets_curr)  # seed tracks at frame t
    snapshot_curr: Dict[int, MaskDetection] = {
        t.track_id: MaskDetection(mask=t.mask, bbox=t.bbox, centroid=t.centroid,
                                  area=t.area, class_name=t.class_name,
                                  score=t.score, embedding=t.embedding)
        for t in tracker.tracks
    }

    tracks_next = tracker.update(dets_next)  # match into frame t+1
    paired: List[Tuple[MaskDetection, MaskDetection]] = []
    for t in tracks_next:
        # Keep only tracks that: existed at frame t, were matched (not just
        # aged) this update, and have >=2 total hits (i.e. genuinely tracked
        # across the pair, not a track spawned fresh THIS frame).
        if t.track_id in snapshot_curr and t.time_since_update == 0 and t.hits >= 2:
            det_next = MaskDetection(mask=t.mask, bbox=t.bbox, centroid=t.centroid,
                                     area=t.area, class_name=t.class_name,
                                     score=t.score, embedding=t.embedding)
            paired.append((snapshot_curr[t.track_id], det_next))

    diag["n_matched_pairs"] = len(paired)
    if not paired:
        return ([], diag) if return_diag else []

    # Re-use filter_dynamic_objects by wrapping each pair's CURRENT-FRAME state
    # in a throwaway Track (filter only reads .mask / .class_name).
    #
    # v6.2.1 (Bug 1 -- the primary fix in this patch): the optical-flow field returned by
    # compute_dense_flow(rgb_curr_np, rgb_next_np) is defined on FRAME-t PIXELS (it answers
    # "where did this frame-t pixel move to"). The object mask supplied to the flow-based
    # stationarity test must therefore also be the FRAME-t mask (det_curr.mask). The previous
    # implementation passed det_next.mask -- a frame-(t+1) mask -- into that frame-t-indexed
    # flow field, sampling flow at the wrong image location and plausibly rejecting genuinely
    # stationary tracked objects. The frame-(t+1) mask remains available as det_next.mask and
    # is unchanged for the downstream stationary-object geometric-consistency losses (Part 5),
    # which still receive both det_curr and det_next via `paired`.
    pseudo_tracks = [Track(track_id=i, mask=det_curr.mask, bbox=det_curr.bbox,
                           centroid=det_curr.centroid, area=det_curr.area,
                           class_name=det_curr.class_name, score=det_curr.score)
                     for i, (det_curr, det_next) in enumerate(paired)]
    stationary_pseudo = filter_dynamic_objects(pseudo_tracks, rgb_curr_np, rgb_next_np,
                                               mode=filter_mode)
    keep_ids = {t.track_id for t in stationary_pseudo}   # unique per-pair ids (enumerate), not a shared -1
    result = [pair for i, pair in enumerate(paired) if i in keep_ids]
    diag["n_stationary_pairs"] = len(result)
    return (result, diag) if return_diag else result


def get_batch_stationary_pairs(batch: Dict,
                               tracker_type: str = TRACKER_TYPE,
                               filter_mode: str = DYNAMIC_FILTER_MODE,
                               return_diag: bool = False
                               ):
    """
    Batch-level wrapper: expects `batch["rgb_raw"]` / `batch["next_rgb_raw"]`
    (CPU float tensors in [0, 1], as produced by the dataset before the
    `.to(device)` transfer in the training loop — reused directly here to
    avoid an extra GPU round-trip) and `batch["rgb_path"]` / `batch["next_rgb_path"]`
    (string lists). Returns, for each sample b, its list of stationary
    (det_t, det_{t+1}) pairs (possibly empty).

    return_diag=True (v6.2.1, default False, no change to any existing caller) additionally
    returns a per-sample list of the diagnostic dicts from `track_and_filter_pair`, used by
    the real-data coverage report below.
    """
    rgb_raw = batch.get("rgb_raw")
    rgb_next_raw = batch.get("next_rgb_raw")
    rgb_paths = batch.get("rgb_path")
    rgb_next_paths = batch.get("next_rgb_path")
    if rgb_raw is None or rgb_next_raw is None or rgb_paths is None or rgb_next_paths is None:
        return ([], []) if return_diag else []

    B = rgb_raw.shape[0]
    flipped = batch.get("flipped")
    flipped = flipped.cpu().numpy().astype(bool) if flipped is not None else np.zeros(B, dtype=bool)
    curr_np = (rgb_raw.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
    next_np = (rgb_next_raw.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)

    out: List[List[Tuple[MaskDetection, MaskDetection]]] = []
    diags: List[Dict] = []
    for b in range(B):
        try:
            if return_diag:
                pairs, d = track_and_filter_pair(curr_np[b], next_np[b], rgb_paths[b], rgb_next_paths[b],
                                                 tracker_type=tracker_type, filter_mode=filter_mode,
                                                 flipped=bool(flipped[b]), return_diag=True)
                diags.append(d)
            else:
                pairs = track_and_filter_pair(curr_np[b], next_np[b], rgb_paths[b], rgb_next_paths[b],
                                              tracker_type=tracker_type, filter_mode=filter_mode,
                                              flipped=bool(flipped[b]))
        except Exception as e:
            log.warning("[get_batch_stationary_pairs] sample %d failed: %s", b, e)
            pairs = []
            if return_diag:
                diags.append({"n_dets_curr": 0, "n_dets_next": 0, "class_hist_curr": Counter(),
                             "n_matched_pairs": 0, "n_stationary_pairs": 0, "error": str(e)})
        out.append(pairs)
    return (out, diags) if return_diag else out


print("Batch-level stationary track-pair extraction ready: get_batch_stationary_pairs(batch) -> "
      "List[List[Tuple[MaskDetection, MaskDetection]]]")


# ─────────────────────────────────────────────────────────────────────────────
# v6.1 — per-pixel "potentially dynamic" mask for the depth-consistency losses
# ─────────────────────────────────────────────────────────────────────────────
_DYN_STATS = {"frames": 0, "empty": 0, "warned": False}

def get_batch_dynamic_masks(batch: Dict, H: int, W: int,
                            dilate_px: int = DYN_MASK_DILATE_PX) -> torch.Tensor:
    """
    SAM3 instance masks of the CURRENT frame t whose class is *potentially moving*
    (`is_class_dynamic`: cars, pedestrians, cyclists, …), resized to the network resolution, mirrored for
    flipped samples and dilated by `dilate_px`.  Returned as float [B,1,H,W] (1 = exclude from
    temporal / geometric depth-consistency).

    This is a CLASS PRIOR: parked cars are masked too (conservative).  Only the CURRENT-frame mask is used:
    the temporal / geometric losses are evaluated on frame-t pixels, and the t+1 mask lives in frame-(t+1)
    coordinates — uniting the two (v6.1) removed legitimate background pixels wherever the object moved.
    (Warping the t+1 mask into frame t would need predicted geometry/pose and is deliberately not done.)
    If no SAM3 masks exist on disk the mask is all zeros and a warning is logged once.
    """
    B = batch["rgb"].shape[0]
    flipped = batch.get("flipped")
    flipped = flipped.cpu().numpy().astype(bool) if flipped is not None else np.zeros(B, dtype=bool)
    out = np.zeros((B, 1, H, W), dtype=np.float32)
    for b in range(B):
        for key in ("rgb_path",):
            dets = load_sam3_masks_for_frame(batch[key][b])
            _DYN_STATS["frames"] += 1
            _DYN_STATS["empty"] += int(len(dets) == 0)
            if flipped[b]:
                dets = [_flip_detection(d) for d in dets]
            for d in dets:
                if not is_class_dynamic(d.class_name):
                    continue
                m = d.mask.astype(np.uint8)
                if m.shape != (H, W):
                    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
                out[b, 0] = np.maximum(out[b, 0], m.astype(np.float32))
        if dilate_px > 0 and out[b, 0].any():
            out[b, 0] = cv2.dilate(out[b, 0], np.ones((2 * dilate_px + 1,) * 2, np.uint8))
    if (not _DYN_STATS["warned"]) and _DYN_STATS["frames"] >= 40 and _DYN_STATS["empty"] == _DYN_STATS["frames"]:
        _DYN_STATS["warned"] = True
        log.warning("get_batch_dynamic_masks: NO SAM3 detections found for any of %d frames "
                    "(SAM3_MASK_DIR=%s) -> dynamic mask is all zeros.", _DYN_STATS["frames"], SAM3_MASK_DIR)
    return torch.from_numpy(out)

print("Batch-level tracking + dynamic-mask helpers ready.")

Batch-level stationary track-pair extraction ready: get_batch_stationary_pairs(batch) -> List[List[Tuple[MaskDetection, MaskDetection]]]
Batch-level tracking + dynamic-mask helpers ready.


### 3.4 · Metrics (source Section 10 — 4 scale-alignment strategies)
Unscaled metrics are the **primary** result of this study; median / mean / affine alignment are reported only as diagnostics.

In [7]:
def eigen_crop_mask(shape: Tuple[int, int]) -> np.ndarray:
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    top = 44; bot = min(top + 352, h)
    mask[top:bot, 0:w] = True
    return mask


def compute_depth_errors(gt: np.ndarray, pred: np.ndarray,
                         mask: np.ndarray) -> Dict[str, float]:
    gt_v   = gt[mask].astype(np.float64)
    pred_v = pred[mask].astype(np.float64)
    pred_v = np.maximum(pred_v, 1e-8)
    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    mae      = float(np.mean(np.abs(gt_v - pred_v)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    log10    = float(np.mean(np.abs(np.log10(gt_v) - np.log10(pred_v))))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    mean_err = float(np.mean(gt_v - pred_v))
    n_pts    = int(mask.sum())
    return dict(n_pts=n_pts, rmse=rmse, mae=mae, abs_rel=abs_rel,
                sq_rel=sq_rel, rmse_log=rmse_log, log10=log10,
                delta1=delta1, delta2=delta2, delta3=delta3,
                mean_error=mean_err,
                lidar_mean=float(np.mean(gt_v)), pred_mean=float(np.mean(pred_v)))


def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> Dict[str, Dict]:
    # Compute metrics under 4 scale-alignment strategies.
    gt_v   = gt_depth[mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # Unscaled
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {"scale": 1.0, "shift": 0.0, **m0}

    # Median scale
    s_med = float(np.median(gt_v) / max(np.median(pred_c[mask]), EPS))
    pred_med = np.clip(pred_c * s_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {"scale": s_med, "shift": 0.0, **m1}

    # Mean scale
    s_mean = float(np.mean(gt_v) / max(np.mean(pred_c[mask]), EPS))
    pred_mean = np.clip(pred_c * s_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {"scale": s_mean, "shift": 0.0, **m2}

    # Affine (OLS)
    pf = pred_c[mask].ravel()
    A  = np.vstack([pf, np.ones_like(pf)]).T
    params, _, _, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    s_aff = float(params[0]); sh_aff = float(params[1])
    pred_aff = np.clip(s_aff * pred_c + sh_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {"scale": s_aff, "shift": sh_aff, **m3}

    return results


def true_rmse(df_sub: pd.DataFrame, rmse_col: str = "rmse") -> float:
    valid = df_sub[[rmse_col, "n_pts"]].dropna()
    if valid.empty or valid["n_pts"].sum() == 0:
        return float("nan")
    total_sse = (valid[rmse_col] ** 2 * valid["n_pts"]).sum()
    total_pts = valid["n_pts"].sum()
    return float(np.sqrt(total_sse / total_pts))


def aggregate_results(frame_rows: List[Dict], label: str) -> Dict[str, float]:
    df = pd.DataFrame(frame_rows)
    if df.empty:
        return {}
    row: Dict[str, float] = {"config": label}
    scale_variants = ["unscaled", "median", "mean", "affine"]
    for sv in scale_variants:
        for col in ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                    "log10", "delta1", "delta2", "delta3", "mean_error"]:
            full_col = f"{sv}_{col}"
            if full_col not in df.columns:
                continue
            if col == "rmse":
                n_col = f"{sv}_n_pts"
                if n_col in df.columns:
                    tmp = df[[full_col, n_col]].rename(
                        columns={full_col: "rmse", n_col: "n_pts"})
                    row[full_col] = true_rmse(tmp)
                else:
                    row[full_col] = float(df[full_col].mean())
            else:
                row[full_col] = float(df[full_col].dropna().mean())
    return row


def compute_torch_metrics(pred, gt, mask) -> Dict[str, float]:
    pred_c = pred.clamp(EPS, MAX_DEPTH)
    mask_b = mask.bool()
    if not mask_b.any():
        return {k: float("nan") for k in
                ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    p = pred_c[mask_b].float()
    g = gt[mask_b].float().clamp(EPS)
    thresh = torch.maximum(g / p, p / g)
    return {
        "abs_rel":  float(((g - p).abs() / g).mean()),
        "sq_rel":   float(((g - p)**2 / g).mean()),
        "rmse":     float(torch.sqrt(((g - p)**2).mean())),
        "rmse_log": float(torch.sqrt((_safe_log(p) - _safe_log(g)).pow(2).mean())),
        "d1":       float((thresh < 1.25).float().mean()),
        "d2":       float((thresh < 1.25**2).float().mean()),
        "d3":       float((thresh < 1.25**3).float().mean()),
    }

print("Metric helpers ready.")

Metric helpers ready.


### 3.5 · Test-set evaluation helpers (source Section 12)

In [8]:
def index_kitti_test(test_root: str) -> pd.DataFrame:
    rows = []
    pattern = os.path.join(test_root, "*", "*_sync",
                            "proj_depth", "groundtruth", "image_02", "*.png")
    gt_paths = glob.glob(pattern)
    if not gt_paths:
        pattern = os.path.join(test_root, "*", "*",
                               "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
    for gt_path in tqdm(gt_paths, desc="Indexing test split"):
        parts = gt_path.split(os.sep)
        try:
            gt_idx = parts.index("proj_depth")
            drive  = parts[gt_idx - 1]; date = parts[gt_idx - 2]; fname = parts[-1]
        except (ValueError, IndexError):
            continue
        rgb_path = os.path.join(test_root, date, drive, "image_02", "data", fname)
        if not os.path.exists(rgb_path):
            continue
        rows.append(dict(date=date, drive=drive, filename=fname,
                         gt_path=gt_path, rgb_path=rgb_path))
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Test index: %d frames, %d drives", len(df), df["drive"].nunique())
    return df


def evaluate_model(model: nn.Module, test_df: pd.DataFrame,
                   device: torch.device,
                   config_label: str) -> List[Dict]:
    model.model.eval()
    frame_rows = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc=f"Eval: {config_label}"):
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = _read_kitti_depth(row["gt_path"])
            crop_mask = eigen_crop_mask(gt_depth.shape)
            valid_mask = ((gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & crop_mask)
            if not np.any(valid_mask):
                continue
            pred_raw = run_da3_inference(model, image_np, device)
            scale_res = compute_all_scales(gt_depth, pred_raw, valid_mask)
            frame_row = {"config": config_label}
            for sv, res in scale_res.items():
                for k, v in res.items():
                    if k not in ("scale", "shift"):
                        frame_row[f"{sv}_{k}"] = v
            frame_rows.append(frame_row)
        except Exception as e:
            log.error("Error evaluating %s: %s", row["rgb_path"], e)
    return frame_rows

print("Evaluation helpers ready.")

Evaluation helpers ready.


### 3.6 · Optimizer, scheduler, forward step, training and validation loops  *(source Section 11)*

* **Scheduler by optimizer updates**, not raw batches: `updates_per_epoch = ceil(len(loader)/ACCUM_STEPS)`; warm-up and cosine length follow `FINAL_EPOCHS`. (Only textual change: `ABLATION_EPOCHS → FINAL_EPOCHS` inside `build_scheduler`.)
* **Gradient accumulation** divides by the *actual* window size (the final partial window is flushed) and, if a batch had a non-finite loss, rescales by `window/valid_in_window` so the update is the exact mean of the finite batches.
* **Non-finite updates** are detected and counted (`diag_updates_skipped`); a run whose first epoch applies **zero** updates aborts (`FAIL_FAST_NO_UPDATES`).
* Depth and PoseNet gradients are clipped **separately** (`MAX_GRAD_NORM`), so PoseNet cannot squash the depth update.
* `prepare_step` builds exactly what the recipe's `LossSpec` declares (PoseNet poses **or** OXTS poses, the second DA3 forward for `t+1`, the dynamic mask) and hands the loss `gt=None, mask=None` unless `uses_gt_depth`.

In [9]:
def build_optimizer(param_groups, optim_mode: str = "adamw",
                    lr: float = LEARNING_RATE) -> torch.optim.Optimizer:
    # `param_groups` is a list of {"params": [...], "lr": ...} dicts (v6: depth group + pose group).
    if optim_mode == "adam":
        return torch.optim.Adam(param_groups, lr=lr)
    if optim_mode == "sgd":
        return torch.optim.SGD(param_groups, lr=lr, momentum=0.9,
                               weight_decay=WEIGHT_DECAY, nesterov=True)
    return torch.optim.AdamW(param_groups, lr=lr, weight_decay=WEIGHT_DECAY)


def updates_per_epoch_for(n_batches: int) -> int:
    # v6 FIX: the scheduler is stepped once per OPTIMIZER UPDATE, and an update happens
    # every ACCUM_STEPS batches (plus a final partial one).  v5 sized the schedule with
    # len(train_loader), making warm-up/cosine ~ACCUM_STEPS x too long.
    return max(1, math.ceil(n_batches / ACCUM_STEPS))


def build_scheduler(optimizer, updates_per_epoch: int,
                    sched_mode: str = "cosine_warmup",
                    lr: float = LEARNING_RATE):
    from torch.optim.lr_scheduler import LambdaLR, OneCycleLR, StepLR
    if sched_mode == "onecycle":
        return OneCycleLR(optimizer, max_lr=[g["lr"] for g in optimizer.param_groups],
                          steps_per_epoch=updates_per_epoch, epochs=FINAL_EPOCHS, pct_start=0.1)
    if sched_mode == "step":
        return StepLR(optimizer, step_size=max(1, FINAL_EPOCHS // 2), gamma=0.5)
    warm  = max(1, WARMUP_EPOCHS * updates_per_epoch)
    total = max(warm + 1, FINAL_EPOCHS * updates_per_epoch)

    def factor(s: int) -> float:                       # multiplies EVERY group's base lr
        if s < warm:
            return (s + 1) / warm
        prog = min(1.0, (s - warm) / max(total - warm, 1))
        return 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * prog))
    return LambdaLR(optimizer, lr_lambda=factor)


def _grad_norm(params) -> Optional[float]:
    # None = none of the parameters received a gradient; nan/inf = non-finite gradient
    sq = None
    for p in params:
        if p.grad is not None:
            g = p.grad.detach().float().pow(2).sum()
            sq = g if sq is None else sq + g
    return None if sq is None else float(sq.sqrt())


def _clip(params, max_norm):
    params = [p for p in params if p.grad is not None]
    if params and max_norm > 0:
        nn.utils.clip_grad_norm_(params, max_norm)


def _to(x, device):
    return x.to(device, non_blocking=True)


# ─────────────────────────────────────────────────────────────────────────────
# One forward pass shared by train / validate / warm-up / sanity checks
# ─────────────────────────────────────────────────────────────────────────────
def prepare_step(batch, model, pose_net, spec: "LossSpec", device,
                 epoch: int = 0, train_depth: bool = True, stationary_pairs=None, dyn_mask=None):
    """
    Runs DA3 (+ PoseNet / GT-pose / pred_next as declared by `spec`) and returns
        pred [B,1,H,W] fp32, gt, mask (evaluation targets), loss_kwargs (dict), loss_gt, loss_mask
    `loss_gt / loss_mask` are None unless spec.uses_gt_depth -> a pure-SSL loss cannot
    read LiDAR even by accident.  `gt` / `mask` themselves are None for load_gt=False (SSL-training) loaders.
    """
    rgb   = _to(batch["rgb"], device);      rgb_r = _to(batch["rgb_raw"], device)
    K     = _to(batch["K"], device);        has_K = _to(batch["has_K"], device)
    gt    = _to(batch["depth_gt"], device)   if "depth_gt"   in batch else None   # absent when load_gt=False
    mask  = _to(batch["valid_mask"], device) if "valid_mask" in batch else None
    if spec.uses_gt_depth and (gt is None or mask is None):
        raise SSLInputError("this loss uses sparse GT depth but the loader was built with load_gt=False")

    pref     = [_OFFSET_PREFIX[o] for o in SOURCE_OFFSETS]
    src_norm = [_to(batch[f"{p}_rgb"], device) for p in pref]
    src_raw  = [_to(batch[f"{p}_rgb_raw"], device) for p in pref]
    ni       = SOURCE_OFFSETS.index(+1)
    K_next, has_K_next = _to(batch["next_K"], device), _to(batch["next_has_K"], device)

    use_amp = MIXED_PREC and torch.cuda.is_available()
    with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
        with (nullcontext() if train_depth else torch.no_grad()):
            pred = forward_batch(model, rgb, K, has_K)
            pred_next = (forward_batch(model, src_norm[ni], K_next, has_K_next)
                         if spec.needs_pred_next else None)
        poses = None
        if spec.needs_pred_pose:
            if pose_net is None:
                raise SSLInputError("spec.needs_pred_pose=True but no PoseNet was built")
            poses = [pose_to_se3(pose_net(rgb, s)) for s in src_norm]
    pred = pred.float()
    pred_next = pred_next.float() if pred_next is not None else None

    gt_poses = None
    if spec.needs_gt_pose:
        if "has_gt_pose" not in batch:
            raise SSLInputError("this loss needs OXTS GT pose: use loaders_for(spec) which builds the OXTS loaders")
        assert bool(batch["has_gt_pose"].all()), "batch contains samples WITHOUT GT pose — refusing to continue"
        gt_poses = [_to(batch[f"gt_pose_{p}"], device) for p in pref]

    pose_next    = poses[ni] if poses is not None else None
    gt_pose_next = gt_poses[ni] if gt_poses is not None else None

    if (ENABLE_GEOMETRIC_FILTER and stationary_pairs is not None and pred_next is not None
            and epoch > GEOMETRIC_FILTER_WARMUP_EPOCHS and pose_next is not None):
        stationary_pairs = geometric_consistency_filter(stationary_pairs, pred.detach(),
                                                        pred_next.detach(), K, pose_next.detach())
    ctx = dict(rgb=rgb_r, rgb_srcs=src_raw, rgb_next=src_raw[ni], K=K, has_K=has_K,
               poses=poses, gt_poses=gt_poses, pred_next=pred_next,
               pose_next=pose_next, gt_pose_next=gt_pose_next,
               pose=pose_next, gt_pose=gt_pose_next,            # aliases used by the stationary losses
               stationary_pairs=stationary_pairs,
               dyn_mask=(_to(dyn_mask, device) if (dyn_mask is not None and USE_DYNAMIC_MASK) else None))
    return pred, gt, mask, ctx, (gt if spec.uses_gt_depth else None), (mask if spec.uses_gt_depth else None)


def _iter_batches(loader, needs_tracking: bool, needs_dyn: bool = False,
                  max_batches: Optional[int] = None):
    """Yields (index, batch, stationary_pairs, dyn_mask).  SAM3-based work (tracking pairs / dynamic masks) for
    batch i+1 runs in worker threads while the GPU processes batch i.  (v5 submitted the future ONCE and then
    re-read the same result for every batch, so every step used the first batch's masks.)"""
    it = iter(loader)
    want_dyn = bool(needs_dyn and USE_DYNAMIC_MASK)
    want_trk = bool(needs_tracking and ENABLE_TRACKING)

    def _next():
        try:
            b = next(it)
        except StopIteration:
            return None
        H, W = b["rgb"].shape[-2:]
        f_trk = executor.submit(get_batch_stationary_pairs, b) if want_trk else None
        f_dyn = executor.submit(get_batch_dynamic_masks, b, H, W) if want_dyn else None
        return b, f_trk, f_dyn

    cur, n = _next(), 0
    while cur is not None and (max_batches is None or n < max_batches):
        nxt = _next() if (max_batches is None or n + 1 < max_batches) else None
        yield (n, cur[0],
               cur[1].result() if cur[1] is not None else None,
               cur[2].result() if cur[2] is not None else None)
        cur, n = nxt, n + 1


def train_one_epoch(model, pose_net, loader, criterion, optimizer, scaler,
                    scheduler, device, epoch, global_step, spec: "LossSpec",
                    sched_mode: str = "cosine_warmup",
                    gpu_duty_cycle: float = 1.0,
                    train_depth: bool = True,
                    depth_params: Optional[List] = None,
                    pose_params: Optional[List] = None,
                    max_batches: Optional[int] = None) -> Tuple[int, Dict]:
    """
    One epoch.  Returns (global_step, averaged_terms).  The dict also carries health diagnostics
    (keys prefixed `diag_`): depth/pose gradient norms, skipped (non-finite) updates …
    These exist because v5 only looked at validation RMSE and never noticed that no update
    was ever applied.
    """
    depth_params = depth_params if depth_params is not None else [p for p in model.parameters() if p.requires_grad]
    pose_params  = pose_params  if pose_params  is not None else (list(pose_net.parameters()) if pose_net is not None else [])
    if train_depth:
        model.model.train()
    else:
        model.model.eval()
    if pose_net is not None and spec.needs_pred_pose:
        pose_net.train()

    n_batches = len(loader) if max_batches is None else min(len(loader), max_batches)
    running: Dict[str, float] = {}
    diag = dict(n_updates=0, n_skipped=0, n_bad_loss=0, dg_sum=0.0, pg_sum=0.0, n_gn=0)
    n_ok = 0
    valid_in_window = 0                       # finite-loss batches that contributed gradient to the CURRENT window
    optimizer.zero_grad(set_to_none=True)

    for bi, batch, stationary_pairs, dyn_mask in _iter_batches(loader, spec.needs_tracking, spec.needs_dyn_mask, max_batches):
        if bi % 50 == 0:
            log.info("    batch %d/%d", bi, n_batches)
        t0 = time.perf_counter()

        pred, gt, mask, ctx, l_gt, l_mask = prepare_step(
            batch, model, pose_net, spec, device, epoch=epoch,
            train_depth=train_depth, stationary_pairs=stationary_pairs, dyn_mask=dyn_mask)
        total, losses = _unified_forward(criterion, pred, l_gt, l_mask, **ctx)

        # size of THIS accumulation window (the last one can be shorter)
        window_start = (bi // ACCUM_STEPS) * ACCUM_STEPS
        window_size  = min(ACCUM_STEPS, n_batches - window_start)
        if torch.isfinite(total):
            scaler.scale(total / window_size).backward()     # nominal 1/window_size; corrected below if batches were skipped
            valid_in_window += 1
            n_ok += 1
            for k, v in losses.items():
                running[k] = running.get(k, 0.0) + float(v.detach() if torch.is_tensor(v) else v)
        else:
            diag["n_bad_loss"] += 1                       # non-finite loss -> skip this batch entirely

        is_last = (bi + 1 == n_batches)
        if (bi + 1) % ACCUM_STEPS == 0 or is_last:
            has_grads = any(p.grad is not None for p in (depth_params + pose_params))
            if has_grads and USE_GRAD_SCALER:
                scaler.unscale_(optimizer)
            dn = _grad_norm(depth_params) if (train_depth and has_grads) else None
            pn = _grad_norm(pose_params) if has_grads else None
            finite = has_grads and all(math.isfinite(x) for x in (dn, pn) if x is not None)
            if finite and 0 < valid_in_window < window_size:
                # v6.2: some batches of this window had a non-finite loss and contributed nothing; the gradient is
                # the sum of `valid_in_window` terms each divided by `window_size` -> rescale to a true mean.
                # (grads are already unscaled here when the GradScaler is active, so this is scale-safe)
                _corr = window_size / valid_in_window
                for _p in depth_params + pose_params:
                    if _p.grad is not None:
                        _p.grad.mul_(_corr)
                dn = None if dn is None else dn * _corr
                pn = None if pn is None else pn * _corr
            if finite:
                if train_depth: _clip(depth_params, MAX_GRAD_NORM)     # clipped per group: PoseNet gradients
                _clip(pose_params, MAX_GRAD_NORM)                      # can no longer squash the depth update
                scaler.step(optimizer); scaler.update()
                diag["n_updates"] += 1
                diag["dg_sum"] += dn or 0.0
                diag["pg_sum"] += pn or 0.0
                diag["n_gn"] += 1
            else:
                if has_grads and USE_GRAD_SCALER:
                    scaler.step(optimizer); scaler.update()            # scaler skips + lowers its scale
                diag["n_skipped"] += 1
            optimizer.zero_grad(set_to_none=True)
            valid_in_window = 0
            if scheduler is not None and sched_mode in ("cosine_warmup", "onecycle"):
                scheduler.step()
            global_step += 1

        if gpu_duty_cycle < 1.0:
            torch.cuda.synchronize(device)
            t_sleep = (time.perf_counter() - t0) * (1.0 / gpu_duty_cycle - 1.0)
            if t_sleep > 0:
                time.sleep(t_sleep)

    if scheduler is not None and sched_mode == "step":
        scheduler.step()

    avg = {k: v / max(n_ok, 1) for k, v in running.items()}
    tot_upd = diag["n_updates"] + diag["n_skipped"]
    avg.update({
        "diag_updates_applied":  diag["n_updates"],
        "diag_updates_skipped":  diag["n_skipped"],
        "diag_skip_frac":        diag["n_skipped"] / max(tot_upd, 1),
        "diag_bad_loss_batches": diag["n_bad_loss"],
        "diag_depth_grad_norm":  diag["dg_sum"] / max(diag["n_gn"], 1) if train_depth else float("nan"),
        "diag_pose_grad_norm":   diag["pg_sum"] / max(diag["n_gn"], 1) if pose_params else float("nan"),
    })
    if diag["n_skipped"] or diag["n_bad_loss"]:
        log.warning("    [health] %d/%d optimizer updates SKIPPED (non-finite grads), %d batches with non-finite loss",
                    diag["n_skipped"], tot_upd, diag["n_bad_loss"])
    return global_step, avg


@torch.no_grad()
def validate(model, pose_net, loader, criterion, device, epoch,
             spec: "LossSpec", max_batches: Optional[int] = None) -> Dict[str, float]:
    model.model.eval()
    if pose_net is not None:
        pose_net.eval()
    all_m: List[Dict] = []
    total_loss, n_batches = 0.0, 0
    for bi, batch, stationary_pairs, dyn_mask in _iter_batches(loader, spec.needs_tracking, spec.needs_dyn_mask, max_batches):
        pred, gt, mask, ctx, l_gt, l_mask = prepare_step(
            batch, model, pose_net, spec, device, epoch=epoch,
            train_depth=False, stationary_pairs=stationary_pairs, dyn_mask=dyn_mask)
        assert gt is not None and mask is not None, "validation loader must be built with load_gt=True"
        loss, _ = _unified_forward(criterion, pred, l_gt, l_mask, **ctx)
        total_loss += float(loss); n_batches += 1
        all_m.append(compute_torch_metrics(pred, gt, mask))       # evaluation only — GT is used HERE, not in the loss

    def _avg(k):
        return float(np.nanmean([x[k] for x in all_m]))

    agg = {k: _avg(k) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    agg["val_loss"] = total_loss / max(n_batches, 1)
    return agg


print("Training / validation loops ready (v6: spec-driven routing, fp32 losses, gradient diagnostics).")

Training / validation loops ready (v6: spec-driven routing, fp32 losses, gradient diagnostics).


## 4 · Dataset (multi-frame sequences t−1, t, t+1)  *(copied verbatim)*

* Horizontal flip and colour jitter are drawn **once per sample** and applied to every frame; on a flip the principal point mirrors (`cx → W−1−cx`) — v5 flipped the images but not `K`.
* `load_gt=False` for all six training loaders ⇒ **no LiDAR is read, decoded or transferred during training.**

In [10]:
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]
_train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])

_DEFAULT_K = np.array([
    [721.5377,   0.0,   609.5593],
    [  0.0,   721.5377, 172.8540],
    [  0.0,     0.0,     1.0  ],
], dtype=np.float32)

_OFFSET_PREFIX = {+1: "next", -1: "prev"}          # batch-key prefix per source offset


def _read_kitti_depth(path: str) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.uint16)
    return raw.astype(np.float32) / 256.0


def _parse_kitti_calib(date_dir: Path, drive_dir: Path) -> Optional[np.ndarray]:
    calib: Dict[str, np.ndarray] = {}
    for fname in ["calib_cam_to_cam.txt"]:
        for search in [date_dir, drive_dir]:
            p = search / fname
            if p.exists():
                for line in p.read_text().splitlines():
                    if ":" in line:
                        k, v = line.split(":", 1)
                        try:
                            calib[k.strip()] = np.array(v.split(), dtype=np.float64)
                        except Exception:
                            pass
    K = None
    try:
        if "P_rect_02" in calib:
            P = calib["P_rect_02"].reshape(3, 4); K = P[:3, :3].copy()
        elif "P2" in calib:
            P = calib["P2"].reshape(3, 4); K = P[:3, :3].copy()
    except Exception:
        pass
    return K


# ─── Augmentation (v6) ───────────────────────────────────────────────────────
# v5 flipped the images but left K's principal point (and, for OXTS runs, the
# GT pose) un-flipped -> wrong geometry on 50 % of the samples.  v6 draws the
# flip / colour-jitter parameters ONCE per sample, applies them to every frame
# of the sample, and updates K (and gt_pose, see the OXTS dataset) accordingly.
# Random crops stay excluded (they move the principal point).
#                   (flip, colour-jitter)
_AUG_SPECS = {"none": (False, False), "flip": (True, False),
              "base": (True, True),   "full": (True, True)}
AUG_FNS = _AUG_SPECS      # kept for backward-compat with prints in later cells


def _draw_aug_params(aug_mode: str, rng: np.random.RandomState) -> Dict:
    do_flip, do_jit = _AUG_SPECS.get(aug_mode, (False, False))
    flip = bool(do_flip and rng.rand() < 0.5)
    br = rng.uniform(0.8, 1.2) if do_jit else 1.0
    co = rng.uniform(0.8, 1.2) if do_jit else 1.0
    return dict(flip=flip, brightness=br, contrast=co, jitter=do_jit)


def _apply_color_jitter(rgb: np.ndarray, br: float, co: float) -> np.ndarray:
    f = rgb.astype(np.float32) / 255.0
    f = np.clip((f - f.mean()) * co + f.mean() * br, 0.0, 1.0)
    return (f * 255.0).astype(np.uint8)


# ─── Sequential multi-frame dataset (required for ALL SSL losses) ────────────
class KITTISequentialDepthDataset(Dataset):
    """
    Emits (I_t and its source frames I_{t+off}) from one KITTI drive, for every
    off in `source_offsets` (default (-1, +1) = Monodepth2 3-frame training).

    Batch keys
        current frame : rgb, rgb_raw, depth_gt, valid_mask, K, has_K, rgb_path, depth_path
        next   (+1)   : next_rgb, next_rgb_raw, next_depth_gt, next_valid_mask, next_K, ...
        prev   (-1)   : prev_rgb, prev_rgb_raw, prev_K, ...            (no GT depth is loaded)
        flipped       : bool  — whether the sample was horizontally flipped

    GT depth of the CURRENT frame (and +1 frame) is loaded only when load_gt=True
    (validation metrics / supervised controls).  Pure-SSL TRAINING uses load_gt=False,
    so no LiDAR is read, decoded or transferred; the trainer additionally passes
    gt=None to every loss that does not declare uses_gt_depth.
    """

    def __init__(self, root, split="train",
                 target_hw=(TARGET_H, TARGET_W),
                 min_depth=MIN_DEPTH, max_depth=MAX_DEPTH,
                 aug_mode="base", max_samples=0, seed=42,
                 source_offsets=SOURCE_OFFSETS,
                 load_gt: bool = True,
                 allowed_paths: Optional[set] = None):
        self.root      = Path(root)
        self.split     = split
        self.target_hw = target_hw
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.aug_mode  = aug_mode
        self.augment   = (split == "train" and aug_mode != "none")
        self.rng       = np.random.RandomState(seed)
        self.offsets   = tuple(source_offsets)
        self.load_gt   = load_gt          # False for pure-SSL training: no LiDAR I/O / decode / transfer
        assert +1 in self.offsets, "source_offsets must contain +1"
        assert all(o in _OFFSET_PREFIX for o in self.offsets), \
            f"only offsets {list(_OFFSET_PREFIX)} are supported, got {self.offsets}"
        self._samples: List[Dict[int, Dict]] = []       # {0: rec, +1: rec, -1: rec}
        self._build_samples()
        if allowed_paths is not None:     # canonical sample set (identical ids for SSL and oracle-pose runs)
            self._samples = [s for s in self._samples if s[0]["rgb_path"] in allowed_paths]
        if max_samples > 0 and len(self._samples) > max_samples:
            idx = self.rng.choice(len(self._samples), max_samples, replace=False)
            self._samples = [self._samples[i] for i in sorted(idx)]
        log.info("[KITTI-Seq-%s] %d samples (offsets=%s, aug=%s)",
                 split, len(self._samples), self.offsets, aug_mode)

    # -- index ---------------------------------------------------------------
    def _build_drive_index(self) -> Dict:
        depth_glob = str(self.root / "*" / "*_sync" /
                         "proj_depth" / "groundtruth" / "image_02" / "*.png")
        depth_paths = sorted(glob.glob(depth_glob))
        if not depth_paths:
            depth_glob = str(self.root / "*" / "*" /
                             "proj_depth" / "groundtruth" / "image_02" / "*.png")
            depth_paths = sorted(glob.glob(depth_glob))

        by_drive: Dict = {}
        for dp in depth_paths:
            parts = Path(dp).parts
            try:
                gi        = list(parts).index("proj_depth")
                drive_dir = Path(*parts[:gi])
                date_dir  = drive_dir.parent
                fname     = parts[-1]
            except (ValueError, IndexError):
                continue
            rgb_path = drive_dir / "image_02" / "data" / fname
            if not rgb_path.exists():
                continue
            K = _parse_kitti_calib(date_dir, drive_dir)
            key = (date_dir.name, drive_dir.name)
            rec = dict(rgb_path=str(rgb_path), depth_path=dp, K=K,
                       frame_id=int(Path(fname).stem))
            by_drive.setdefault(key, []).append(rec)

        for key in by_drive:
            by_drive[key].sort(key=lambda r: r["frame_id"])
        return by_drive

    def _build_samples(self):
        # A sample exists only if EVERY required source frame is present and
        # frame ids are consecutive (offset == frame-id difference).
        for frames in self._build_drive_index().values():
            by_id = {r["frame_id"]: r for r in frames}
            for r in frames:
                fid = r["frame_id"]
                if all((fid + off) in by_id for off in self.offsets):
                    s = {0: r}
                    for off in self.offsets:
                        s[off] = by_id[fid + off]
                    self._samples.append(s)

    # -- loading -------------------------------------------------------------
    def _load_frame(self, rec: Dict, load_depth: bool = True) -> Dict:
        H, W    = self.target_hw
        rgb     = cv2.cvtColor(cv2.imread(rec["rgb_path"]), cv2.COLOR_BGR2RGB)
        orig_hw = rgb.shape[:2]
        rgb     = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR)

        depth = valid = None
        if load_depth:
            depth   = _read_kitti_depth(rec["depth_path"])
            valid_o = (depth > self.min_depth) & (depth < self.max_depth)
            depth = cv2.resize(depth, (W, H), interpolation=cv2.INTER_NEAREST)
            valid = cv2.resize(valid_o.astype(np.uint8), (W, H),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
            depth = np.clip(depth, 0.0, self.max_depth)
            eigen_crop = np.zeros((H, W), dtype=bool)          # Eigen crop rows 44:396
            eigen_crop[44:min(44 + 352, H), :] = True
            valid = valid & eigen_crop

        K = rec["K"]
        if K is not None:
            K = K.copy()
            K[0] *= W / orig_hw[1]
            K[1] *= H / orig_hw[0]
        return dict(rgb_np=rgb, depth=depth, valid=valid, K=K,
                    rgb_path=rec["rgb_path"], depth_path=rec["depth_path"],
                    frame_id=rec["frame_id"])

    def _to_tensors(self, frame: Dict) -> Dict:
        rgb_t   = _train_transform(Image.fromarray(frame["rgb_np"]))
        rgb_raw = torch.from_numpy(frame["rgb_np"]).permute(2, 0, 1).float() / 255.0
        K = frame["K"]
        K_t   = torch.from_numpy(K).float() if K is not None else torch.from_numpy(_DEFAULT_K.copy())
        has_K = torch.tensor(K is not None, dtype=torch.bool)
        out = {"rgb": rgb_t, "rgb_raw": rgb_raw, "K": K_t, "has_K": has_K,
               "rgb_path": frame["rgb_path"]}
        if frame["depth"] is not None:
            out["depth_gt"]   = torch.from_numpy(frame["depth"]).unsqueeze(0).float()
            out["valid_mask"] = torch.from_numpy(frame["valid"].astype(np.float32)).unsqueeze(0)
            out["depth_path"] = frame["depth_path"]
        return out

    def _flip_frame(self, frame: Dict) -> None:
        W = self.target_hw[1]
        frame["rgb_np"] = np.ascontiguousarray(frame["rgb_np"][:, ::-1])
        if frame["depth"] is not None:
            frame["depth"] = np.ascontiguousarray(frame["depth"][:, ::-1])
            frame["valid"] = np.ascontiguousarray(frame["valid"][:, ::-1])
        if frame["K"] is not None:                               # principal point mirrors
            frame["K"][0, 2] = (W - 1) - frame["K"][0, 2]

    def __len__(self):
        return len(self._samples)

    def _aug_params(self) -> Dict:
        if not self.augment:
            return dict(flip=False, brightness=1.0, contrast=1.0, jitter=False)
        rng = np.random.RandomState(int.from_bytes(os.urandom(4), "little"))
        return _draw_aug_params(self.aug_mode, rng)

    def __getitem__(self, idx):
        recs = self._samples[idx]
        frames = {0: self._load_frame(recs[0], load_depth=self.load_gt)}
        for off in self.offsets:
            frames[off] = self._load_frame(recs[off], load_depth=(self.load_gt and off == +1))

        ap = self._aug_params()
        for fr in frames.values():                               # IDENTICAL params for every frame
            if ap["flip"]:
                self._flip_frame(fr)
            if ap["jitter"]:
                fr["rgb_np"] = _apply_color_jitter(fr["rgb_np"], ap["brightness"], ap["contrast"])
        if self.augment:                                         # keep valid ⊂ (depth in range)
            for fr in frames.values():
                if fr["depth"] is not None:
                    fr["valid"] = (fr["depth"] > self.min_depth) & (fr["depth"] < self.max_depth) & fr["valid"]

        result = self._to_tensors(frames[0])
        for off in self.offsets:
            t = self._to_tensors(frames[off])
            result.update({f"{_OFFSET_PREFIX[off]}_{k}": v for k, v in t.items()})
        result["flipped"] = torch.tensor(ap["flip"], dtype=torch.bool)
        return result


def collate_fn(batch):
    from torch.utils.data.dataloader import default_collate
    tensor_keys = [k for k in batch[0] if isinstance(batch[0][k], torch.Tensor)]
    str_keys    = [k for k in batch[0] if isinstance(batch[0][k], str)]
    out = {k: default_collate([b[k] for b in batch]) for k in tensor_keys}
    out.update({k: [b[k] for b in batch] for k in str_keys})
    return out


print("Dataset helpers ready (multi-frame sequences).")
print(f"  KITTISequentialDepthDataset -> (I_t, sources at offsets {SOURCE_OFFSETS})")
print("  flip / colour-jitter are drawn once per sample; K is mirrored on flip.")

Dataset helpers ready (multi-frame sequences).
  KITTISequentialDepthDataset -> (I_t, sources at offsets (-1, 1))
  flip / colour-jitter are drawn once per sample; K is mirrored on flip.


## 5 · Pose / OXTS utilities  *(copied verbatim)*

OXTS poses are converted to the **camera-02 frame** by *conjugation*:

```
T_cam = T_ic · inv(W_j) · W_i · inv(T_ic)          (p_cam_j = T_cam · p_cam_i)
```

(v5 used `inv(T_ic·W_j)·T_ic·W_i`, in which `T_ic` cancels — the pose stayed in IMU axes.) There is **no silent fallback**: samples without OXTS data or calibration are dropped, the trainer asserts `has_gt_pose.all()`, and on a horizontal flip the pose becomes `T' = F·T·F`, `F = diag(−1,1,1,1)`.

OXTS is used **only** by A1 / A3 / B1, and only as camera motion — never as depth supervision.

In [11]:
import math

# =============================================================================
# OXTS / GPS-IMU utilities
# =============================================================================
# Reference: KITTI raw data devkit (A. Geiger, P. Lenz, C. Stiller, R. Urtasun)
# Each oxts/data/NNNNNNNNNN.txt has 30 space-separated floats in the order
# defined by dataformat.txt (lat, lon, alt, roll, pitch, yaw, ...).
# =============================================================================

_EARTH_RADIUS_M = 6378137.0    # WGS-84 semi-major axis


def _latlon_to_mercator(lat_deg: float, lon_deg: float, scale: float) -> Tuple[float, float]:
    """Flat-earth Mercator projection: (lat, lon) in degrees -> (mx, my) in metres."""
    mx = scale * math.radians(lon_deg) * _EARTH_RADIUS_M
    my = scale * _EARTH_RADIUS_M * math.log(
        math.tan(math.pi / 4.0 + math.radians(lat_deg) / 2.0))
    return mx, my


def _rpy_to_rotmat_np(roll: float, pitch: float, yaw: float) -> np.ndarray:
    """Z-Y-X Euler angles -> rotation matrix R = Rz(yaw) @ Ry(pitch) @ Rx(roll)."""
    cr, sr = math.cos(roll),  math.sin(roll)
    cp, sp = math.cos(pitch), math.sin(pitch)
    cy, sy = math.cos(yaw),   math.sin(yaw)
    return np.array([
        [ cy*cp,  cy*sp*sr - sy*cr,  cy*sp*cr + sy*sr],
        [ sy*cp,  sy*sp*sr + cy*cr,  sy*sp*cr - cy*sr],
        [-sp,     cp*sr,             cp*cr            ],
    ], dtype=np.float64)


def _parse_oxts_file(path: str) -> Optional[List[float]]:
    """Parse one OXTS data file -> list of 30 floats, or None on error."""
    try:
        vals = [float(v) for v in Path(path).read_text().strip().split()]
        return vals if len(vals) >= 6 else None
    except Exception:
        return None


def _oxts_packet_to_world_se3(pkt: List[float], scale: float) -> np.ndarray:
    """Convert one OXTS packet to a 4x4 SE(3) in flat-earth Mercator world frame."""
    lat, lon, alt, roll, pitch, yaw = pkt[:6]
    mx, my = _latlon_to_mercator(lat, lon, scale)
    R = _rpy_to_rotmat_np(roll, pitch, yaw)
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = R
    T[:3,  3] = [mx, my, alt]
    return T


def _load_imu_to_cam02(date_dir: Path) -> Optional[np.ndarray]:
    """
    Compose T_cam02_from_imu from KITTI calibration files found at <date_dir>/:
      calib_imu_to_velo.txt  -> Tr_imu_to_velo  [4x4]
      calib_velo_to_cam.txt  -> Tr_velo_to_cam0 [4x4]
      calib_cam_to_cam.txt   -> R_rect_00 + P_rect_02

    Full chain (maps a point given in IMU coordinates to cam02 coordinates):
      T_cam02_from_imu = T_02_offset @ R_rect_00 @ Tr_velo_to_cam0 @ Tr_imu_to_velo

    Returns None if any file is absent or cannot be parsed.
    """
    def _kv(path: Path) -> Dict[str, np.ndarray]:
        d: Dict[str, np.ndarray] = {}
        for line in path.read_text().splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                try:
                    d[k.strip()] = np.array(v.split(), dtype=np.float64)
                except Exception:
                    pass
        return d

    try:
        f_iv = date_dir / "calib_imu_to_velo.txt"
        f_vc = date_dir / "calib_velo_to_cam.txt"
        f_cc = date_dir / "calib_cam_to_cam.txt"
        if not all(f.exists() for f in [f_iv, f_vc, f_cc]):
            return None

        kv_iv = _kv(f_iv); kv_vc = _kv(f_vc); kv_cc = _kv(f_cc)

        # Tr_imu->velo [4x4]
        Tr_iv = np.eye(4, dtype=np.float64)
        Tr_iv[:3, :3] = kv_iv["R"].reshape(3, 3)
        Tr_iv[:3,  3] = kv_iv["T"].ravel()

        # Tr_velo->cam0 [4x4]
        Tr_vc = np.eye(4, dtype=np.float64)
        Tr_vc[:3, :3] = kv_vc["R"].reshape(3, 3)
        Tr_vc[:3,  3] = kv_vc["T"].ravel()

        # R_rect_00 (rectification of cam0)
        R_rect = np.eye(4, dtype=np.float64)
        R_rect[:3, :3] = kv_cc["R_rect_00"].reshape(3, 3)

        # T offset from cam0 to cam02 (from P_rect_02 baseline)
        P02 = kv_cc["P_rect_02"].reshape(3, 4)
        tx_offset = P02[0, 3] / (P02[0, 0] + 1e-12)
        T_02_offset = np.eye(4, dtype=np.float64)
        T_02_offset[0, 3] = tx_offset

        return T_02_offset @ R_rect @ Tr_vc @ Tr_iv

    except Exception as e:
        log.debug("[OXTS calib] %s: %s", date_dir, e)
        return None


def _load_oxts_indexed(oxts_data_dir: Path) -> Dict[int, np.ndarray]:
    """
    Load all OXTS frames from <drive>/oxts/data/ and return {frame_id: W_k}
    where W_k is the 4x4 pose of the IMU at frame k in the drive-local world
    frame (IMU_k -> IMU_0, i.e.  p_imu0 = W_k @ p_imu_k).  Everything here is
    in the IMU body frame (x fwd, y left, z up); the conversion to the CAMERA
    frame happens in `_relative_cam_pose`.

    v6 FIX (found while auditing the GT-pose path): v5 multiplied the local
    poses by T_imu_to_cam and then formed inv(P_j) @ P_i.  The calibration
    matrix cancels algebraically in that product, so v5's "camera-frame"
    relative pose was really the relative pose in the IMU frame (x fwd) while
    `warp_image` expects the camera frame (z fwd).  See `_relative_cam_pose`.
    """
    txt_files = sorted(glob.glob(str(oxts_data_dir / "*.txt")))
    if not txt_files:
        return {}
    frame_ids, packets = [], []
    for fpath in txt_files:
        pkt = _parse_oxts_file(fpath)
        if pkt is not None:
            frame_ids.append(int(Path(fpath).stem))
            packets.append(pkt)
    if not packets:
        return {}
    scale = math.cos(math.radians(packets[0][0]))                  # Mercator scale from 1st latitude
    world = [_oxts_packet_to_world_se3(p, scale) for p in packets]
    T0_inv = np.linalg.inv(world[0])
    return {fid: T0_inv @ W for fid, W in zip(frame_ids, world)}


def _relative_cam_pose(W_i: np.ndarray, W_j: np.ndarray, T_imu_to_cam: np.ndarray) -> np.ndarray:
    """
    Relative pose  T_{i->j}  in the CAMERA-02 frame:  p_cam_j = T @ p_cam_i.

        p_imu_j = inv(W_j) @ W_i @ p_imu_i          (IMU frame, from OXTS)
        p_cam_k = T_ic @ p_imu_k                    (T_ic = T_imu_to_cam)
      =>  T_cam = T_ic @ inv(W_j) @ W_i @ inv(T_ic)  (a CONJUGATION, not a product)

    This is exactly the `pose_curr_to_next` convention consumed by
    `Project3D` / `warp_image` (maps coordinates of frame i into frame j).
    """
    T_imu = np.linalg.inv(W_j) @ W_i
    return T_imu_to_cam @ T_imu @ np.linalg.inv(T_imu_to_cam)


_FLIP_X = np.diag([-1.0, 1.0, 1.0, 1.0]).astype(np.float64)


def _flip_pose(T: np.ndarray) -> np.ndarray:
    """Pose of the mirrored scene under a horizontal image flip: T' = F T F, F = diag(-1,1,1,1)."""
    return _FLIP_X @ T @ _FLIP_X


# =============================================================================
# Dataset subclass: KITTISequentialOXTSDataset  (ORACLE-POSE experiments only)
# =============================================================================

class KITTISequentialOXTSDataset(KITTISequentialDepthDataset):
    """
    Adds GPS/IMU ground-truth relative poses (camera-02 frame, `pose_curr_to_src`)
    to every sample of KITTISequentialDepthDataset.

    Extra batch keys
        gt_pose_next : [4,4]  T_{t -> t+1}
        gt_pose_prev : [4,4]  T_{t -> t-1}          (only if -1 in source_offsets)
        has_gt_pose  : bool   (always True when require_gt_pose=True)

    v6 behaviour
      * There is NO silent identity/None fallback.  With `require_gt_pose=True`
        (default) samples whose OXTS pose or calibration is missing are DROPPED
        (and counted in the log) so that every remaining sample has GT pose.
      * The trainer additionally asserts `has_gt_pose.all()`.
      * If a drive has no calibration files, its samples are dropped — the old
        "IMU frame (approx)" behaviour is gone because it was in the wrong axes.
      * On a horizontal flip the pose is mirrored (T' = F T F).
    """

    def __init__(self, *args, oxts_subdir: str = "oxts", require_gt_pose: bool = True, **kwargs):
        self._oxts_subdir     = oxts_subdir
        self._require_gt_pose = require_gt_pose
        self._drive_world: Dict[Tuple, Dict[int, np.ndarray]] = {}
        self._drive_Tic:   Dict[Tuple, np.ndarray] = {}
        self._n_dropped = 0
        super().__init__(*args, **kwargs)          # calls _build_samples via polymorphism
        log.info("[OXTSDataset] %d drives with GT pose | %d samples kept | %d dropped (no GT pose)",
                 len(self._drive_world), len(self._samples), self._n_dropped)

    def _build_drive_index(self) -> Dict:
        by_drive = super()._build_drive_index()
        for (date, drive_name), frames in by_drive.items():
            if not frames:
                continue
            rgb_0 = Path(frames[0]["rgb_path"])
            try:
                parts = list(rgb_0.parts)
                img02_idx = parts.index("image_02")
                drive_dir = Path(*parts[:img02_idx])
                date_dir  = drive_dir.parent
            except (ValueError, IndexError):
                continue
            oxts_data_dir = drive_dir / self._oxts_subdir / "data"
            if not oxts_data_dir.exists():
                log.warning("[OXTS] no oxts dir for %s/%s", date, drive_name)
                continue
            T_ic = _load_imu_to_cam02(date_dir)
            if T_ic is None:
                log.warning("[OXTS] calibration files missing for %s -> drive %s excluded from oracle runs",
                            date, drive_name)
                continue
            try:
                self._drive_world[(date, drive_name)] = _load_oxts_indexed(oxts_data_dir)
                self._drive_Tic[(date, drive_name)]   = T_ic
            except Exception as e:
                log.warning("[OXTS] failed %s/%s: %s", date, drive_name, e)
        return by_drive

    def _build_samples(self):
        for (date, drive_name), frames in self._build_drive_index().items():
            world = self._drive_world.get((date, drive_name), {})
            T_ic  = self._drive_Tic.get((date, drive_name))
            by_id = {r["frame_id"]: r for r in frames}
            for r in frames:
                fid = r["frame_id"]
                if not all((fid + off) in by_id for off in self.offsets):
                    continue
                gt = {}
                ok = (T_ic is not None) and (fid in world)
                if ok:
                    for off in self.offsets:
                        if (fid + off) not in world:
                            ok = False
                            break
                        gt[off] = _relative_cam_pose(world[fid], world[fid + off], T_ic)
                if not ok and self._require_gt_pose:
                    self._n_dropped += 1
                    continue
                s = {0: r}
                for off in self.offsets:
                    s[off] = by_id[fid + off]
                s["gt"] = gt if ok else None
                self._samples.append(s)

    def __getitem__(self, idx: int) -> Dict:
        result = super().__getitem__(idx)
        gt = self._samples[idx].get("gt")
        flipped = bool(result["flipped"])
        if gt is None:
            # only reachable with require_gt_pose=False -> explicit flag, NO fake pose is used by the trainer
            for off in self.offsets:
                result[f"gt_pose_{_OFFSET_PREFIX[off]}"] = torch.eye(4, dtype=torch.float32)
            result["has_gt_pose"] = torch.tensor(False, dtype=torch.bool)
            return result
        for off in self.offsets:
            T = gt[off]
            T = _flip_pose(T) if flipped else T
            result[f"gt_pose_{_OFFSET_PREFIX[off]}"] = torch.from_numpy(T.astype(np.float32))
        result["has_gt_pose"] = torch.tensor(True, dtype=torch.bool)
        return result


def sample_ids(ds) -> List[str]:
    """Canonical id of every sample = RGB path of the CURRENT frame."""
    return [s[0]["rgb_path"] for s in ds._samples]


def assert_same_samples(ds_a, idx_a, ds_b, idx_b, what: str = "") -> None:
    """Oracle-pose and predicted-pose runs are only comparable (B vs C) if they see EXACTLY the same images."""
    ids_a = [sample_ids(ds_a)[i] for i in idx_a]
    ids_b = [sample_ids(ds_b)[i] for i in idx_b]
    assert ids_a == ids_b, (f"{what}: the SSL and oracle-pose datasets do not contain identical samples "
                            f"({len(ids_a)} vs {len(ids_b)}). Set CANONICAL_SAMPLE_SET=True.")


def probe_oxts_sample_ids(train_root: str) -> set:
    """Ids of all samples that have complete OXTS GT pose + calibration (the canonical sample set)."""
    probe = KITTISequentialOXTSDataset(train_root, split="val", aug_mode="none", max_samples=0,
                                       load_gt=False, require_gt_pose=True)
    return set(sample_ids(probe))


print("OXTS utilities ready (v6).")
print("  _relative_cam_pose          -> T_cam = T_ic @ inv(W_j) @ W_i @ inv(T_ic)   (frame-correct)")
print("  KITTISequentialOXTSDataset  -> gt_pose_next / gt_pose_prev, no silent fallback")
print("  probe_oxts_sample_ids / assert_same_samples -> canonical sample set shared with the SSL loaders")

OXTS utilities ready (v6).
  _relative_cam_pose          -> T_cam = T_ic @ inv(W_j) @ W_i @ inv(T_ic)   (frame-correct)
  KITTISequentialOXTSDataset  -> gt_pose_next / gt_pose_prev, no silent fallback
  probe_oxts_sample_ids / assert_same_samples -> canonical sample set shared with the SSL loaders


### 5b · Canonical sample set + data loaders (source Section 13a)

With `CANONICAL_SAMPLE_SET=True` every loader — for the OXTS recipes *and* the predicted-pose recipes — uses exactly the samples that have complete OXTS pose + calibration and **one shared train/val index split** (asserted). All six recipes therefore train on the same images ("Design B" of the source). The test set (`TEST_ROOT`) is indexed here but is touched **only after training**, for the final evaluation.

> **Population note:** "full available dataset" means *all* canonical (OXTS-complete) samples (`MAX_TRAIN = 0`), minus the 10 % validation split.

In [12]:
# ════════════════════════════════════════════════════════════════════════════════
# Build train / val / test data (multi-frame sequences)
# ════════════════════════════════════════════════════════════════════════════════
# v6.1 design
#   * ONE canonical list of samples.  With CANONICAL_SAMPLE_SET=True, only samples that have complete OXTS
#     pose + calibration are used — by the SSL loaders AND the oracle-pose loaders — and the train/val
#     index split is shared, so "oracle pose (B) vs predicted pose (C)" differ ONLY in where the pose comes
#     from (same images; augmentation is random but identically distributed).
#   * Pure-SSL TRAINING loaders are built with load_gt=False (no LiDAR read / decoded / transferred);
#     validation loaders and supervised / hybrid controls use load_gt=True.
_canonical_ids = None
if CANONICAL_SAMPLE_SET:
    try:
        _canonical_ids = probe_oxts_sample_ids(TRAIN_ROOT) or None
        log.info("Canonical sample set: %s samples with complete OXTS pose + calibration",
                 f"{len(_canonical_ids):,}" if _canonical_ids else "0 (-> falling back to all samples)")
    except Exception as e:
        log.warning("Could not probe the OXTS sample set (%s) — SSL and oracle loaders will NOT be forced identical.", e)

_val_ds = KITTISequentialDepthDataset(
    TRAIN_ROOT, split="val", aug_mode="none", max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED,
    load_gt=True, allowed_paths=_canonical_ids)
n_total = len(_val_ds)
_rng    = np.random.RandomState(VAL_SPLIT_SEED)
all_idx = _rng.permutation(n_total)
n_val   = max(1, int(n_total * VAL_SPLIT_FRAC))
_val_idx = sorted(all_idx[:n_val].tolist())
_tr_idx  = sorted(all_idx[n_val:].tolist())

_val_kw   = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
                 prefetch_factor=2, collate_fn=collate_fn, timeout=120)
_train_kw = dict(_val_kw, persistent_workers=False)      # several train loaders may exist -> don't pin idle workers
_loader_kw = _train_kw                                    # (alias used by older cells)
_val_loader = DataLoader(Subset(_val_ds, _val_idx), batch_size=BATCH_SIZE, shuffle=False, **_val_kw)

def _check_same_samples(ds_other, what: str) -> None:
    same = sample_ids(_val_ds) == sample_ids(ds_other)
    if CANONICAL_SAMPLE_SET and _canonical_ids is not None:
        assert same, f"{what}: sample list differs from the SSL sample list — B-vs-C comparison would be invalid"
    elif not same:
        log.warning("%s: sample list differs from the SSL sample list (CANONICAL_SAMPLE_SET off / probe failed) — "
                    "oracle-vs-predicted comparisons are NOT like-for-like.", what)

_train_loader_cache: Dict[Tuple, DataLoader] = {}
def make_train_loader(aug_mode: str = "base", load_gt: bool = False, oracle: bool = False) -> DataLoader:
    key = (aug_mode, load_gt, oracle)
    if key not in _train_loader_cache:
        cls_kw = dict(split="train", aug_mode=aug_mode, max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED,
                      load_gt=load_gt, allowed_paths=_canonical_ids)
        ds = (KITTISequentialOXTSDataset(TRAIN_ROOT, require_gt_pose=True, **cls_kw) if oracle
              else KITTISequentialDepthDataset(TRAIN_ROOT, **cls_kw))
        assert len(ds) == n_total, f"train dataset has {len(ds)} samples but the val dataset {n_total}"
        _check_same_samples(ds, f"train loader {key}")
        _train_loader_cache[key] = DataLoader(Subset(ds, _tr_idx), batch_size=BATCH_SIZE, shuffle=True,
                                              drop_last=True, **_train_kw)
    return _train_loader_cache[key]

_oracle_val_cache: Dict[str, DataLoader] = {}
def get_oracle_val_loader() -> DataLoader:
    if "v" not in _oracle_val_cache:
        ds = KITTISequentialOXTSDataset(TRAIN_ROOT, split="val", aug_mode="none", max_samples=MAX_TRAIN,
                                        seed=VAL_SPLIT_SEED, load_gt=True, require_gt_pose=True,
                                        allowed_paths=_canonical_ids)
        assert len(ds) == n_total, f"oracle val dataset has {len(ds)} samples but the SSL val dataset {n_total}"
        _check_same_samples(ds, "oracle val loader")
        _oracle_val_cache["v"] = DataLoader(Subset(ds, _val_idx), batch_size=BATCH_SIZE, shuffle=False, **_val_kw)
    return _oracle_val_cache["v"]

def loaders_for(spec: "LossSpec", aug_mode: str = "base") -> Tuple[DataLoader, DataLoader]:
    """(train, val) loaders for a loss: OXTS loaders only for oracle-pose losses; LiDAR loaded only when the
    loss really uses it.  Validation always has LiDAR (evaluation only)."""
    train = make_train_loader(aug_mode, load_gt=spec.uses_gt_depth, oracle=spec.needs_gt_pose)
    val   = get_oracle_val_loader() if spec.needs_gt_pose else _val_loader
    return train, val

test_df = index_kitti_test(TEST_ROOT)
if MAX_TEST > 0:
    test_df = test_df.head(MAX_TEST)

log.info("Dataset split summary:")
log.info("  TRAIN  : %6d samples  (aug=base, offsets=%s, LiDAR loaded only for supervised/hybrid losses)", len(_tr_idx), SOURCE_OFFSETS)
log.info("  VAL    : %6d samples  (aug=none)", len(_val_idx))
log.info("  TEST   : %6d frames (from TEST_ROOT, never used for training / selection)", len(test_df))
log.info("  optimizer updates / epoch = %d  (batches=%d, ACCUM_STEPS=%d)",
         updates_per_epoch_for(len(_tr_idx) // BATCH_SIZE), len(_tr_idx) // BATCH_SIZE, ACCUM_STEPS)

21:05:40  INFO      [KITTI-Seq-val] 23092 samples (offsets=(-1, 1), aug=none)
21:05:40  INFO      [OXTSDataset] 33 drives with GT pose | 23092 samples kept | 0 dropped (no GT pose)
21:05:40  INFO      Canonical sample set: 23,092 samples with complete OXTS pose + calibration
21:05:44  INFO      [KITTI-Seq-val] 23092 samples (offsets=(-1, 1), aug=none)


Indexing test split:   0%|          | 0/652 [00:00<?, ?it/s]

21:05:44  INFO      Test index: 652 frames, 28 drives
21:05:44  INFO      Dataset split summary:
21:05:44  INFO        TRAIN  :  20783 samples  (aug=base, offsets=(-1, 1), LiDAR loaded only for supervised/hybrid losses)
21:05:44  INFO        VAL    :   2309 samples  (aug=none)
21:05:44  INFO        TEST   :    652 frames (from TEST_ROOT, never used for training / selection)
21:05:44  INFO        optimizer updates / epoch = 650  (batches=2597, ACCUM_STEPS=4)


## 6 · Model — DA3METRIC-LARGE wrapper, PoseNet, fp32 differentiable warping  *(copied verbatim)*

Key methodological points preserved from the source (v6 fixes):

* **fp32 geometry.** Backprojection / projection / warping and all loss maths run with autocast **disabled**. Under fp16 the product `K·X` reaches ≈ 1e5 > 65 504 → `inf` → NaN gradients → the `GradScaler` silently skipped every update in v5. *Nothing here moves geometry into half precision.*
* **PoseNet bootstrapping (C1/C3/C5).** No zero-init (the warp is independent of depth at exactly `R=I, t=0` ⇒ zero depth gradient), output scale `POSE_OUT_SCALE=0.1`, a depth-frozen warm-up epoch, and a separate `POSE_LR`.
* **`decoder_only`** freezes encoder / backbone by name and then **asserts** the trainable-parameter fraction (`describe_trainable`).

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# DA3 model wrapper
# ─────────────────────────────────────────────────────────────────────────────
_DA3_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def load_da3_model(device: torch.device) -> nn.Module:
    from depth_anything_3.api import DepthAnything3
    log.info("Loading %s ...", MODEL_ID)
    model = DepthAnything3.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR).to(device)
    return model


def _apply_arch_impl(model: nn.Module, arch_mode: str) -> None:
    if arch_mode == "full":
        for p in model.parameters():
            p.requires_grad_(True)
        model.model.train()
        if hasattr(model.model, "gradient_checkpointing_enable"):
            model.model.gradient_checkpointing_enable()
        log.info("  [arch=full] All parameters trainable.")
        return

    if arch_mode == "decoder_only":
        for name, param in model.named_parameters():
            freeze = any(k in name for k in
                         ["encoder", "pretrained", "backbone", "net.net"])
            param.requires_grad_(not freeze)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=decoder_only] %.1fM params trainable", trainable/1e6)
        return

    if arch_mode == "last2_blocks":
        for p in model.parameters():
            p.requires_grad_(False)
        decoder_block_names = [
            name for name, _ in model.named_parameters()
            if "decoder" in name.lower() or "head" in name.lower()
               or "depth_head" in name.lower() or "scratch" in name.lower()
        ]
        last_n = max(1, int(0.30 * len(decoder_block_names)))
        for name in decoder_block_names[-last_n:]:
            model.get_parameter(name).requires_grad_(True)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=last2_blocks] %.1fM params trainable", trainable/1e6)
        return

    raise ValueError(f"Unknown arch_mode: {arch_mode}")


def describe_trainable(model: nn.Module, arch_mode: str) -> Dict[str, float]:
    """
    Explicit report of WHICH parts of DA3 are trainable (the name-substring freezing in
    `_apply_arch_impl` is brittle: a renamed module inside DA3 would silently change what trains).
    Logs trainable/total parameters per top-level module of `model.model` and asserts that the
    trainable fraction lies in EXPECTED_TRAINABLE_FRACTION[arch_mode].
    """
    per_top: Dict[str, List[int]] = {}
    for n, p in model.model.named_parameters():
        top = n.split(".")[0]
        t = per_top.setdefault(top, [0, 0])
        t[1] += p.numel()
        t[0] += p.numel() if p.requires_grad else 0
    tot = sum(v[1] for v in per_top.values()); trn = sum(v[0] for v in per_top.values())
    frac = trn / max(tot, 1)
    log.info("  [arch=%s] trainable %.1fM / %.1fM params (%.1f %%)", arch_mode, trn / 1e6, tot / 1e6, 100 * frac)
    for top, (t_, a_) in sorted(per_top.items(), key=lambda kv: -kv[1][1]):
        log.info("      %-24s trainable %7.2fM / %7.2fM", top, t_ / 1e6, a_ / 1e6)
    lo, hi = EXPECTED_TRAINABLE_FRACTION[arch_mode]
    ok = (trn > 0) and (lo <= frac <= hi)
    if not ok:
        msg = (f"arch_mode={arch_mode}: trainable fraction {frac:.4f} outside the expected range "
               f"[{lo}, {hi}] — DA3's parameter names probably changed; inspect the module list above.")
        if STRICT_ARCH_CHECK:
            raise RuntimeError(msg)
        log.warning(msg)
    return {"trainable": trn, "total": tot, "fraction": frac}


def _apply_arch(model: nn.Module, arch_mode: str) -> None:
    _apply_arch_impl(model, arch_mode)
    describe_trainable(model, arch_mode)


def forward_batch(model: nn.Module, rgb_batch: torch.Tensor,
                  K_batch: torch.Tensor, has_K: torch.Tensor) -> torch.Tensor:
    # Differentiable forward pass -> metric depth [B, 1, H, W].
    B, C, H, W = rgb_batch.shape
    inp = rgb_batch.unsqueeze(1)
    preds = model.model(inp, export_feat_layers=[])

    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
        fx_pred = torch.where(
            has_K.view(B, 1, 1, 1),
            (0.5 * (K_batch[:, 0, 0] + K_batch[:, 1, 1])).view(B, 1, 1, 1),   # v6: avg(fx, fy) as in the DA3 metric recipe
            torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                       device=rgb_batch.device, dtype=rgb_batch.dtype),
        )
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)
    else:
        raw = preds
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)

    if raw.dim() == 3:
        raw = raw.unsqueeze(1)
    depth = (fx_pred.float() * raw.float()) / ORTHO_NORM      # fp32 from here on (geometry / losses need it)
    if depth.shape[-2:] != (H, W):
        depth = F.interpolate(depth, size=(H, W), mode="bilinear",
                              align_corners=False)
    return depth


@torch.no_grad()
def run_da3_inference(model: nn.Module, image_np: np.ndarray,
                      device: torch.device, fx: float = FOCAL_LENGTH_X) -> np.ndarray:
    # Run DA3METRIC-LARGE inference on one uint8 RGB image (H x W x 3).
    orig_h, orig_w = image_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)
    img_pil = Image.fromarray(image_np).resize((new_w, new_h),
                                                Image.Resampling.BICUBIC)
    t_img = _DA3_TRANSFORM(img_pil)
    inp   = t_img.unsqueeze(0).unsqueeze(1).to(device)
    model.model.eval()
    preds = model(inp, export_feat_layers=[])
    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
    else:
        raw = preds
    focal_used: float = fx
    if isinstance(raw, torch.Tensor):
        if raw.dim() == 3:
            raw = raw.unsqueeze(1)
        depth = (focal_used * raw) / ORTHO_NORM
        depth = F.interpolate(depth, size=(orig_h, orig_w),
                              mode="bilinear", align_corners=False)
    else:
        raise RuntimeError(f"Unexpected model output type {type(raw)}")
    return depth.squeeze().cpu().float().numpy()


# ─────────────────────────────────────────────────────────────────────────────
# Pose Network — SfMLearner / Monodepth2-style ResNet18 with 6-DoF head
# ─────────────────────────────────────────────────────────────────────────────
class PoseNet(nn.Module):
    # Predicts the 6-DoF relative pose T_{curr -> src} (axis-angle + translation)
    # from a pair (I_curr, I_src) stacked along the channel dim.
    #
    # v6 changes (PoseNet bootstrapping):
    #   * NO zero-init of the last layer.  v5 started at exactly R=I, t=0, where
    #     the reprojection is independent of depth (K·I·K^-1·(d·p) = p) so the
    #     photometric loss has ZERO gradient w.r.t. depth.  The default init
    #     gives a small non-degenerate random pose from step 0.
    #   * output scale POSE_OUT_SCALE (0.1) instead of 0.01: with METRIC depth the
    #     translation is in metres (~1 m / frame on KITTI).
    #   * PoseNet is pre-trained with depth frozen (see `warmup_pose_net`).
    def __init__(self, out_scale: float = POSE_OUT_SCALE):
        super().__init__()
        self.out_scale = float(out_scale)
        weights = None
        try:
            weights = tvm.ResNet18_Weights.DEFAULT
        except Exception:
            pass
        try:
            backbone = tvm.resnet18(weights=weights)
        except Exception as e:                       # offline machine
            log.warning("PoseNet: ImageNet weights unavailable (%s) -> random init", e)
            backbone = tvm.resnet18(weights=None)

        old_conv = backbone.conv1
        new_conv = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(torch.cat([old_conv.weight, old_conv.weight], dim=1) / 2.0)
        backbone.conv1 = new_conv
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])   # (B, 512, H/32, W/32)

        self.pose_head = nn.Sequential(
            nn.Conv2d(512, 256, 1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 6, 1),
        )
        nn.init.zeros_(self.pose_head[-1].bias)      # bias 0, weights keep the default (non-zero) init

    def forward(self, rgb_curr: torch.Tensor, rgb_src: torch.Tensor) -> torch.Tensor:
        # rgb_* : [B, 3, H, W] (ImageNet-normalised)  ->  pose [B, 6]
        x = torch.cat([rgb_curr, rgb_src], dim=1)
        p = self.pose_head(self.encoder(x))
        return self.out_scale * p.mean(dim=(2, 3))


# ─────────────────────────────────────────────────────────────────────────────
# SE(3) helpers
# ─────────────────────────────────────────────────────────────────────────────
def axisangle_to_rotmat(aa: torch.Tensor) -> torch.Tensor:
    # Rodrigues' formula: [B, 3] -> [B, 3, 3]
    aa    = aa.float()
    theta = aa.norm(dim=1, keepdim=True).clamp(min=EPS)      # [B, 1]
    k     = aa / theta
    cos_t         = torch.cos(theta).unsqueeze(-1)            # [B, 1, 1]
    sin_t         = torch.sin(theta).unsqueeze(-1)
    one_minus_cos = 1.0 - cos_t
    K = torch.zeros(aa.shape[0], 3, 3, device=aa.device, dtype=aa.dtype)
    K[:, 0, 1] = -k[:, 2]; K[:, 0, 2] =  k[:, 1]
    K[:, 1, 0] =  k[:, 2]; K[:, 1, 2] = -k[:, 0]
    K[:, 2, 0] = -k[:, 1]; K[:, 2, 1] =  k[:, 0]
    I = torch.eye(3, device=aa.device, dtype=aa.dtype).unsqueeze(0).expand_as(K)
    return cos_t * I + sin_t * K + one_minus_cos * (k.unsqueeze(2) @ k.unsqueeze(1))


def pose_to_se3(pose: torch.Tensor) -> torch.Tensor:
    # [B, 6] (axisangle[3], translation[3]) -> SE(3) [B, 4, 4]   (always fp32)
    pose = pose.float()
    B = pose.shape[0]
    R = axisangle_to_rotmat(pose[:, :3])
    T_ = torch.eye(4, device=pose.device, dtype=pose.dtype).unsqueeze(0).repeat(B, 1, 1)
    T_[:, :3, :3] = R
    T_[:, :3,  3] = pose[:, 3:]
    return T_


def invert_se3(T_: torch.Tensor) -> torch.Tensor:
    R, t = T_[:, :3, :3], T_[:, :3, 3:]
    Ti = torch.eye(4, device=T_.device, dtype=T_.dtype).unsqueeze(0).repeat(T_.shape[0], 1, 1)
    Ti[:, :3, :3] = R.transpose(1, 2)
    Ti[:, :3, 3:] = -R.transpose(1, 2) @ t
    return Ti


# ─────────────────────────────────────────────────────────────────────────────
# Differentiable inverse-warping  (v6: ALWAYS fp32, autocast disabled)
# ─────────────────────────────────────────────────────────────────────────────
# v5 executed these matmuls under fp16 autocast.  K·X reaches ~u·Z ≈ 1e5 > 65504
# (fp16 max) for far right-hand pixels -> inf -> NaN gradients -> GradScaler
# skipped the optimizer step -> the SSL runs never updated the network.
# Even below the overflow limit fp16 quantises pixel coordinates in [1024, 2048)
# to 1.0 px.  Geometry therefore must not run in half precision.
class BackprojectDepth(nn.Module):
    # Backproject depth to 3-D camera-frame points using inverse intrinsics.
    def __init__(self, height, width):
        super().__init__()
        self.H, self.W = height, width
        u, v = torch.meshgrid(
            torch.arange(0, height, dtype=torch.float32),
            torch.arange(0, width,  dtype=torch.float32),
            indexing="ij")
        pix = torch.stack([v, u], dim=0).view(2, -1)
        ones = torch.ones(1, height * width)
        self.register_buffer("pix_homog", torch.cat([pix, ones], dim=0).unsqueeze(0))   # [1, 3, H*W]

    def forward(self, depth: torch.Tensor, inv_K: torch.Tensor) -> torch.Tensor:
        # depth [B,1,H,W]; inv_K [B,3,3]; returns [B,4,H*W]  (fp32)
        B = depth.shape[0]
        with torch.autocast(device_type=depth.device.type, enabled=False):
            depth = depth.float(); inv_K = inv_K.float()
            pix = self.pix_homog.expand(B, -1, -1)
            cam_pts = torch.matmul(inv_K, pix)
            cam_pts = depth.reshape(B, 1, -1) * cam_pts
            return torch.cat([cam_pts, torch.ones(B, 1, self.H * self.W,
                                                   device=depth.device, dtype=torch.float32)], dim=1)


class Project3D(nn.Module):
    # Project 3-D camera-frame points with intrinsics + pose  ->  normalised grid.
    def __init__(self, height, width):
        super().__init__()
        self.H, self.W = height, width

    def forward(self, points_3d, K, T, return_z: bool = False):
        B = points_3d.shape[0]
        with torch.autocast(device_type=points_3d.device.type, enabled=False):
            P = torch.matmul(K.float(), T.float()[:, :3, :])                # [B,3,4]
            cam_pts = torch.matmul(P, points_3d.float())                    # [B,3,H*W]
            z_raw = cam_pts[:, 2:3]
            pix = cam_pts[:, :2] / z_raw.clamp(min=0.01)
            pix = pix.view(B, 2, self.H, self.W)
            pix_norm = torch.stack([2.0 * pix[:, 0] / (self.W - 1) - 1.0,
                                    2.0 * pix[:, 1] / (self.H - 1) - 1.0], dim=1)
            pix_norm = pix_norm.clamp(-10.0, 10.0).permute(0, 2, 3, 1)
        if return_z:
            return pix_norm, z_raw.view(B, 1, self.H, self.W)
        return pix_norm


# ─── Cached BackprojectDepth / Project3D (avoid per-batch GPU allocation) ─────
_backproj_cache: Dict[Tuple, "BackprojectDepth"] = {}
_project_cache:  Dict[Tuple, "Project3D"]        = {}

def _get_backproj(H: int, W: int, device: torch.device) -> "BackprojectDepth":
    key = (H, W, device)
    if key not in _backproj_cache:
        _backproj_cache[key] = BackprojectDepth(H, W).to(device)
    return _backproj_cache[key]

def _get_project(H: int, W: int, device: torch.device) -> "Project3D":
    key = (H, W, device)
    if key not in _project_cache:
        _project_cache[key] = Project3D(H, W).to(device)
    return _project_cache[key]


def _safe_inv_K(K: torch.Tensor) -> torch.Tensor:
    """Batched K inversion in fp32."""
    B = K.shape[0]
    with torch.autocast(device_type=K.device.type, enabled=False):
        K_f32 = K.float()
        eye = torch.eye(3, device=K.device, dtype=torch.float32).unsqueeze(0).expand(B, -1, -1)
        try:
            return torch.linalg.solve(K_f32, eye)
        except torch.linalg.LinAlgError:
            K_safe = K_f32.clone()
            diag = K_safe.diagonal(dim1=-2, dim2=-1)
            diag[diag.abs() < 1e-6] = 1.0
            return torch.linalg.solve(K_safe, eye)


def warp_image(image_src: torch.Tensor,
               depth_curr: torch.Tensor,
               pose_curr_to_src: torch.Tensor,
               K: torch.Tensor,
               has_K: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Inverse-warp `image_src` into the CURRENT frame using the depth of the
    # current frame and T_{curr -> src}.  Returns (warped [B,C,H,W], valid [B,1,H,W]).
    # `valid` = projected pixel inside the source image AND in front of the camera.
    B, C, H, W = image_src.shape
    with torch.autocast(device_type=image_src.device.type, enabled=False):
        image_src = image_src.float()
        inv_K = _safe_inv_K(K)
        pts3d = _get_backproj(H, W, image_src.device)(depth_curr, inv_K)
        pix, z = _get_project(H, W, image_src.device)(pts3d, K, pose_curr_to_src, return_z=True)
        warped = F.grid_sample(image_src, pix, mode="bilinear",
                               padding_mode="zeros", align_corners=True)
        valid = ((pix[..., 0].abs() <= 1.0) & (pix[..., 1].abs() <= 1.0)).float().unsqueeze(1)
        valid = valid * (z > 0.01).float()
        warped = torch.nan_to_num(warped, nan=0.0, posinf=0.0, neginf=0.0)
    return warped, valid


def warp_depth(depth_src: torch.Tensor,
               depth_curr: torch.Tensor,
               pose_curr_to_src: torch.Tensor,
               K: torch.Tensor, has_K: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Sample the SOURCE depth map at the locations where the current pixels land.
    return warp_image(depth_src, depth_curr, pose_curr_to_src, K)


print("DA3 model wrapper + PoseNet + fp32 differentiable warping ready (v6).")

DA3 model wrapper + PoseNet + fp32 differentiable warping ready (v6).


### 6b · Load DA3METRIC-LARGE **once** and keep the pristine state (recipe isolation)

`_base_state` is a deep copy of the pretrained weights. **Every recipe starts by re-loading this exact state** (`load_da3_model_fast`), so the recipes are six independent fine-tunings of the same starting point — never a sequence `A1 → A3 → B1 → …`.

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Running on: %s", device)

_base_model = load_da3_model(device)
_base_state = copy.deepcopy(_base_model.state_dict())

def load_da3_model_fast(device):
    _base_model.load_state_dict(copy.deepcopy(_base_state))
    return _base_model

21:05:45  INFO      Running on: cuda


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


21:05:47  INFO      Loading depth-anything/DA3METRIC-LARGE ...
21:05:48  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
21:05:48  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


21:05:51  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"


## 7 · Reused Loss Implementations  *(copied verbatim — nothing re-implemented)*

Design rules inherited from the source (v6):

1. A self-supervised loss **never touches GT depth**: the trainer passes `gt=None, mask=None` to every loss whose `LossSpec.uses_gt_depth` is `False`.
2. A loss that misses an input raises `SSLInputError`. There is **no** `if pose is None: return |pred − gt|` fallback (that silent fallback invalidated the v5 self-supervised runs).
3. Predicted-pose and GT-pose (OXTS) losses are different experiments, routed by `LossSpec`.
4. All loss maths in fp32 (`_unified_forward` disables autocast).

Copied blocks: `PhotoL1Loss`, `MinReprojPhotoLoss`, `EdgeAwareSmoothLoss`, `FeatureMetricLoss`, `RigidFlowConsistencyLoss`, `TemporalWarpConsistencyLoss`, `GTPosePhotoReprojLoss`, `GTPoseRigidFlowLoss`, `GTPoseTemporalWarpLoss` and their helpers (`photometric_residual`, `edge_aware_smoothness_map`, `_depth_consistency_terms`, …).
*Not copied:* the loss classes no final recipe uses (SSIM-only, multi-scale, SC-SfM geometric, naive-temporal, Monodepth2-combined, `CombinedRecommendedLoss`, `LiDARAnchorOnlyLoss`), the ablation `LOSS_REGISTRY` / `LOSS_SPECS` tables and the five stationary-object losses.

In [15]:
class SSLInputError(RuntimeError):
    """A loss did not receive an input it requires (no supervised fallback exists)."""


def _require(loss_name: str, **named) -> None:
    missing = [k for k, v in named.items() if v is None]
    if missing:
        raise SSLInputError(
            f"{loss_name} requires {missing} but received None. Refusing to fall back to "
            f"GT-depth supervision (see LOSS_SPECS for the inputs this loss declares).")


def _safe_log(x: torch.Tensor) -> torch.Tensor:
    return torch.log(x.clamp(min=EPS))


def _depth(pred: torch.Tensor) -> torch.Tensor:
    return pred.float().clamp(SSL_MIN_DEPTH, SSL_MAX_DEPTH)


def _masked_mean(x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    return (x * m).sum() / (m.sum() + EPS)


def photometric_residual(warped: torch.Tensor, target: torch.Tensor,
                         alpha: float = SSIM_ALPHA) -> torch.Tensor:
    # Per-pixel  alpha·SSIM + (1-alpha)·L1   ->  [B,1,H,W]  (Monodepth2).
    ch = warped.shape[1]
    k = (torch.ones(1, 1, 3, 3, device=warped.device, dtype=warped.dtype) / 9.0).expand(ch, 1, 3, 3)
    mu_p  = F.conv2d(warped, k, padding=1, groups=ch)
    mu_g  = F.conv2d(target, k, padding=1, groups=ch)
    mu_pp = F.conv2d(warped * warped, k, padding=1, groups=ch)
    mu_gg = F.conv2d(target * target, k, padding=1, groups=ch)
    mu_pg = F.conv2d(warped * target, k, padding=1, groups=ch)
    sigma_p  = (mu_pp - mu_p * mu_p).clamp(min=0)
    sigma_g  = (mu_gg - mu_g * mu_g).clamp(min=0)
    sigma_pg = mu_pg - mu_p * mu_g
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu_p * mu_g + C1) * (2 * sigma_pg + C2)) / (
                (mu_p**2 + mu_g**2 + C1) * (sigma_p + sigma_g + C2))
    ssim_l = ((1 - ssim_map) / 2.0).clamp(0, 1).mean(dim=1, keepdim=True)
    l1     = (warped - target).abs().mean(dim=1, keepdim=True)
    return alpha * ssim_l + (1 - alpha) * l1


def edge_aware_smoothness_map(disp: torch.Tensor, image: torch.Tensor,
                              decay: float = SMOOTHNESS_DECAY) -> torch.Tensor:
    # Edge-aware smoothness of the mean-normalised disparity (Godard 2017).
    mean_disp = disp.mean(dim=(2, 3), keepdim=True).clamp(min=EPS)
    dn = disp / mean_disp
    gdx = (dn[:, :, :, :-1] - dn[:, :, :, 1:]).abs()
    gdy = (dn[:, :, :-1, :] - dn[:, :, 1:, :]).abs()
    gix = (image[:, :, :, :-1] - image[:, :, :, 1:]).abs().mean(1, keepdim=True)
    giy = (image[:, :, :-1, :] - image[:, :, 1:, :]).abs().mean(1, keepdim=True)
    return (gdx * torch.exp(-decay * gix)).mean() + (gdy * torch.exp(-decay * giy)).mean()


def _warp_sources(rgb_srcs, poses, depth, K, has_K=None):
    # -> lists of (warped_i, valid_i) for each source frame
    out = []
    for src, T_ in zip(rgb_srcs, poses):
        out.append(warp_image(src, depth, T_, K, has_K=has_K))
    return out


def _check_src_pose_lists(name, rgb_srcs, poses):
    _require(name, rgb_srcs=rgb_srcs, poses=poses)
    if len(rgb_srcs) != len(poses) or len(rgb_srcs) == 0:
        raise SSLInputError(f"{name}: got {len(rgb_srcs)} source frames but {len(poses)} poses")


# ─────────────────────────────────────────────────────────────────────────────
# 1. photo_l1 — masked L1 photometric reprojection (Garg et al., ECCV 2016)
# ─────────────────────────────────────────────────────────────────────────────
class PhotoL1Loss(nn.Module):
    # Reference: Garg et al. ECCV 2016 (arXiv:1603.04992).
    # v6: per-pixel channel-mean L1 averaged over VALID pixels only (v5 averaged over all pixels).
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        _require("photo_l1", rgb=rgb, K=K)
        _check_src_pose_lists("photo_l1", rgb_srcs, poses)
        d = _depth(pred)
        ls, vf = [], []
        for w, v in _warp_sources(rgb_srcs, poses, d, K, has_K):
            l1 = (w - rgb).abs().mean(dim=1, keepdim=True)
            ls.append(_masked_mean(l1, v)); vf.append(v.mean())
        l = torch.stack(ls).mean()
        return l, {"total": l, "photo_l1": l, "valid_frac": torch.stack(vf).mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 5 (+3). MinReprojPhotoLoss — one class, three behaviours
# ─────────────────────────────────────────────────────────────────────────────
class MinReprojPhotoLoss(nn.Module):
    # alpha·SSIM + (1-alpha)·L1 photometric loss over one or more source frames.
    #   use_min=False, use_automask=False : mean over sources           (Godard 2017 / "photo_ssim_l1")
    #   use_min=False, use_automask=True  : + Monodepth2 auto-mask
    #   use_min=True,  use_automask=True  : per-pixel MIN over sources + auto-mask
    #                                       = Monodepth2 (Godard et al., ICCV 2019)
    # With SOURCE_OFFSETS=(-1,+1) this is the canonical 3-frame Monodepth2 loss.
    # With a single source (+1 only) it degenerates to
    #   "single-source reprojection vs. identity auto-mask" and is NOT full Monodepth2.
    # Each identity error gets ~1e-5 noise BEFORE the min to break ties (as in the Monodepth2 reference code).
    def __init__(self, alpha: float = SSIM_ALPHA,
                 auto_mask_eps: float = MIN_REPROJ_AUTO_MASK_THRESHOLD,
                 use_min: bool = True, use_automask: bool = True):
        super().__init__()
        self.alpha, self.eps = alpha, auto_mask_eps
        self.use_min, self.use_automask = use_min, use_automask

    def extra_repr(self) -> str:
        return f"alpha={self.alpha}, eps={self.eps}, use_min={self.use_min}, use_automask={self.use_automask}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        _require("MinReprojPhotoLoss", rgb=rgb, K=K)
        _check_src_pose_lists("MinReprojPhotoLoss", rgb_srcs, poses)
        d = _depth(pred)
        errs, valids, idents = [], [], []
        for src, (w, v) in zip(rgb_srcs, _warp_sources(rgb_srcs, poses, d, K, has_K)):
            errs.append(photometric_residual(w, rgb, self.alpha)); valids.append(v)
            if self.use_automask:
                idents.append(photometric_residual(src, rgb, self.alpha))
        E, V = torch.cat(errs, 1), torch.cat(valids, 1)                  # [B,S,H,W]
        any_valid = V.max(dim=1, keepdim=True).values
        if self.use_min:
            reproj = (E + (1.0 - V) * 1e3).min(dim=1, keepdim=True).values
        else:
            reproj = (E * V).sum(1, keepdim=True) / (V.sum(1, keepdim=True) + EPS)
        if self.use_automask:
            Id = torch.cat(idents, 1)
            Id = Id + torch.randn_like(Id) * 1e-5                     # per-source noise BEFORE the min (Monodepth2)
            ident = Id.min(dim=1, keepdim=True).values if self.use_min else Id.mean(dim=1, keepdim=True)
            auto = (reproj < ident + self.eps).float()
        else:
            auto = torch.ones_like(any_valid)
        final = any_valid * auto
        l = _masked_mean(reproj, final)
        return l, {"total": l, "photo": l, "valid_frac": any_valid.mean(),
                   "auto_mask_frac": (auto * any_valid).sum() / (any_valid.sum() + EPS)}


# ─────────────────────────────────────────────────────────────────────────────
# 4. edge_aware_smooth — regulariser ONLY (degenerate constant depth on its own)
# ─────────────────────────────────────────────────────────────────────────────
class EdgeAwareSmoothLoss(nn.Module):
    def __init__(self, decay: float = SMOOTHNESS_DECAY):
        super().__init__()
        self.decay = decay

    def forward(self, pred, gt=None, mask=None, *, rgb=None, **kw):
        _require("edge_aware_smooth", rgb=rgb)
        l = edge_aware_smoothness_map(1.0 / _depth(pred), rgb, self.decay)
        return l, {"total": l, "smooth": l}


# ─────────────────────────────────────────────────────────────────────────────
# Shared: depth-consistency residual between t and t+1  (used by 7 and 10)
# ─────────────────────────────────────────────────────────────────────────────
def _depth_consistency_terms(d_curr, d_next, pose_next, K):
    # d_proj    : depth of frame t+1 sampled where the current pixels land
    # d_compute : z-coordinate of the current 3-D points expressed in frame t+1
    # valid     : in-bounds & in front of the camera & both depths positive
    d_proj, vmask = warp_depth(d_next, d_curr, pose_next, K)
    B, _, H, W = d_curr.shape
    pts = _get_backproj(H, W, d_curr.device)(d_curr, _safe_inv_K(K))              # [B,4,HW]
    d_compute = torch.matmul(pose_next.float()[:, :3, :], pts)[:, 2:3].reshape(B, 1, H, W)
    valid = vmask * (d_proj > EPS).float() * (d_compute > EPS).float()
    diff = (d_proj - d_compute).abs() / (d_proj + d_compute + EPS)
    return diff, valid


# ─────────────────────────────────────────────────────────────────────────────
# 8. feature_metric — feature-space photometric loss (FeatDepth-style, simplified)
# ─────────────────────────────────────────────────────────────────────────────
class FeatureMetricLoss(nn.Module):
    # Compares frozen ImageNet ResNet18 (conv1..layer1) features at 1/4 resolution
    # + a small raw-pixel L1 term.  NOT the learned auto-encoder features of the
    # original FeatDepth (Shu et al., ECCV 2020) — a simplified variant.
    def __init__(self, alpha: float = SSIM_ALPHA):
        super().__init__()
        self.alpha = alpha
        self.fe = None
        self.register_buffer("_mean", torch.tensor(_IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("_std",  torch.tensor(_IMAGENET_STD).view(1, 3, 1, 1))

    def _build_extractor(self, device):
        try:
            try:
                weights = tvm.ResNet18_Weights.DEFAULT
            except Exception:
                weights = None
            rn = tvm.resnet18(weights=weights).to(device)
        except Exception:
            rn = tvm.resnet18(weights=None).to(device)
        for p in rn.parameters():
            p.requires_grad_(False)
        self.fe = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool, rn.layer1).eval()

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None, pose_next=None,
                K=None, has_K=None, **kw):
        _require("feature_metric", rgb=rgb, rgb_next=rgb_next, pose_next=pose_next, K=K)
        if self.fe is None:
            self._build_extractor(pred.device)
        d = _depth(pred)
        warped, vmask = warp_image(rgb_next, d, pose_next, K)
        l_pix = _masked_mean((warped - rgb).abs().mean(1, keepdim=True), vmask)

        with torch.no_grad():
            f_t = self.fe((rgb - self._mean) / self._std)
        f_n = self.fe((rgb_next - self._mean) / self._std).detach()
        hf, wf = f_t.shape[-2:]
        d_f = F.interpolate(d, size=(hf, wf), mode="area")
        K_f = K.clone()
        K_f[:, 0, :] = K_f[:, 0, :] * (wf / rgb.shape[-1])          # fx, cx scale with the WIDTH ratio
        K_f[:, 1, :] = K_f[:, 1, :] * (hf / rgb.shape[-2])          # fy, cy scale with the HEIGHT ratio
        f_w, v_f = warp_image(f_n, d_f, pose_next, K_f)
        l_feat = _masked_mean((f_w - f_t).abs().mean(1, keepdim=True), v_f)
        l = 0.5 * l_feat + 0.5 * l_pix
        return l, {"total": l, "feature": l_feat, "pixel": l_pix, "valid_frac": vmask.mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 9. rigid_flow_consistency — depth+pose induced ("rigid") flow, photometric + smooth
#    (v5 name: flow_depth_consistency, described as GeoNet.  It is NOT GeoNet: GeoNet
#     learns a separate optical-flow branch and enforces rigid/full-flow consistency.)
# ─────────────────────────────────────────────────────────────────────────────
class RigidFlowConsistencyLoss(nn.Module):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None, pose_next=None,
                K=None, has_K=None, **kw):
        _require("rigid_flow_consistency", rgb=rgb, rgb_next=rgb_next, pose_next=pose_next, K=K)
        d = _depth(pred)
        B, _, H, W = d.shape
        pts  = _get_backproj(H, W, d.device)(d, _safe_inv_K(K))
        pix_in_next = _get_project(H, W, d.device)(pts, K, pose_next)                    # [B,H,W,2] normalised
        u, v = torch.meshgrid(torch.arange(W, dtype=torch.float32, device=d.device),
                              torch.arange(H, dtype=torch.float32, device=d.device), indexing="xy")
        pix0 = torch.stack([2.0 * u / (W - 1) - 1.0, 2.0 * v / (H - 1) - 1.0], dim=-1)
        flow = pix_in_next - pix0.unsqueeze(0)
        warped, vmask = warp_image(rgb_next, d, pose_next, K)
        photo = _masked_mean(photometric_residual(warped, rgb, SSIM_ALPHA), vmask)
        flow_mag = flow.permute(0, 3, 1, 2).pow(2).sum(dim=1, keepdim=True).add(EPS).sqrt()
        flow_smooth = edge_aware_smoothness_map(flow_mag, rgb)
        l = photo + 0.1 * flow_smooth
        return l, {"total": l, "photo": photo, "flow_smooth": flow_smooth, "valid_frac": vmask.mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 10. temporal_warp_consistency — warped-depth consistency (+ small photometric anchor)
# ─────────────────────────────────────────────────────────────────────────────
class TemporalWarpConsistencyLoss(nn.Module):
    # Same residual as SC-SfM's geometric loss, plus an optional photometric term.
    #
    # v6 MASKING FIX.  v5 built its "static" mask from the SAME residual it then
    # minimised (keep pixels with rel_diff < 0.5), i.e. it ignored exactly the
    # pixels that needed correcting.  v6 separates:
    #    visibility mask   : in-bounds, in front of camera, positive depths
    #    dynamic-object mask (optional `dyn_mask`, 1 = moving object)
    #    residual handling : bounded L1 (rel. diff is in [0,1]) — robust by
    #                        construction, no residual-based pixel selection.
    def __init__(self, photo_weight: float = 0.5):
        super().__init__()
        self.photo_weight = photo_weight

    def extra_repr(self) -> str:
        return f"photo_weight={self.photo_weight}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None,
                pred_next=None, pose_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        _require("temporal_warp_consistency", pred_next=pred_next, pose_next=pose_next, K=K)
        d_curr, d_next = _depth(pred), _depth(pred_next)
        diff, valid = _depth_consistency_terms(d_curr, d_next, pose_next, K)
        if dyn_mask is not None:
            valid = valid * (1.0 - dyn_mask.float())
        l_tc = _masked_mean(diff, valid)
        with torch.no_grad():                                   # diagnostic only, never used in the loss
            large_frac = ((diff > 0.5).float() * valid).sum() / (valid.sum() + EPS)

        if self.photo_weight > 0:
            _require("temporal_warp_consistency(photo term)", rgb=rgb, rgb_next=rgb_next)
            warped, vm = warp_image(rgb_next, d_curr, pose_next, K)
            l_p = _masked_mean(photometric_residual(warped, rgb, SSIM_ALPHA), vm)
        else:
            l_p = torch.zeros((), device=pred.device)
        l = l_tc + self.photo_weight * l_p
        return l, {"total": l, "temporal_consistency": l_tc, "photo": l_p,
                   "valid_frac": valid.mean(), "large_resid_frac": large_frac}


# =============================================================================
# ORACLE-POSE variants (GT OXTS pose + learned depth) — NOT self-supervised
# =============================================================================
# Strict wrappers: they require `gt_poses` / `gt_pose_next` and never look at the
# PoseNet output.  If OXTS pose is missing they raise (no silent substitution).
class GTPosePhotoReprojLoss(MinReprojPhotoLoss):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None,
                gt_poses=None, K=None, has_K=None, **kw):
        _require("gt_photo_reproj", gt_poses=gt_poses)
        return super().forward(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=gt_poses, K=K, has_K=has_K)


class GTPoseRigidFlowLoss(RigidFlowConsistencyLoss):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None,
                gt_pose_next=None, K=None, has_K=None, **kw):
        _require("gt_rigid_flow_consistency", gt_pose_next=gt_pose_next)
        return super().forward(pred, rgb=rgb, rgb_next=rgb_next, pose_next=gt_pose_next, K=K, has_K=has_K)


class GTPoseTemporalWarpLoss(TemporalWarpConsistencyLoss):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None, pred_next=None,
                gt_pose_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        _require("gt_temporal_warp_consistency", gt_pose_next=gt_pose_next)
        return super().forward(pred, rgb=rgb, rgb_next=rgb_next, pred_next=pred_next,
                               pose_next=gt_pose_next, K=K, has_K=has_K, dyn_mask=dyn_mask)

### 7b · `_GTCombo` — the OXTS-pose recipe composer  *(verbatim from source Group 5b)*

This is the *exact* class the ablation used for the validated Group 5b **H** (`{"gt_photo": 1.0}`) and **K** (`{"gt_flow": 1.0}`) recipes; A1 / A3 / B1 reuse it. It contains an `anchor` branch (used by the ablation's hybrid recipes J and L) that calls `_silog`. **That branch is unreachable in this notebook, three times over:** (i) recipe terms never contain the key `"anchor"` (asserted in Section 8), (ii) `W_ANCHOR = 0`, (iii) the trainer passes `gt=None, mask=None` when `uses_gt_depth=False`, so `_silog` would raise `SSLInputError` immediately. `_silog` is copied only so the verbatim class has no dangling name.

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# SILog helper + supervised control
# ─────────────────────────────────────────────────────────────────────────────
def _silog(pred, gt, mask, lam: float = 0.85):
    _require("SILog anchor", gt=gt, mask=mask)
    p = pred.float().clamp(EPS, MAX_DEPTH).squeeze(1)
    g = gt.squeeze(1).clamp(EPS)
    m = mask.squeeze(1).bool()
    if not m.any():
        return p.sum() * 0.0
    d = _safe_log(p[m]) - _safe_log(g[m])
    return torch.sqrt((d ** 2).mean() - lam * d.mean() ** 2 + EPS)


# ── Group 5b: oracle-pose combinations (GT OXTS pose; NOT self-supervised) ─────────────
class _GTCombo(nn.Module):
    def __init__(self, terms: Dict[str, float], lam_silog: float = 0.85):
        super().__init__()
        self.terms, self.lam_silog = dict(terms), lam_silog
        self.gt_photo, self.gt_flow = GTPosePhotoReprojLoss(), GTPoseRigidFlowLoss()
        self.gt_temporal = GTPoseTemporalWarpLoss(photo_weight=0.0)
        self.smooth = EdgeAwareSmoothLoss()

    def extra_repr(self) -> str:
        return f"terms={self.terms}, lam_silog={self.lam_silog}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, rgb_next=None,
                gt_poses=None, gt_pose_next=None, pred_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        total, out = torch.zeros((), device=pred.device), {}
        if "gt_photo" in self.terms:
            l, _ = self.gt_photo(pred, rgb=rgb, rgb_srcs=rgb_srcs, gt_poses=gt_poses, K=K, has_K=has_K)
            total = total + self.terms["gt_photo"] * l; out["gt_photo"] = l
        if "gt_flow" in self.terms:
            l, _ = self.gt_flow(pred, rgb=rgb, rgb_next=rgb_next, gt_pose_next=gt_pose_next, K=K, has_K=has_K)
            total = total + self.terms["gt_flow"] * l; out["gt_flow"] = l
        if "gt_temporal" in self.terms:
            l, _ = self.gt_temporal(pred, rgb=rgb, rgb_next=rgb_next, pred_next=pred_next,
                                    gt_pose_next=gt_pose_next, K=K, has_K=has_K, dyn_mask=dyn_mask)
            total = total + self.terms["gt_temporal"] * l; out["gt_temporal"] = l
        if "smooth" in self.terms:
            l, _ = self.smooth(pred, rgb=rgb)
            total = total + self.terms["smooth"] * l; out["smooth"] = l
        if "anchor" in self.terms:
            l = _silog(pred, gt, mask, self.lam_silog)
            total = total + self.terms["anchor"] * l; out["anchor"] = l
        out["total"] = total
        return total, out

### 7c · `LossSpec` (routing table entry) and the unified fp32 dispatcher  *(verbatim from source Section 9b)*

In [17]:
# =============================================================================
# LOSS_SPECS — the SINGLE source of truth for what each loss needs
# =============================================================================
# v5 kept three independent sets (NEED_POSE_LOSSES, NEED_NEXT_DEPTH_LOSSES,
# NEED_GT_POSE_LOSSES) that silently drifted apart: the gt_* losses were in none of
# them, so no pose was built and every gt_* loss fell back to GT-depth L1.
# v6 derives ALL routing from this table and validates it against the registry.
@dataclass(frozen=True)
class LossSpec:
    needs_pred_pose: bool = False     # build PoseNet poses  T_{t->src}
    needs_gt_pose:   bool = False     # use OXTS poses (needs OXTS dataset; asserted per batch)
    needs_pred_next: bool = False     # second DA3 forward on I_{t+1}
    needs_tracking:  bool = False     # SAM3 masks + tracker (CPU)
    uses_gt_depth:   bool = False     # loss legitimately reads sparse LiDAR (supervised / hybrid)
    needs_dyn_mask:  bool = False     # temporal/geometric depth-consistency: exclude potentially-moving objects
    category:        str  = "ssl"     # ssl | regulariser | stationary_ssl | hybrid | supervised_control | oracle_pose


# =============================================================================
# Unified dispatcher — every loss gets the same keyword context
# =============================================================================
def _unified_forward(criterion, pred, gt=None, mask=None, **ctx):
    """
    Calls `criterion(pred, gt, mask, **ctx)` with autocast DISABLED and fp32 inputs.
    Each loss picks the keys it needs from ctx and ignores the rest (**kw).  Missing
    required inputs raise SSLInputError inside the loss — there is no fallback path.
    Stationary losses use the aliases ctx['pose'] (=pose_next), ctx['gt_pose'] (=gt_pose_next)
    and ctx['rgb_next'].
    """
    with torch.autocast(device_type=pred.device.type, enabled=False):
        out = criterion(pred.float(), gt, mask, **ctx)
    if not isinstance(out, tuple):
        out = (out, {"total": out})
    return out


def spec_for(loss_name: str, override: Optional[LossSpec] = None) -> LossSpec:
    return override if override is not None else FINAL_LOSS_SPECS[loss_name]

## 8 · Final Recipe Definitions

Six independent fine-tunings, each starting from the **same pretrained DA3METRIC-LARGE**.

| ID | Group | Pose source | Loss composition | Status |
|---|---|---|---|---|
| `A1_gt_pose_photo` | A — OXTS-pose SSL | OXTS | `1.0·GT_Pose_Photo_Reproj` | validated in the ablation (Group 5b **H**) |
| `A3_gt_pose_photo_smooth_temporal` | A — OXTS-pose SSL | OXTS | `1.0·GT_Pose_Photo_Reproj + 1e-3·Edge_Aware_Smooth + 0.1·GT_Pose_Temporal_Warp_Consistency` | **new combination** (not directly validated) |
| `B1_gt_pose_rigid_flow` | B — OXTS-pose SSL | OXTS | `1.0·GT_Pose_Rigid_Flow` | validated in the ablation (Group 5b **K**) |
| `C1_rigid_flow` | C — predicted-pose SSL | PoseNet | `1.0·Rigid_Flow_Consistency` | validated in the ablation (Group 1a) |
| `C3_photo_l1_flow_smooth` | C — predicted-pose SSL | PoseNet | `1.0·Photo_L1 + 0.1·Flow_Smooth` | **new combination** |
| `C5_photo_l1_feature_metric` | C — predicted-pose SSL | PoseNet | `1.0·Photo_L1 + 0.1·Feature_Metric` | **new combination** |

*Status statements repeat the project's own designation; the ablation result tables are **not** embedded in this notebook and no ablation numbers are quoted here.* An earlier variant `GT_Pose_Photo_Reproj + 1e-3·Edge_Aware_Smooth` (**A2**) was evaluated in the ablation and is deliberately **not** carried as a separate final recipe.

**Terminology.** A/B are *self-supervised with respect to depth*: OXTS only supplies the camera motion the reprojection needs. There is **no** LiDAR depth loss, sparse depth anchor or GT-depth target in any recipe (`uses_gt_depth=False` is asserted for all six). The notebook never calls the OXTS recipes "supervised depth".

### A1 — `A1_gt_pose_photo` · GT-pose photometric reprojection *(validated: Group 5b H)*
1. **Contains:** `GTPosePhotoReprojLoss` = `MinReprojPhotoLoss` (Monodepth2: SSIM+L1 residual, per-pixel **min** over the t−1 / t+1 sources, **auto-mask** vs. identity, valid-pixel masked mean) evaluated with the OXTS relative poses. Weight 1.0, composed through the ablation's own `_GTCombo`.
2. **OXTS pose:** yes (camera-frame, conjugation formula, mirrored on flip). 3. **PoseNet:** no. 4. **GT depth:** no.
5. **Why selected:** carried forward from the ablation as the project's validated GT-pose photometric recipe.
6. **Loss:** `L_A1 = 1.0 · L_GT_Pose_Photo_Reprojection`

### A3 — `A3_gt_pose_photo_smooth_temporal` · GT-pose photometric + smoothness + temporal *(new combination)*
1. **Contains:** A1's photometric term + `EdgeAwareSmoothLoss` on `1/depth` + `GTPoseTemporalWarpLoss(photo_weight=0.0)` — warped-depth consistency between `t` and `t+1` using the OXTS pose and a second DA3 forward on `I_{t+1}`, with the SAM3 dynamic-class mask of frame `t` removing potentially-moving pixels. (`photo_weight=0.0` is the value the ablation's `_GTCombo` uses.)
2. **OXTS pose:** yes. 3. **PoseNet:** no. 4. **GT depth:** **no** — Group 5b **J** contained an additional `0.5·SILog(LiDAR)` anchor; it is **not** inherited (`w_anchor = 0`, no `"anchor"` key, `gt=None` routing).
5. **Why selected:** the enhanced GT-pose recipe chosen after the ablation instead of A2. *This exact pure-SSL combination has not been directly validated by a dedicated ablation run.*
6. **Loss:** `L_A3 = 1.0·L_GT_Pose_Photo_Reprojection + 1e-3·L_Edge_Aware_Smooth + 0.1·L_GT_Pose_Temporal_Warp_Consistency`

### B1 — `B1_gt_pose_rigid_flow` · GT-pose rigid flow *(validated: Group 5b K)*
1. **Contains:** `GTPoseRigidFlowLoss` = `RigidFlowConsistencyLoss` with the OXTS pose to `t+1` (`photometric_residual` on the warped `I_{t+1}` + `0.1·` edge-aware smoothness of the rigid-flow magnitude — its internal formulation, unchanged). No extra smoothness or temporal term.
2. **OXTS pose:** yes. 3. **PoseNet:** no. 4. **GT depth:** no.
5. **Why selected:** carried forward from the ablation as the project's validated GT-pose rigid-flow recipe.
6. **Loss:** `L_B1 = 1.0 · L_GT_Pose_Rigid_Flow`

### C1 — `C1_rigid_flow` · rigid-flow consistency, predicted pose *(validated: Group 1a)*
1. **Contains:** `RigidFlowConsistencyLoss` (depth + pose induced flow; photometric residual + `0.1·flow_smooth`, both already inside the class). It is *not* GeoNet (no learned flow branch).
2. **OXTS pose:** no. 3. **PoseNet:** yes (ResNet-18, warm-up with depth frozen, `POSE_LR`). 4. **GT depth:** no.
5. **Why selected:** carried forward from the ablation as the project's validated pure-SSL recipe.
6. **Loss:** `L_C1 = 1.0 · L_Rigid_Flow_Consistency`

### C3 — `C3_photo_l1_flow_smooth` · Photo-L1 + 0.1 · flow-smoothness *(new combination)*
1. **Contains:** `PhotoL1Loss` (masked L1 over both sources) + `0.1 ×` the **flow-smoothness term of the source's `RigidFlowConsistencyLoss`**. The source has no stand-alone helper for that term (it is computed inline), so C3 obtains it by *calling the source class itself* and reading its `flow_smooth` output — the formulation is therefore identical by construction, not re-implemented. (Cost: the class also evaluates its own photometric term, which is discarded; this only costs a little compute and contributes no gradient.)
2. **OXTS pose:** no. 3. **PoseNet:** yes. 4. **GT depth:** no.
5. **Why selected / how it differs from C1:** C1 = `photometric_residual (SSIM+L1) + 0.1·flow_smooth`; C3 = `pure L1 + 0.1·flow_smooth` (and L1 over both sources rather than one). The pair isolates whether the simpler pure-L1 photometric objective benefits from the flow-smoothness regulariser. C3 is **not** an alias of C1 (checked numerically in the pre-flight).
6. **Loss:** `L_C3 = 1.0·L_Photo_L1 + 0.1·L_Flow_Smooth`

### C5 — `C5_photo_l1_feature_metric` · Photo-L1 + 0.1 · feature-metric *(new combination)*
1. **Contains:** `PhotoL1Loss` + `0.1 ×` the source `FeatureMetricLoss` (frozen ImageNet ResNet-18 `conv1…layer1` features at ¼ resolution + a raw-pixel L1 term; a *simplified* FeatDepth-style variant). Extractor, normalisation, masking and warping are the source's, untouched; the extractor is pre-built once on CPU exactly as in the source registry.
2. **OXTS pose:** no. 3. **PoseNet:** yes. 4. **GT depth:** no.
5. **Why selected:** combination proposed from the ablation findings; not directly validated.
6. **Loss:** `L_C5 = 1.0·L_Photo_L1 + 0.1·L_Feature_Metric`

In [18]:
# =============================================================================
# FINAL RECIPE REGISTRY  (NEW code, following the source's LOSS_REGISTRY / LOSS_SPECS convention)
# =============================================================================
class _SSLCombo(nn.Module):
    """C3 / C5 composer — predicted-pose, PURE self-supervised (NEW; same convention as the source's `_GTCombo`).

    Terms (all built from UNMODIFIED source losses):
        photo_l1       -> PhotoL1Loss()
        flow_smooth    -> the `flow_smooth` output of RigidFlowConsistencyLoss()  (the SAME formulation, obtained by calling
                          the source class; its own photometric output is ignored)
        feature_metric -> FeatureMetricLoss()
    Never reads GT depth: a supplied `gt` / `mask` is an error.
    """
    _ALLOWED = ("photo_l1", "flow_smooth", "feature_metric")

    def __init__(self, terms: Dict[str, float]):
        super().__init__()
        assert set(terms) <= set(self._ALLOWED) and "photo_l1" in terms, f"unsupported terms {terms}"
        self.terms = dict(terms)
        self.photo_l1    = PhotoL1Loss()
        self.rigid_flow  = RigidFlowConsistencyLoss() if "flow_smooth" in terms else None
        self.feature     = FeatureMetricLoss()        if "feature_metric" in terms else None

    def extra_repr(self) -> str:
        return f"terms={self.terms}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, rgb_next=None, poses=None,
                pose_next=None, K=None, has_K=None, **kw):
        assert gt is None and mask is None, "pure-SSL recipe received GT depth"
        total, out = torch.zeros((), device=pred.device), {}
        l, d = self.photo_l1(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=poses, K=K, has_K=has_K)
        total = total + self.terms["photo_l1"] * l; out["photo_l1"] = l; out["valid_frac"] = d["valid_frac"]
        if "flow_smooth" in self.terms:
            _, d = self.rigid_flow(pred, rgb=rgb, rgb_next=rgb_next, pose_next=pose_next, K=K, has_K=has_K)
            l = d["flow_smooth"]
            total = total + self.terms["flow_smooth"] * l; out["flow_smooth"] = l
        if "feature_metric" in self.terms:
            l, _ = self.feature(pred, rgb=rgb, rgb_next=rgb_next, pose_next=pose_next, K=K, has_K=has_K)
            total = total + self.terms["feature_metric"] * l; out["feature_metric"] = l
        out["total"] = total
        return total, out


@dataclass(frozen=True, eq=False)
class FinalRecipe:
    recipe_id: str
    group: str                     # "A" | "B" (OXTS-pose SSL)  |  "C" (predicted-pose pure SSL)
    name: str
    description: str
    loss_composition: str
    terms: Dict[str, float]
    spec: "LossSpec"               # the source's routing record (drives prepare_step / loaders)
    pose_source: str
    validated_in_ablation: bool
    source_reference: str
    criterion_proto: nn.Module

    needs_pred_pose = property(lambda s: s.spec.needs_pred_pose)
    needs_gt_pose   = property(lambda s: s.spec.needs_gt_pose)
    needs_pred_next = property(lambda s: s.spec.needs_pred_next)
    needs_dyn_mask  = property(lambda s: s.spec.needs_dyn_mask)
    uses_gt_depth   = property(lambda s: s.spec.uses_gt_depth)

    def build_criterion(self) -> nn.Module:
        return copy.deepcopy(self.criterion_proto)


_EXPECTED_WEIGHTS = {
    "A1_gt_pose_photo":                 {"gt_photo": 1.0},
    "A3_gt_pose_photo_smooth_temporal": {"gt_photo": 1.0, "smooth": 1e-3, "gt_temporal": 0.1},
    "B1_gt_pose_rigid_flow":            {"gt_flow": 1.0},
    "C1_rigid_flow":                    {"rigid_flow_consistency": 1.0},
    "C3_photo_l1_flow_smooth":          {"photo_l1": 1.0, "flow_smooth": 0.1},
    "C5_photo_l1_feature_metric":       {"photo_l1": 1.0, "feature_metric": 0.1},
}
assert RECIPE_WEIGHTS == _EXPECTED_WEIGHTS, "RECIPE_WEIGHTS is part of the recipe definition and must not be edited"
_OXTS, _PRED = "OXTS (GT camera-frame relative pose)", "PoseNet (predicted, warm-started with depth frozen)"


def _build_final_recipes() -> Dict[str, "FinalRecipe"]:
    W, R = RECIPE_WEIGHTS, {}

    def add(**kw):
        r = FinalRecipe(**kw); R[r.recipe_id] = r

    add(recipe_id="A1_gt_pose_photo", group="A", name="GT-pose photometric reprojection",
        description="OXTS-pose SSL: 1.0*GT_Pose_Photo_Reproj",
        loss_composition="1.0*GT_Pose_Photo_Reproj", terms=W["A1_gt_pose_photo"],
        spec=LossSpec(needs_gt_pose=True, category="oxts_pose_ssl"), pose_source=_OXTS,
        validated_in_ablation=True, source_reference="Group 5b H (H_gt_photo_only)",
        criterion_proto=_GTCombo(W["A1_gt_pose_photo"]))

    add(recipe_id="A3_gt_pose_photo_smooth_temporal", group="A", name="GT-pose photometric + smoothness + temporal",
        description="OXTS-pose SSL: 1.0*GT_Pose_Photo_Reproj + 1e-3*Edge_Aware_Smooth + 0.1*GT_Pose_Temporal_Warp (no LiDAR anchor)",
        loss_composition="1.0*GT_Pose_Photo_Reproj + 1e-3*Edge_Aware_Smooth + 0.1*GT_Pose_Temporal_Warp_Consistency",
        terms=W["A3_gt_pose_photo_smooth_temporal"],
        spec=LossSpec(needs_gt_pose=True, needs_pred_next=True, needs_dyn_mask=True, category="oxts_pose_ssl"),
        pose_source=_OXTS, validated_in_ablation=False, source_reference="new (Group 5b J terms WITHOUT the 0.5*SILog anchor)",
        criterion_proto=_GTCombo(W["A3_gt_pose_photo_smooth_temporal"]))

    add(recipe_id="B1_gt_pose_rigid_flow", group="B", name="GT-pose rigid flow",
        description="OXTS-pose SSL: 1.0*GT_Pose_Rigid_Flow",
        loss_composition="1.0*GT_Pose_Rigid_Flow", terms=W["B1_gt_pose_rigid_flow"],
        spec=LossSpec(needs_gt_pose=True, category="oxts_pose_ssl"), pose_source=_OXTS,
        validated_in_ablation=True, source_reference="Group 5b K (K_gt_rigid_flow_only)",
        criterion_proto=_GTCombo(W["B1_gt_pose_rigid_flow"]))

    add(recipe_id="C1_rigid_flow", group="C", name="Rigid-flow consistency",
        description="Predicted-pose pure SSL: 1.0*Rigid_Flow_Consistency",
        loss_composition="1.0*Rigid_Flow_Consistency", terms=W["C1_rigid_flow"],
        spec=LossSpec(needs_pred_pose=True, category="pred_pose_ssl"), pose_source=_PRED,
        validated_in_ablation=True, source_reference="Group 1a rigid_flow_consistency",
        criterion_proto=RigidFlowConsistencyLoss())

    add(recipe_id="C3_photo_l1_flow_smooth", group="C", name="Photo-L1 + flow smoothness",
        description="Predicted-pose pure SSL: 1.0*Photo_L1 + 0.1*Flow_Smooth",
        loss_composition="1.0*Photo_L1 + 0.1*Flow_Smooth", terms=W["C3_photo_l1_flow_smooth"],
        spec=LossSpec(needs_pred_pose=True, category="pred_pose_ssl"), pose_source=_PRED,
        validated_in_ablation=False, source_reference="new (PhotoL1Loss + flow_smooth term of RigidFlowConsistencyLoss)",
        criterion_proto=_SSLCombo(W["C3_photo_l1_flow_smooth"]))

    c5 = _SSLCombo(W["C5_photo_l1_feature_metric"])
    try:                                    # pre-build the frozen ResNet-18 extractor once (as the source registry does)
        c5.feature._build_extractor(torch.device("cpu"))
        log.info("FeatureMetricLoss extractor pre-loaded on CPU (C5).")
    except Exception as _e:
        log.warning("FeatureMetricLoss extractor pre-load failed (%s) — built lazily instead.", _e)
    add(recipe_id="C5_photo_l1_feature_metric", group="C", name="Photo-L1 + feature metric",
        description="Predicted-pose pure SSL: 1.0*Photo_L1 + 0.1*Feature_Metric",
        loss_composition="1.0*Photo_L1 + 0.1*Feature_Metric", terms=W["C5_photo_l1_feature_metric"],
        spec=LossSpec(needs_pred_pose=True, category="pred_pose_ssl"), pose_source=_PRED,
        validated_in_ablation=False, source_reference="new (PhotoL1Loss + FeatureMetricLoss)",
        criterion_proto=c5)
    return R


FINAL_RECIPES: Dict[str, FinalRecipe] = _build_final_recipes()
FINAL_LOSS_SPECS: Dict[str, "LossSpec"] = {rid: r.spec for rid, r in FINAL_RECIPES.items()}
assert list(FINAL_RECIPES) == list(RECIPE_WEIGHTS)


def assert_recipe_safety(r: FinalRecipe) -> None:
    """Hard safety assertions executed before a recipe is trained (and once for all six below)."""
    assert not r.uses_gt_depth and r.uses_gt_depth is False, f"{r.recipe_id}: must not use GT depth"
    assert not r.spec.needs_tracking, f"{r.recipe_id}: stationary-object tracking is not a final recipe component"
    assert "anchor" not in r.terms and W_ANCHOR == 0.0, f"{r.recipe_id}: LiDAR/SILog anchor is forbidden"
    if r.group in ("A", "B"):
        assert r.needs_gt_pose and not r.needs_pred_pose, f"{r.recipe_id}: OXTS recipe must use GT pose, not PoseNet"
    else:
        assert not r.needs_gt_pose and r.needs_pred_pose, f"{r.recipe_id}: pure-SSL recipe must use PoseNet, not OXTS"


for _r in FINAL_RECIPES.values():
    assert_recipe_safety(_r)
# C1 and C3 must stay mathematically distinct; A3 must not carry the anchor; C5 must use the source FeatureMetricLoss
assert type(FINAL_RECIPES["C1_rigid_flow"].criterion_proto) is RigidFlowConsistencyLoss
assert isinstance(FINAL_RECIPES["C3_photo_l1_flow_smooth"].criterion_proto, _SSLCombo)
assert FINAL_RECIPES["C1_rigid_flow"].terms != FINAL_RECIPES["C3_photo_l1_flow_smooth"].terms
assert "anchor" not in FINAL_RECIPES["A3_gt_pose_photo_smooth_temporal"].criterion_proto.terms
assert type(FINAL_RECIPES["C5_photo_l1_feature_metric"].criterion_proto.feature) is FeatureMetricLoss

# ── resolve which recipes run ───────────────────────────────────────────────
_run_flags = dict(zip(FINAL_RECIPES, [RUN_A1, RUN_A3, RUN_B1, RUN_C1, RUN_C3, RUN_C5]))
if SELECTED_RECIPES is None:
    ACTIVE_RECIPES = [rid for rid, on in _run_flags.items() if on]
else:
    _unknown = [r for r in SELECTED_RECIPES if r not in FINAL_RECIPES]
    assert not _unknown, f"unknown recipe id(s) {_unknown}; valid: {list(FINAL_RECIPES)}"
    ACTIVE_RECIPES = [r for r in FINAL_RECIPES if r in SELECTED_RECIPES]
assert ACTIVE_RECIPES, "no recipe selected"


def print_final_banner() -> None:
    print("FINAL DA3METRIC-LARGE SELF-SUPERVISED FINE-TUNING")
    print("-" * 49)
    print(f"Model: {MODEL_ID.split('/')[-1]}")
    print(f"Architecture: {ARCH_MODE}")
    print(f"Learning rate: {LEARNING_RATE:g}   (PoseNet LR {POSE_LR:g}, C-recipes only)   Weight decay: {WEIGHT_DECAY:g}")
    print(f"Optimizer: {OPTIM_MODE.upper() if OPTIM_MODE != 'adamw' else 'AdamW'}")
    print(f"Scheduler: cosine warmup ({WARMUP_EPOCHS} warm-up epoch(s), stepped per optimizer update)")
    print(f"Augmentation: {AUG_MODE}")
    print(f"Epochs: {FINAL_EPOCHS}   (early-stop patience {EARLY_STOP_PATIENCE})")
    print(f"Batch size: {BATCH_SIZE}   Accumulation: {ACCUM_STEPS}   Effective batch size: {BATCH_SIZE * ACCUM_STEPS}")
    print(f"Mixed precision: {AMP_DTYPE_MODE} autocast for DA3; geometry + losses in fp32")
    print(f"Experiment hash: {EXPERIMENT_HASH}")
    print("\nSelected recipes:")
    for rid in ACTIVE_RECIPES:
        r = FINAL_RECIPES[rid]
        print(f"  {rid:34s} [{r.group}] {r.loss_composition}")
        print(f"  {'':34s}     pose={r.pose_source.split(' (')[0]:8s} pred_pose={r.needs_pred_pose!s:5} gt_pose={r.needs_gt_pose!s:5} "
              f"pred_next={r.needs_pred_next!s:5} dyn_mask={r.needs_dyn_mask!s:5} uses_gt_depth={r.uses_gt_depth}  "
              f"[{'validated in ablation' if r.validated_in_ablation else 'NEW combination'}: {r.source_reference}]")


print_final_banner()

21:06:08  INFO      FeatureMetricLoss extractor pre-loaded on CPU (C5).


FINAL DA3METRIC-LARGE SELF-SUPERVISED FINE-TUNING
-------------------------------------------------
Model: DA3METRIC-LARGE
Architecture: decoder_only
Learning rate: 1e-06   (PoseNet LR 0.0001, C-recipes only)   Weight decay: 0.0001
Optimizer: AdamW
Scheduler: cosine warmup (1 warm-up epoch(s), stepped per optimizer update)
Augmentation: base
Epochs: 50   (early-stop patience 10)
Batch size: 8   Accumulation: 4   Effective batch size: 32
Mixed precision: fp16 autocast for DA3; geometry + losses in fp32
Experiment hash: b093e0aeb3

Selected recipes:
  B1_gt_pose_rigid_flow              [B] 1.0*GT_Pose_Rigid_Flow
                                         pose=OXTS     pred_pose=False gt_pose=True  pred_next=False dyn_mask=False uses_gt_depth=False  [validated in ablation: Group 5b K (K_gt_rigid_flow_only)]


## 9 · Run-level helpers, pretrained-baseline cache, PoseNet warm-up, per-recipe trial  *(source Section 13)*

* `get_baseline_metrics`: validation metrics of the **pretrained** DA3 (epoch 0), computed once and cached.
* `get_warm_pose_state`: PoseNet warm-up (depth frozen, photometric loss only, **no GT depth**), computed once and cached under a hash of every setting that influences it. Only C1 / C3 / C5 use it.
* `_train_one_final_run` is the source's `_train_one_hpt_trial` — train + validate + early stop + checkpoint — run **once per recipe** (the source's one-entry `hpt_cfg = {"lr": LEARNING_RATE}`; there is no sweep). Edits are additive/asserted and listed in the ledger: per-recipe checkpoint directory, richer checkpoint metadata, recipe check on resume, and per-epoch scheduled-LR / epoch-time columns for the plots.

In [19]:
def _stable_hash(obj) -> str:
    import hashlib, json
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]


def _trial_hash(hpt_cfg: Dict, loss_name: str, arch_mode: str, aug_mode: str, optim_mode: str,
                sched_mode: str, spec: "LossSpec", criterion: nn.Module, train_loader: DataLoader) -> str:
    """Identity of one trial's experiment: everything that changes what the trial computes.  A checkpoint whose
    hash differs is never resumed (v6 keyed only on label + LR).  `repr(criterion)` includes the loss weights via
    the modules' extra_repr."""
    return _stable_hash({"hpt": hpt_cfg, "loss": loss_name, "arch": arch_mode, "aug": aug_mode,
                         "optim": optim_mode, "sched": sched_mode, "spec": repr(spec),
                         "criterion": repr(criterion), "train_samples": _loader_fingerprint(train_loader),
                         "experiment": EXPERIMENT_HASH})

def _hpt_tag(hpt_cfg: Dict) -> str:
    return "__".join(f"{k}={v}" for k, v in sorted(hpt_cfg.items()))

def _param_fingerprint(params) -> float:
    # float64 sum(|w|): detects "the network never changed" (v5's silent failure)
    return float(sum(p.detach().double().abs().sum().item() for p in params))


def _loader_fingerprint(loader: DataLoader) -> str:
    """Stable short hash of the exact samples a loader iterates (+ dataset class, aug mode, root, offsets)."""
    import hashlib
    ds = loader.dataset
    idx = list(ds.indices) if hasattr(ds, "indices") else None
    base = ds.dataset if hasattr(ds, "dataset") else ds
    paths = [s[0]["rgb_path"] for s in base._samples]
    if idx is not None:
        paths = [paths[i] for i in idx]
    h = hashlib.md5("|".join(paths).encode())
    h.update(f"{type(base).__name__}|{getattr(base, 'aug_mode', '')}|{getattr(base, 'offsets', '')}|{TRAIN_ROOT}".encode())
    return h.hexdigest()[:12]


# ── Pretrained-baseline validation metrics: computed ONCE per validation set (v6 recomputed them per HPT trial) ──
_BASELINE_CACHE: Dict[str, Dict[str, float]] = {}

@torch.no_grad()
def get_baseline_metrics(val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    import json as _json
    fp = _loader_fingerprint(val_loader)
    if fp in _BASELINE_CACHE:
        return _BASELINE_CACHE[fp]
    path = os.path.join(TEMP_CKPT_DIR, f"baseline_{MODEL_ID.replace('/', '-')}_{fp}.json")
    if os.path.exists(path):
        _BASELINE_CACHE[fp] = _json.load(open(path))
        log.info("[baseline] loaded cached pretrained-baseline metrics (rmse=%.4f)", _BASELINE_CACHE[fp]["rmse"])
        return _BASELINE_CACHE[fp]
    model = load_da3_model_fast(device)
    model.model.eval()
    use_amp = MIXED_PREC and torch.cuda.is_available()
    all_m: List[Dict] = []
    for batch in val_loader:
        rgb, K, has_K = _to(batch["rgb"], device), _to(batch["K"], device), _to(batch["has_K"], device)
        gt, mask = _to(batch["depth_gt"], device), _to(batch["valid_mask"], device)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            pred = forward_batch(model, rgb, K, has_K)
        all_m.append(compute_torch_metrics(pred.float(), gt, mask))
    agg = {k: float(np.nanmean([x[k] for x in all_m])) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    agg["val_loss"] = float("nan")
    _BASELINE_CACHE[fp] = agg
    _json.dump(agg, open(path, "w"))
    log.info("[baseline] pretrained DA3 on validation set: rmse=%.4f  d1=%.4f  (computed once, cached)", agg["rmse"], agg["d1"])
    return agg


# ══════════════════════════════════════════════════════════════════════════════
# PoseNet warm-up (depth FROZEN, photometric loss only, no GT depth) — computed once, cached
# ══════════════════════════════════════════════════════════════════════════════
_POSE_WARM_CACHE: Dict[str, Dict] = {}

def get_warm_pose_state(train_loader: DataLoader, device: torch.device, tag_extra: str = "") -> Dict:
    """
    v5 started PoseNet at R=I,t=0 where the warp does not depend on depth => no depth gradient.
    Here PoseNet is first trained for POSE_WARMUP_EPOCHS with the (pretrained, metric) DA3 depth
    frozen and the Monodepth2 photometric loss, so joint training starts with a pose that
    already carries real motion.  The result is identical for every HPT trial of the same
    data/offsets, so it is computed once and cached on disk.
    """
    ds = train_loader.dataset
    ds = getattr(ds, "dataset", ds)
    # v6.2 cache key: hash of a canonical dict with EVERY setting that influences the warm-up
    warm_cfg = {"code_version": POSE_WARMUP_CODE_VERSION, "experiment": EXPERIMENT_HASH, "model_id": MODEL_ID,
                "dataset_class": type(ds).__name__, "samples_aug_root_offsets": _loader_fingerprint(train_loader),
                "pose_lr": POSE_LR, "warmup_epochs": POSE_WARMUP_EPOCHS, "warmup_max_batches": POSE_WARMUP_MAX_BATCHES,
                "pose_out_scale": POSE_OUT_SCALE, "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE,
                "accum_steps": ACCUM_STEPS, "max_grad_norm": MAX_GRAD_NORM, "amp_mode": AMP_DTYPE_MODE,
                "use_grad_scaler": USE_GRAD_SCALER, "target_hw": (TARGET_H, TARGET_W),
                "source_offsets": SOURCE_OFFSETS, "tag_extra": tag_extra}
    tag = _stable_hash(warm_cfg)
    if tag in _POSE_WARM_CACHE:
        return _POSE_WARM_CACHE[tag]
    path = os.path.join(TEMP_CKPT_DIR, f"pose_warmup__{tag}.pt")   # tag = hash of warm_cfg
    if os.path.exists(path):
        try:
            _POSE_WARM_CACHE[tag] = torch.load(path, map_location="cpu", weights_only=False)
            log.info("[pose warm-up] loaded cached state %s", os.path.basename(path))
            return _POSE_WARM_CACHE[tag]
        except Exception as e:
            log.warning("[pose warm-up] cache unreadable (%s) — recomputing", e)

    log.info("[pose warm-up] training PoseNet for %d epoch(s), depth frozen (lr=%.1e)", POSE_WARMUP_EPOCHS, POSE_LR)
    model = load_da3_model_fast(device)
    for p in model.parameters():
        p.requires_grad_(False)
    pose_net = PoseNet().to(device)
    pose_params = list(pose_net.parameters())
    opt = torch.optim.AdamW(pose_params, lr=POSE_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda" if torch.cuda.is_available() else "cpu", enabled=USE_GRAD_SCALER)
    crit = MinReprojPhotoLoss().to(device)
    wspec = LossSpec(needs_pred_pose=True)
    gstep = 0
    for ep in range(1, POSE_WARMUP_EPOCHS + 1):
        gstep, tl = train_one_epoch(model, pose_net, train_loader, crit, opt, scaler, None, device, ep, gstep,
                                    wspec, train_depth=False, depth_params=[], pose_params=pose_params,
                                    max_batches=(POSE_WARMUP_MAX_BATCHES or None))
        log.info("[pose warm-up] epoch %d  photo=%.4f  pose_grad=%.3e  skipped=%d",
                 ep, tl.get("total", float("nan")), tl["diag_pose_grad_norm"], tl["diag_updates_skipped"])
    state = {k: v.detach().cpu() for k, v in pose_net.state_dict().items()}
    torch.save(state, path)
    _POSE_WARM_CACHE[tag] = state
    del pose_net, opt, scaler, crit
    torch.cuda.empty_cache(); gc.collect()
    return state


# ══════════════════════════════════════════════════════════════════════════════
# Core HPT trial — train + validate + early-stop + checkpoint
# ══════════════════════════════════════════════════════════════════════════════
FAIL_FAST_NO_UPDATES = True      # abort a trial whose first epoch applied zero optimizer updates

def _train_one_final_run(
    ablation_label: str,
    hpt_cfg:        Dict,
    loss_name:      str,
    arch_mode:      str,
    aug_mode:       str,
    optim_mode:     str,
    sched_mode:     str,
    device:         torch.device,
    train_loader:   DataLoader,
    val_loader:     DataLoader,
    custom_criterion: Optional[nn.Module] = None,
    spec:           Optional["LossSpec"] = None,
    ckpt_dir:       str = None,
    ckpt_meta:      Optional[Dict] = None,
) -> Tuple[float, str, List[Dict], float]:
    spec = spec_for(loss_name, spec)
    tag  = _hpt_tag(hpt_cfg)
    if custom_criterion is not None:
        criterion = copy.deepcopy(custom_criterion).to(device)
    else:
        criterion = FINAL_RECIPES[loss_name].build_criterion().to(device)
    thash = _trial_hash(hpt_cfg, loss_name, arch_mode, aug_mode, optim_mode, sched_mode, spec, criterion, train_loader)
    assert ckpt_dir is not None, "each recipe needs its own isolated checkpoint directory"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt = os.path.join(ckpt_dir, "best_model.pt")
    latest_ckpt = os.path.join(ckpt_dir, "latest_resume.pt")

    lr = float(hpt_cfg.get("lr", LEARNING_RATE))
    log.info("  [final run] recipe=%s  tag=%s  depth_lr=%.2e  pose_lr=%.2e  spec=%s",
             ablation_label, tag, lr, POSE_LR, spec)

    # PoseNet warm-up FIRST (it borrows the shared DA3 instance, the trial re-loads it afterwards)
    warm_state = None
    if spec.needs_pred_pose and POSE_WARMUP_EPOCHS > 0:
        warm_state = get_warm_pose_state(train_loader, device)

    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)

    pose_net = None
    if spec.needs_pred_pose:
        pose_net = PoseNet().to(device)
        if warm_state is not None:
            pose_net.load_state_dict(warm_state)

    depth_params = [p for p in model.parameters() if p.requires_grad]
    pose_params  = list(pose_net.parameters()) if pose_net is not None else []
    groups = [{"params": depth_params, "lr": lr}]
    if pose_params:
        groups.append({"params": pose_params, "lr": POSE_LR})          # separate depth / pose learning rates
    optimizer = build_optimizer(groups, optim_mode, lr)

    upe       = updates_per_epoch_for(len(train_loader))
    scheduler = build_scheduler(optimizer, upe, sched_mode, lr)
    scaler    = GradScaler("cuda" if torch.cuda.is_available() else "cpu", enabled=USE_GRAD_SCALER)

    best_val_rmse = float("inf")
    baseline_rmse = float("nan")
    no_improve    = 0
    global_step   = 0
    start_epoch   = 1
    history: List[Dict] = []

    # ── Resumability ─────────────────────────────────────────────────────
    resumed = False
    if os.path.exists(latest_ckpt):
        log.info("  [RESUME] Resuming from %s", os.path.basename(latest_ckpt))
        try:
            checkpoint = torch.load(latest_ckpt, map_location=device, weights_only=False)
            if (checkpoint.get("trial_hash") != thash or checkpoint.get("experiment_hash") != EXPERIMENT_HASH
                    or checkpoint.get("recipe_id") != (ckpt_meta or {}).get("recipe_id")):
                raise RuntimeError("checkpoint was produced by a different experiment configuration "
                                   f"(trial_hash {checkpoint.get('trial_hash')} vs {thash})")
            model.model.load_state_dict(checkpoint["model_state"])
            if pose_net is not None and "pose_state" in checkpoint:
                pose_net.load_state_dict(checkpoint["pose_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            if scheduler and checkpoint.get("scheduler_state"):
                scheduler.load_state_dict(checkpoint["scheduler_state"])
            scaler.load_state_dict(checkpoint["scaler_state"])
            start_epoch   = checkpoint["epoch"] + 1
            best_val_rmse = checkpoint["best_val_rmse"]
            baseline_rmse = checkpoint.get("baseline_rmse", float("nan"))
            no_improve    = checkpoint["no_improve"]
            global_step   = checkpoint["global_step"]
            history       = checkpoint.get("history", [])
            resumed = True
            log.info("  [RESUME] From epoch %d  best_val_rmse=%.4f", start_epoch, best_val_rmse)
        except Exception as e:
            log.warning("  [RESUME] Failed: %s — starting fresh.", e)
            best_val_rmse = float("inf"); no_improve = 0
            global_step = 0; start_epoch = 1; history = []

    # ── Epoch 0: pretrained baseline (cached per validation set — NOT recomputed per trial) ──
    if not resumed:
        v0 = dict(get_baseline_metrics(val_loader, device))
        baseline_rmse = v0["rmse"]
        log.info("  Ep00 (pretrained DA3 baseline)  val_rmse=%.4f  d1=%.4f", v0["rmse"], v0["d1"])
        history.append({"config": ablation_label, "hpt_tag": tag, "epoch": 0, "lr": lr,
                        "train_loss": float("nan"), **{f"val_{k}": v for k, v in v0.items()}})

    if not math.isfinite(baseline_rmse):
        baseline_rmse = get_baseline_metrics(val_loader, device)["rmse"]
    fp0 = _param_fingerprint(depth_params)

    # ── Training ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, FINAL_EPOCHS + 1):
        t0 = time.time()
        global_step, train_losses = train_one_epoch(
            model, pose_net, train_loader, criterion, optimizer,
            scaler, scheduler, device, epoch, global_step, spec,
            sched_mode=sched_mode, gpu_duty_cycle=gpu_duty_cycle,
            depth_params=depth_params, pose_params=pose_params)
        val_m = validate(model, pose_net, val_loader, criterion, device, epoch, spec)
        elapsed = time.time() - t0

        val_rmse = val_m["rmse"]
        improved = val_rmse < best_val_rmse
        log.info("  Ep%02d/%02d  lr=%.2e  train=%.4f  val_loss=%.4f  val_rmse=%.4f (base %.4f)  d1=%.4f  "
                 "|g_depth|=%.2e |g_pose|=%.2e skip=%.0f%%  (%.0fs)%s",
                 epoch, FINAL_EPOCHS, lr, train_losses.get("total", float("nan")),
                 val_m["val_loss"], val_rmse, baseline_rmse, val_m["d1"],
                 train_losses["diag_depth_grad_norm"], train_losses["diag_pose_grad_norm"],
                 100 * train_losses["diag_skip_frac"], elapsed, "  IMPROVED" if improved else "")

        rec = {"config": ablation_label, "hpt_tag": tag, "epoch": epoch, "lr": lr,
               "lr_depth_sched": optimizer.param_groups[0]["lr"],
               "lr_pose_sched": (optimizer.param_groups[1]["lr"] if len(optimizer.param_groups) > 1 else float("nan")),
               "epoch_time_s": elapsed,
               "train_loss": train_losses.get("total", float("nan")),
               **{f"val_{k}": v for k, v in val_m.items()},
               **{f"train_{k}": v for k, v in train_losses.items()
                   if k.startswith("diag_") or k in ("n_objects", "n_samples_used")}}  # v6.2.1: surface Group 1c health metrics (Bug 5); no other loss produces these keys, so this is a no-op for every other trial (pandas fills the column with NaN for them)
        history.append(rec)

        if epoch == start_epoch and FAIL_FAST_NO_UPDATES:
            if train_losses["diag_updates_applied"] == 0:
                raise RuntimeError(
                    f"[{ablation_label}/{tag}] epoch {epoch}: ZERO optimizer updates were applied "
                    f"({train_losses['diag_updates_skipped']} skipped, "
                    f"{train_losses['diag_bad_loss_batches']} non-finite losses). "
                    f"The pipeline is not training — refusing to burn GPU time.")
            fp1 = _param_fingerprint(depth_params)
            log.info("  [health] depth-parameter fingerprint change after epoch %d: %.3e (relative %.2e)",
                     epoch, fp1 - fp0, abs(fp1 - fp0) / max(fp0, 1e-12))

        def _blob(with_opt: bool) -> Dict:
            b = {"epoch": epoch, "model_state": model.model.state_dict(), "val_rmse": val_rmse,
                 "hpt_cfg": hpt_cfg, "config_label": ablation_label,
                 "trial_hash": thash, "experiment_hash": EXPERIMENT_HASH,
                 "loss_name": loss_name, "arch_mode": arch_mode, **(ckpt_meta or {})}
            if pose_net is not None:
                b["pose_state"] = pose_net.state_dict()
            if with_opt:
                b.update({"optimizer_state": optimizer.state_dict(),
                          "scheduler_state": scheduler.state_dict() if scheduler else None,
                          "scaler_state": scaler.state_dict(), "best_val_rmse": best_val_rmse,
                          "baseline_rmse": baseline_rmse, "no_improve": no_improve,
                          "global_step": global_step, "history": history})
            return b

        if improved:
            best_val_rmse = val_rmse
            no_improve = 0
            torch.save(_blob(False), ckpt)
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                log.info("  Early stop @ epoch %d (no improvement for %d ep)", epoch, EARLY_STOP_PATIENCE)
                torch.save(_blob(True), latest_ckpt)
                break

        torch.save(_blob(True), latest_ckpt)
        if SAVE_EVERY_N_EPOCHS > 0 and epoch % SAVE_EVERY_N_EPOCHS == 0:
            torch.save(_blob(False), os.path.join(ckpt_dir, f"epoch_{epoch:03d}.pt"))

    log.info("  Done.  best_val_rmse=%.4f  baseline=%.4f  (ckpt: %s)",
             best_val_rmse, baseline_rmse, os.path.basename(ckpt))
    if best_val_rmse >= baseline_rmse - 1e-4:
        log.warning("  [health] this trial did NOT improve on the pretrained baseline (%.4f vs %.4f)",
                    best_val_rmse, baseline_rmse)

    del model, criterion, optimizer, scaler
    if pose_net is not None:
        del pose_net
    torch.cuda.empty_cache()
    gc.collect()
    return best_val_rmse, ckpt, history, baseline_rmse

## 10 · Pre-flight Sanity Checks

A compact final version of the source's sanity suite (source Section 13b). The **underlying tests are the source's**: `check_gt_pose_convention` (0a), the smoke-test procedure (0b: real batch → strict routing → backward → gradient at the depth map), `run_sanity_experiment` (short real training: updates applied, skip fraction, non-zero depth gradient, **non-zero PoseNet gradient for C recipes**, parameters actually changed) and `pose_scale_check`. They are re-pointed at the six final recipes; the supervised control (Exp A), the stationary-object coverage check and all ablation tables are gone.

| Check | What must hold |
|---|---|
| **0a** (only if an OXTS recipe is selected) | warping `I_{t+1}` with LiDAR depth + OXTS pose reproduces `I_t` better than identity **and** better than the inverse pose. *(LiDAR is used here as a diagnostic of the pose convention only — never in a loss.)* |
| **0b** (per recipe) | strict routing: `gt=None`; finite, **non-zero** `dL/d(depth)`; `l_gt is None` |
| **0c** (new, numeric) | loss composition matches the recipe definition (A3 has no anchor term; C3 = Photo-L1 + 0.1·flow_smooth; C5 = Photo-L1 + 0.1·feature_metric; C3 ≠ C1; C3's `flow_smooth` equals the one `RigidFlowConsistencyLoss` reports on the same inputs) |
| **short run** (per recipe) | updates applied, ≤ 25 % skipped, depth gradient > 0, parameters changed, PoseNet gradient > 0 (C) |

Training refuses to start if any check fails (`REQUIRE_SANITY_PASS`). The pre-flight uses the *same* loaders as the final training, so the PoseNet warm-up computed here is reused (not recomputed) by the final run.

In [20]:
@torch.no_grad()
def check_gt_pose_convention(loader: DataLoader, device: torch.device, n_batches: int = 4) -> Dict[str, float]:
    """
    LiDAR-based test of the OXTS pose: at pixels with LiDAR depth, warping I_{t+1} into I_t with the
    GT depth and the OXTS pose must reproduce I_t better than (a) identity and (b) the inverse pose.
    This catches wrong direction / wrong axes (the v5 OXTS pose was expressed in IMU axes).
    """
    errs = {"identity": [], "gt_pose": [], "inverse": []}
    tvals = []
    ni = SOURCE_OFFSETS.index(+1)
    for bi, batch in enumerate(loader):
        if bi >= n_batches:
            break
        assert bool(batch["has_gt_pose"].all())
        rgb = batch["rgb_raw"].to(device); nxt = batch["next_rgb_raw"].to(device)
        depth = batch["depth_gt"].to(device); valid = batch["valid_mask"].to(device)
        K = batch["K"].to(device); Tn = batch["gt_pose_next"].to(device)
        tvals.append(Tn[:, :3, 3].abs().mean(0).cpu().numpy())
        for name, T_ in (("identity", torch.eye(4, device=device).expand_as(Tn)),
                         ("gt_pose", Tn), ("inverse", invert_se3(Tn))):
            w, v = warp_image(nxt, depth, T_, K)
            m = valid * v
            errs[name].append(float(((w - rgb).abs().mean(1, keepdim=True) * m).sum() / (m.sum() + EPS)))
    r = {k: float(np.mean(v)) for k, v in errs.items()}
    tabs = np.mean(tvals, axis=0)
    r["mean_abs_t_xyz"] = tabs.tolist()
    r["ok"] = bool(r["gt_pose"] < r["identity"] and r["gt_pose"] < r["inverse"])
    return r


def pose_scale_check(pose_net: nn.Module, oracle_val_loader: DataLoader, device: torch.device, n_batches: int = 4) -> float:
    """Mean |t_pred| / mean |t_gt| on oracle-val batches (1.0 = metric scale).  Reported, not a pass criterion:
    photometric self-supervision does not by itself guarantee metric scale."""
    pose_net.eval(); ratios = []
    ni = SOURCE_OFFSETS.index(+1)
    with torch.no_grad():
        for bi, batch in enumerate(oracle_val_loader):
            if bi >= n_batches: break
            rgb, nxt = _to(batch["rgb"], device), _to(batch["next_rgb"], device)
            t_pred = pose_to_se3(pose_net(rgb, nxt))[:, :3, 3].norm(dim=1)
            t_gt = _to(batch["gt_pose_next"], device)[:, :3, 3].norm(dim=1)
            ratios.append(float((t_pred.mean() / t_gt.mean().clamp(min=1e-6))))
    return float(np.mean(ratios)) if ratios else float("nan")


def run_sanity_experiment(exp_id: str, loss_name: str, train_loader: DataLoader,
                          val_loader: DataLoader, device: torch.device,
                          scale_loader: Optional[DataLoader] = None) -> Dict:
    spec = FINAL_LOSS_SPECS[loss_name]
    warm = get_warm_pose_state(train_loader, device, tag_extra="") if (spec.needs_pred_pose and POSE_WARMUP_EPOCHS > 0) else None
    model = load_da3_model_fast(device); _apply_arch(model, "decoder_only")
    pose_net = None
    if spec.needs_pred_pose:
        pose_net = PoseNet().to(device)
        if warm is not None:
            pose_net.load_state_dict(warm)
    depth_params = [p for p in model.parameters() if p.requires_grad]
    pose_params  = list(pose_net.parameters()) if pose_net is not None else []
    groups = [{"params": depth_params, "lr": SANITY_LR}]
    if pose_params:
        groups.append({"params": pose_params, "lr": POSE_LR})
    opt = build_optimizer(groups, "adamw", SANITY_LR)
    scaler = GradScaler("cuda" if torch.cuda.is_available() else "cpu", enabled=USE_GRAD_SCALER)
    crit = FINAL_RECIPES[loss_name].build_criterion().to(device)

    v0 = validate(model, pose_net, val_loader, crit, device, 0, spec, max_batches=SANITY_VAL_BATCHES)
    fp0 = _param_fingerprint(depth_params)
    _, tl = train_one_epoch(model, pose_net, train_loader, crit, opt, scaler, None, device, 1, 0, spec,
                            depth_params=depth_params, pose_params=pose_params,
                            max_batches=SANITY_TRAIN_BATCHES)
    fp1 = _param_fingerprint(depth_params)
    v1 = validate(model, pose_net, val_loader, crit, device, 1, spec, max_batches=SANITY_VAL_BATCHES)

    dg = tl["diag_depth_grad_norm"]
    r = dict(exp=exp_id, loss=loss_name, category=spec.category,
             val_rmse_before=v0["rmse"], val_rmse_after=v1["rmse"], d_rmse=v1["rmse"] - v0["rmse"],
             train_loss=tl.get("total", float("nan")), depth_grad=dg, pose_grad=tl["diag_pose_grad_norm"],
             updates=tl["diag_updates_applied"], skipped=tl["diag_updates_skipped"],
             skip_frac=tl["diag_skip_frac"], bad_loss_batches=tl["diag_bad_loss_batches"],
             param_rel_change=abs(fp1 - fp0) / max(fp0, 1e-12),
             valid_frac=tl.get("valid_frac", float("nan")), auto_mask_frac=tl.get("auto_mask_frac", float("nan")))
    if pose_net is not None and scale_loader is not None:
        r["pose_scale_ratio"] = pose_scale_check(pose_net, scale_loader, device)   # 1.0 = metric; informational
    checks = {
        "updates_applied":   r["updates"] > 0,
        "skip_frac_ok":      r["skip_frac"] <= SANITY_MAX_SKIP_FRAC,
        "depth_grad_nonzero": bool(np.isfinite(dg) and dg > 1e-9),
        "params_changed":    r["param_rel_change"] > 1e-10,
    }
    if spec.needs_pred_pose:                      # a dead PoseNet can hide behind an indirect depth update
        pg = r["pose_grad"]
        checks["pose_grad_nonzero"] = bool(np.isfinite(pg) and pg > 1e-9)
    if exp_id == "A":
        checks["rmse_improves"] = r["val_rmse_after"] < r["val_rmse_before"]
    r["checks"] = checks
    r["PASS"] = all(checks.values())
    del model, pose_net, opt, scaler, crit
    torch.cuda.empty_cache(); gc.collect()
    return r


# ── 0b / 0c: strict-routing gradient smoke test + numeric composition check (adapted from `gradient_smoke_test`) ──
def final_gradient_smoke_test(device: torch.device, recipe_ids: List[str]) -> Tuple[pd.DataFrame, Dict[str, Dict[str, float]]]:
    """One real batch per recipe (its own loader: OXTS loader for A/B, plain SSL loader for C, LiDAR NOT loaded), pretrained DA3
    depth as a leaf tensor, PoseNet from the warm-up state, `prepare_step` -> `_unified_forward` with the strict routing, backward.
    Reports the gradient reaching the depth map and asserts that no GT depth was visible to the loss."""
    rows, term_vals = [], {}
    need_pose = any(FINAL_RECIPES[r].needs_pred_pose for r in recipe_ids)
    pn = None
    if need_pose:
        ssl_loader = make_train_loader(PREFLIGHT_AUG_MODE, load_gt=False)
        warm = get_warm_pose_state(ssl_loader, device) if POSE_WARMUP_EPOCHS > 0 else None
    model = load_da3_model_fast(device); _apply_arch(model, ARCH_MODE)
    model.model.eval()                         # deterministic forward, so C1 / C3 terms are comparable on the same batch
    if need_pose:
        pn = PoseNet().to(device)
        if warm is not None:
            pn.load_state_dict(warm)
        pn.eval()
    batches = {}
    for rid in recipe_ids:
        spec = FINAL_RECIPES[rid].spec
        key = ("oxts" if spec.needs_gt_pose else "ssl")
        if key not in batches:
            ldr = make_train_loader(PREFLIGHT_AUG_MODE, load_gt=False, oracle=spec.needs_gt_pose)
            batches[key] = next(iter(ldr))
    for rid in recipe_ids:
        r = FINAL_RECIPES[rid]; spec = r.spec
        row = {"recipe": rid, "group": r.group}
        try:
            assert_recipe_safety(r)
            crit = r.build_criterion().to(device)
            batch = batches["oxts" if spec.needs_gt_pose else "ssl"]
            assert "depth_gt" not in batch and "valid_mask" not in batch, "training batch must not carry LiDAR (load_gt=False)"
            dyn = (get_batch_dynamic_masks(batch, *batch["rgb"].shape[-2:]) if (spec.needs_dyn_mask and USE_DYNAMIC_MASK) else None)
            with torch.no_grad():
                pred, gt, mask, ctx, l_gt, l_mask = prepare_step(batch, model, pn if spec.needs_pred_pose else None, spec, device,
                                                                 train_depth=False, dyn_mask=dyn)
            assert l_gt is None and l_mask is None and gt is None and mask is None, "GT depth visible to a self-supervised loss"
            if spec.needs_gt_pose:
                assert ctx["poses"] is None and ctx["gt_poses"] is not None       # OXTS recipes never see PoseNet output
            else:
                assert ctx["gt_poses"] is None and ctx["poses"] is not None       # C recipes never see OXTS pose
            leaf = pred.detach().clone().requires_grad_(True)
            loss, terms = _unified_forward(crit, leaf, l_gt, l_mask, **ctx)
            loss.backward(); g = leaf.grad
            term_vals[rid] = {k: float(v) for k, v in terms.items() if torch.is_tensor(v) and v.numel() == 1}
            row.update(loss_value=float(loss), grad_norm=float(g.norm()), grad_finite=bool(torch.isfinite(g).all()),
                       gt_depth_visible=False, valid_frac=float(terms.get("valid_frac", float("nan"))),
                       dyn_mask_frac=(float(dyn.mean()) if dyn is not None else float("nan")))
            row["status"] = "ok" if (row["grad_finite"] and row["grad_norm"] > 0) else "FAIL (zero / non-finite gradient)"
        except Exception as e:
            row["status"] = f"FAIL ({type(e).__name__}: {e})"
        rows.append(row)
    del model, pn; torch.cuda.empty_cache()
    return pd.DataFrame(rows), term_vals


def check_recipe_compositions(tv: Dict[str, Dict[str, float]], rtol: float = 1e-4) -> List[str]:
    """0c: the returned total must equal the recipe's weighted sum of its logged terms, and the recipes must stay distinct."""
    errs, W = [], RECIPE_WEIGHTS
    close = lambda a, b: abs(a - b) <= rtol * max(1.0, abs(a), abs(b))
    if "A1_gt_pose_photo" in tv:
        t = tv["A1_gt_pose_photo"]
        if not close(t["total"], t["gt_photo"]): errs.append("A1 total != 1.0*gt_photo")
        if set(t) - {"total", "gt_photo"}: errs.append(f"A1 has unexpected terms {set(t) - {'total', 'gt_photo'}}")
    if "A3_gt_pose_photo_smooth_temporal" in tv:
        t = tv["A3_gt_pose_photo_smooth_temporal"]; w = W["A3_gt_pose_photo_smooth_temporal"]
        if "anchor" in t: errs.append("A3 carries an anchor term")
        if set(t) != {"total", "gt_photo", "smooth", "gt_temporal"}: errs.append(f"A3 term set {set(t)}")
        elif not close(t["total"], w["gt_photo"] * t["gt_photo"] + w["smooth"] * t["smooth"] + w["gt_temporal"] * t["gt_temporal"]):
            errs.append("A3 total != 1.0*gt_photo + 1e-3*smooth + 0.1*gt_temporal")
    if "B1_gt_pose_rigid_flow" in tv:
        t = tv["B1_gt_pose_rigid_flow"]
        if not close(t["total"], t["gt_flow"]): errs.append("B1 total != 1.0*gt_flow")
    if "C3_photo_l1_flow_smooth" in tv:
        t = tv["C3_photo_l1_flow_smooth"]
        if not close(t["total"], t["photo_l1"] + 0.1 * t["flow_smooth"]): errs.append("C3 total != photo_l1 + 0.1*flow_smooth")
        if "C1_rigid_flow" in tv:
            c1 = tv["C1_rigid_flow"]
            if not close(c1["total"], c1["photo"] + 0.1 * c1["flow_smooth"]): errs.append("C1 total != photo + 0.1*flow_smooth (source formulation)")
            if not close(t["flow_smooth"], c1["flow_smooth"]): errs.append("C3 flow_smooth differs from RigidFlowConsistencyLoss flow_smooth")
            if close(t["total"], c1["total"]) and close(t["photo_l1"], c1["photo"]): errs.append("C3 is numerically identical to C1")
    if "C5_photo_l1_feature_metric" in tv:
        t = tv["C5_photo_l1_feature_metric"]
        if not close(t["total"], t["photo_l1"] + 0.1 * t["feature_metric"]): errs.append("C5 total != photo_l1 + 0.1*feature_metric")
    return errs


SANITY_PASSED: Optional[bool] = None


def run_final_preflight(device: torch.device, recipe_ids: List[str]) -> bool:
    global SANITY_PASSED
    log.info("=" * 70); log.info("FINAL PRE-FLIGHT  (recipes: %s)", recipe_ids); log.info("=" * 70)
    ok_all = True
    for rid in recipe_ids:
        assert_recipe_safety(FINAL_RECIPES[rid])
    any_oxts = any(FINAL_RECIPES[r].needs_gt_pose for r in recipe_ids)
    oracle_val, oracle_ok = None, False
    try:
        oracle_val = get_oracle_val_loader(); oracle_ok = True
    except Exception as e:
        log.error("OXTS loaders unavailable: %s", e)
    if any_oxts and not oracle_ok:
        ok_all = False
        log.error("an OXTS recipe is selected but the OXTS loaders are unavailable -> pre-flight FAILS")

    # ── 0a: GT pose convention (only needed when an OXTS recipe is selected) ───────────────────────
    if any_oxts and oracle_ok:
        r = check_gt_pose_convention(oracle_val, device)
        log.info("[0a] LiDAR photometric L1 | identity=%.4f  OXTS-pose=%.4f  inverse=%.4f  |t|xyz(m)=%s  -> %s",
                 r["identity"], r["gt_pose"], r["inverse"], np.round(r["mean_abs_t_xyz"], 2), "PASS" if r["ok"] else "FAIL")
        if not r["ok"]:
            ok_all = False
            log.error("[0a] OXTS pose does not beat identity/inverse -> the OXTS recipes are NOT trustworthy. Check calibration / axes.")

    # ── 0b + 0c: strict routing, gradients, numeric composition ────────────────────────────────────
    smoke, tv = final_gradient_smoke_test(device, recipe_ids)
    print(smoke.to_string(index=False))
    smoke.to_csv(os.path.join(RESULTS_DIR, "preflight_gradient_smoke.csv"), index=False)
    bad = smoke[smoke["status"].str.startswith("FAIL")]
    if len(bad):
        ok_all = False
        log.error("[0b] %d recipe(s) failed the gradient smoke test: %s", len(bad), bad["recipe"].tolist())
    comp_errs = check_recipe_compositions(tv)
    for e in comp_errs:
        log.error("[0c] %s", e)
    ok_all &= not comp_errs
    log.info("[0c] recipe composition check: %s", "PASS" if not comp_errs else "FAIL")

    # ── short real training run per recipe ─────────────────────────────────────────────────────────
    rows = []
    if RUN_PREFLIGHT_SHORT_TRAINING:
        for rid in recipe_ids:
            spec = FINAL_RECIPES[rid].spec
            trl, val = loaders_for(spec, aug_mode=PREFLIGHT_AUG_MODE)
            log.info("[short run] %s ...", rid)
            r = run_sanity_experiment(rid, rid, trl, val, device,
                                      scale_loader=(oracle_val if (oracle_ok and spec.needs_pred_pose) else None))
            rows.append(r)
            log.info("[short run] %s | RMSE %.4f -> %.4f | |g_depth|=%.2e |g_pose|=%.2e | updates=%d skipped=%d%s | %s",
                     rid, r["val_rmse_before"], r["val_rmse_after"], r["depth_grad"], r["pose_grad"], r["updates"], r["skipped"],
                     (f" | pose-scale ratio {r['pose_scale_ratio']:.2f}" if "pose_scale_ratio" in r else ""),
                     "PASS" if r["PASS"] else f"FAIL {r['checks']}")
            ok_all &= r["PASS"]
        tab = pd.DataFrame(rows)
        if len(tab):
            show = [c for c in tab.columns if c != "checks"]
            print(tab[show].to_string(index=False))
            tab[show].to_csv(os.path.join(RESULTS_DIR, "preflight_short_runs.csv"), index=False)

    SANITY_PASSED = bool(ok_all)
    log.info("PRE-FLIGHT: %s", "ALL CHECKS PASSED — final training released" if SANITY_PASSED
             else "FAILED — fix the pipeline before training")
    return SANITY_PASSED

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if RUN_SANITY_EXPERIMENTS:
    run_final_preflight(device, ACTIVE_RECIPES)
else:
    log.info("RUN_SANITY_EXPERIMENTS=False — pre-flight gate disabled.")

21:06:09  INFO      ======================================================================
21:06:09  INFO      FINAL PRE-FLIGHT  (recipes: ['B1_gt_pose_rigid_flow'])
21:06:09  INFO      ======================================================================
21:06:15  INFO      [KITTI-Seq-val] 23092 samples (offsets=(-1, 1), aug=none)
21:06:15  INFO      [OXTSDataset] 33 drives with GT pose | 23092 samples kept | 0 dropped (no GT pose)
21:06:21  INFO      [0a] LiDAR photometric L1 | identity=0.0795  OXTS-pose=0.0314  inverse=0.1017  |t|xyz(m)=[0.01 0.01 0.77]  -> PASS
21:06:21  INFO        [arch=decoder_only] 29.8M params trainable
21:06:21  INFO        [arch=decoder_only] trainable 29.8M / 334.2M params (8.9 %)
21:06:21  INFO            backbone                 trainable    0.00M /  304.37M
21:06:21  INFO            head                     trainable   29.80M /   29.80M
21:06:29  INFO      [KITTI-Seq-train] 23092 samples (offsets=(-1, 1), aug=base)
21:06:29  INFO      [OXTSDataset] 33 d

               recipe group  loss_value  grad_norm  grad_finite  gt_depth_visible  valid_frac  dyn_mask_frac status
B1_gt_pose_rigid_flow     B    0.101747   0.000186         True             False         NaN            NaN     ok


21:06:50  INFO          batch 0/120
21:07:49  INFO          batch 50/120
21:08:46  INFO          batch 100/120
21:09:23  INFO      [short run] B1_gt_pose_rigid_flow | RMSE 3.0926 -> 3.3307 | |g_depth|=1.03e+00 |g_pose|=nan | updates=30 skipped=0 | PASS
21:09:23  INFO      PRE-FLIGHT: ALL CHECKS PASSED — final training released


                  exp                  loss      category  val_rmse_before  val_rmse_after   d_rmse  train_loss  depth_grad  pose_grad  updates  skipped  skip_frac  bad_loss_batches  param_rel_change  valid_frac  auto_mask_frac  PASS
B1_gt_pose_rigid_flow B1_gt_pose_rigid_flow oxts_pose_ssl         3.092641        3.330732 0.238091    0.137338    1.030452        NaN       30        0        0.0                 0          0.000021         NaN             NaN  True


## 11 · Final Training Runner & Per-Recipe Training

### 11.1 · One recipe end-to-end: train → select on validation → evaluate on test **once**

`run_final_recipe` (NEW, assembled from the source's runner phases) trains one recipe via `_train_one_final_run`, then — only after training has finished — loads the **best-validation** checkpoint and evaluates it on the Eigen test split exactly once (source Phase 3), persists the CSVs and keeps every checkpoint in that recipe's own directory `checkpoints/<recipe_id>/`.

**Isolation guarantees:** (i) each recipe re-loads the pristine pretrained state (`load_da3_model_fast`); (ii) its checkpoints live in their own directory and are only resumed if `trial_hash`, `experiment_hash` **and** `recipe_id` match; (iii) safety assertions run before training: `uses_gt_depth is False`, the right pose source, and — new — the training dataset was built with `load_gt=False` (no LiDAR I/O at all).

**Test isolation:** the test set is never used for checkpoint selection, early stopping, recipe selection during training, or the learning rate. Selection uses the validation split only. (Note: validation metrics are computed against sparse LiDAR — evaluation only, never inside a loss — as in the ablation.)

In [22]:
def get_completed_recipes() -> set:
    """Recipes already in the FINAL summary CSV **for THIS experiment hash** (rows from other configurations are never skipped-over)."""
    if not os.path.exists(SUMMARY_CSV):
        return set()
    try:
        df = pd.read_csv(SUMMARY_CSV)
    except pd.errors.EmptyDataError:
        return set()
    if "recipe" not in df.columns or "experiment_hash" not in df.columns:
        return set()
    stale = df[df["experiment_hash"] != EXPERIMENT_HASH]
    if len(stale):
        log.warning("SUMMARY_CSV contains %d row(s) from a DIFFERENT configuration (%s) — not treated as completed work.",
                    len(stale), sorted(stale["experiment_hash"].astype(str).unique()))
    return set(df[df["experiment_hash"] == EXPERIMENT_HASH]["recipe"].unique())


def run_final_recipe(rid: str, device: torch.device, test_df: pd.DataFrame) -> Dict:
    recipe, spec = FINAL_RECIPES[rid], FINAL_RECIPES[rid].spec
    assert_recipe_safety(recipe)
    assert not recipe.uses_gt_depth
    if recipe.group in ("A", "B"):
        assert recipe.needs_gt_pose
        assert not recipe.needs_pred_pose
    else:
        assert not recipe.needs_gt_pose

    train_loader, val_loader = loaders_for(spec, aug_mode=AUG_MODE)
    train_base = train_loader.dataset.dataset
    assert train_base.load_gt is False, "final SSL training must not read LiDAR"
    assert isinstance(train_base, KITTISequentialOXTSDataset) == bool(spec.needs_gt_pose), "wrong dataset class for this recipe's pose source"
    ckpt_dir = os.path.join(CHECKPOINT_DIR, rid)
    ckpt_meta = dict(recipe_id=rid, recipe_group=recipe.group, recipe_name=recipe.name, loss_composition=recipe.loss_composition,
                     recipe_terms=dict(recipe.terms), w_anchor=W_ANCHOR, pose_source=recipe.pose_source, uses_gt_depth=False,
                     validated_in_ablation=recipe.validated_in_ablation, model_id=MODEL_ID, learning_rate=LEARNING_RATE,
                     pose_lr=(POSE_LR if spec.needs_pred_pose else None), weight_decay=WEIGHT_DECAY, aug_mode=AUG_MODE,
                     optim_mode=OPTIM_MODE, sched_mode=SCHED_MODE, final_epochs=FINAL_EPOCHS, early_stop_patience=EARLY_STOP_PATIENCE,
                     batch_size=BATCH_SIZE, accum_steps=ACCUM_STEPS, code_version=EXPERIMENT_CODE_VERSION,
                     source_notebook="DA3_SelfSupervised_FT_Ablation_HPT_v6_2_1_stationary_fix.ipynb")

    log.info("")
    log.info("=" * 70)
    log.info("FINAL RECIPE  %s   [%s]", rid, recipe.loss_composition)
    log.info("  pose source: %s | pred_pose=%s gt_pose=%s pred_next=%s dyn_mask=%s | uses_GT_depth=%s",
             recipe.pose_source, spec.needs_pred_pose, spec.needs_gt_pose, spec.needs_pred_next, spec.needs_dyn_mask, spec.uses_gt_depth)
    log.info("  %d epochs max, patience %d, lr %.1e, %s + %s, arch %s, aug %s", FINAL_EPOCHS, EARLY_STOP_PATIENCE, LEARNING_RATE,
             OPTIM_MODE, SCHED_MODE, ARCH_MODE, AUG_MODE)
    log.info("=" * 70)

    t_wall = time.time()
    best_val_rmse, ckpt, history, baseline_rmse = _train_one_final_run(
        ablation_label=rid, hpt_cfg={"lr": LEARNING_RATE}, loss_name=rid, arch_mode=ARCH_MODE, aug_mode=AUG_MODE,
        optim_mode=OPTIM_MODE, sched_mode=SCHED_MODE, device=device, train_loader=train_loader, val_loader=val_loader,
        custom_criterion=recipe.criterion_proto, spec=spec, ckpt_dir=ckpt_dir, ckpt_meta=ckpt_meta)
    wall_s = time.time() - t_wall

    # ── test evaluation, ONCE, on the best-VALIDATION checkpoint ────────────────────────────────────
    log.info("Loading best-validation checkpoint -> test evaluation (ONCE) ...")
    model = load_da3_model_fast(device)
    _apply_arch(model, ARCH_MODE)
    ckpt_data = torch.load(ckpt, map_location=device, weights_only=False)
    assert ckpt_data["experiment_hash"] == EXPERIMENT_HASH and ckpt_data["recipe_id"] == rid, "best checkpoint belongs to another run"
    model.model.load_state_dict(ckpt_data["model_state"])
    log.info("Best ckpt: epoch=%d  val_rmse=%.4f", ckpt_data["epoch"], ckpt_data["val_rmse"])
    frame_rows = evaluate_model(model, test_df, device, rid)
    summary = aggregate_results(frame_rows, rid)

    hist_df = pd.DataFrame(history)
    train_rows = hist_df[hist_df["epoch"] > 0]
    summary.update({
        "recipe": rid, "recipe_description": recipe.description, "recipe_group": recipe.group, "loss_composition": recipe.loss_composition,
        "pose_source": recipe.pose_source, "uses_gt_depth": bool(spec.uses_gt_depth),
        "validated_in_ablation": bool(recipe.validated_in_ablation),
        "architecture": ARCH_MODE, "learning_rate": LEARNING_RATE, "augmentation": AUG_MODE,
        "optimizer": OPTIM_MODE, "scheduler": SCHED_MODE, "weight_decay": WEIGHT_DECAY,
        "epochs_planned": FINAL_EPOCHS, "epochs_run": int(train_rows["epoch"].max()) if len(train_rows) else 0,
        "best_epoch": int(ckpt_data["epoch"]), "best_val_rmse": float(best_val_rmse),
        "final_val_rmse": float(train_rows["val_rmse"].iloc[-1]) if len(train_rows) else float("nan"),
        "baseline_val_rmse": float(baseline_rmse),
        "training_time_s": float(np.nansum(train_rows["epoch_time_s"])) if "epoch_time_s" in train_rows else float("nan"),
        "wall_time_this_session_s": float(wall_s),
        "number_of_updates": int(np.nansum(train_rows["train_diag_updates_applied"])),
        "skipped_updates": int(np.nansum(train_rows["train_diag_updates_skipped"])),
        "best_checkpoint": ckpt, "experiment_hash": EXPERIMENT_HASH, "n_test_frames": len(frame_rows),
    })

    header = not os.path.exists(SUMMARY_CSV)
    pd.DataFrame([summary]).to_csv(SUMMARY_CSV, mode="a", header=header, index=False)
    hist_df.to_csv(os.path.join(RESULTS_DIR, f"history_{rid}.csv"), index=False)
    pd.DataFrame(frame_rows).to_csv(os.path.join(RESULTS_DIR, f"test_frames_{rid}.csv"), index=False)
    log.info("  CSVs written: summary row, history_%s.csv, test_frames_%s.csv", rid, rid)
    del model
    torch.cuda.empty_cache(); gc.collect()
    log.info("  -- recipe %s complete --", rid)
    return summary

print("run_final_recipe ready.")

run_final_recipe ready.


### 11.2 · Run the selected recipes

Runs the selected recipes **sequentially, each as an independent fine-tuning of the same pretrained DA3METRIC-LARGE**. Resumable: a recipe already present in `final_summary.csv` for the current experiment hash is skipped, and an interrupted recipe resumes from its own `latest_resume.pt`. Sections 12 / 13 contain the validation history and the test-evaluation read-outs.

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Gate: never launch the final training on a pipeline that has not proven it trains ──────────
if RUN_SANITY_EXPERIMENTS and REQUIRE_SANITY_PASS and SANITY_PASSED is not True:
    raise RuntimeError("Pre-flight (Section 10) did not pass (SANITY_PASSED=%r). Fix the reported problem and re-run Section 10, or set "
                       "REQUIRE_SANITY_PASS=False if you really want to proceed." % (SANITY_PASSED,))

print_final_banner()
completed_recipes = get_completed_recipes()
if completed_recipes:
    log.info("[RESUME] already complete for this experiment hash (skipped): %s", sorted(completed_recipes))

all_final_results: List[Dict] = []
for rid in ACTIVE_RECIPES:                       # every recipe restarts from the same pretrained weights (load_da3_model_fast)
    if rid in completed_recipes:
        log.info("  [SKIP] %s already complete", rid); continue
    all_final_results.append(run_final_recipe(rid, device, test_df))

log.info("\n*** Final fine-tuning finished for: %s ***", [r["recipe"] for r in all_final_results] or "nothing new")

21:09:23  WARNING   SUMMARY_CSV contains 2 row(s) from a DIFFERENT configuration (['8687cb9f7c', 'e7f5923416']) — not treated as completed work.
21:09:23  INFO      [RESUME] already complete for this experiment hash (skipped): ['C1_rigid_flow', 'C3_photo_l1_flow_smooth']
21:09:23  INFO      
21:09:23  INFO      ======================================================================
21:09:23  INFO      FINAL RECIPE  B1_gt_pose_rigid_flow   [1.0*GT_Pose_Rigid_Flow]
21:09:23  INFO        pose source: OXTS (GT camera-frame relative pose) | pred_pose=False gt_pose=True pred_next=False dyn_mask=False | uses_GT_depth=False
21:09:23  INFO        50 epochs max, patience 10, lr 1.0e-06, adamw + cosine_warmup, arch decoder_only, aug base
21:09:23  INFO      ======================================================================


FINAL DA3METRIC-LARGE SELF-SUPERVISED FINE-TUNING
-------------------------------------------------
Model: DA3METRIC-LARGE
Architecture: decoder_only
Learning rate: 1e-06   (PoseNet LR 0.0001, C-recipes only)   Weight decay: 0.0001
Optimizer: AdamW
Scheduler: cosine warmup (1 warm-up epoch(s), stepped per optimizer update)
Augmentation: base
Epochs: 50   (early-stop patience 10)
Batch size: 8   Accumulation: 4   Effective batch size: 32
Mixed precision: fp16 autocast for DA3; geometry + losses in fp32
Experiment hash: b093e0aeb3

Selected recipes:
  B1_gt_pose_rigid_flow              [B] 1.0*GT_Pose_Rigid_Flow
                                         pose=OXTS     pred_pose=False gt_pose=True  pred_next=False dyn_mask=False uses_gt_depth=False  [validated in ablation: Group 5b K (K_gt_rigid_flow_only)]


21:09:23  INFO        [final run] recipe=B1_gt_pose_rigid_flow  tag=lr=1e-06  depth_lr=1.00e-06  pose_lr=1.00e-04  spec=LossSpec(needs_pred_pose=False, needs_gt_pose=True, needs_pred_next=False, needs_tracking=False, uses_gt_depth=False, needs_dyn_mask=False, category='oxts_pose_ssl')
21:09:23  INFO        [arch=decoder_only] 29.8M params trainable
21:09:23  INFO        [arch=decoder_only] trainable 29.8M / 334.2M params (8.9 %)
21:09:23  INFO            backbone                 trainable    0.00M /  304.37M
21:09:23  INFO            head                     trainable   29.80M /   29.80M
21:12:34  INFO      [baseline] pretrained DA3 on validation set: rmse=3.1397  d1=0.9258  (computed once, cached)
21:12:34  INFO        Ep00 (pretrained DA3 baseline)  val_rmse=3.1397  d1=0.9258
21:12:38  INFO          batch 0/2597
21:13:37  INFO          batch 50/2597
21:14:36  INFO          batch 100/2597
21:15:36  INFO          batch 150/2597
21:16:36  INFO          batch 200/2597
21:17:36  INFO     

Eval: B1_gt_pose_rigid_flow:   0%|          | 0/652 [00:00<?, ?it/s]

07:33:34  INFO        CSVs written: summary row, history_B1_gt_pose_rigid_flow.csv, test_frames_B1_gt_pose_rigid_flow.csv
07:33:34  INFO        -- recipe B1_gt_pose_rigid_flow complete --
07:33:34  INFO      
*** Final fine-tuning finished for: ['B1_gt_pose_rigid_flow'] ***


## 12 · Validation

Read-out of the validation history of every finished recipe (`results/history_<recipe>.csv`). Validation RMSE (metres, unscaled, sparse-LiDAR evaluation only) is the **only** quantity used for checkpoint selection and early stopping. Epoch 0 is the pretrained DA3METRIC-LARGE baseline on the same validation set.

In [24]:
def load_final_summary() -> pd.DataFrame:
    df = pd.read_csv(SUMMARY_CSV)
    # df = df[df["experiment_hash"] == EXPERIMENT_HASH].drop_duplicates(subset="recipe", keep="last")
    df = df.drop_duplicates(subset="recipe", keep="last")
    order = {rid: i for i, rid in enumerate(FINAL_RECIPES)}
    return df.sort_values("recipe", key=lambda s: s.map(order)).reset_index(drop=True)


final_df = load_final_summary()
rows = []
for rid in final_df["recipe"]:
    h = pd.read_csv(os.path.join(RESULTS_DIR, f"history_{rid}.csv"))
    tr = h[h["epoch"] > 0]
    base = float(h.loc[h["epoch"] == 0, "val_rmse"].iloc[0]) if (h["epoch"] == 0).any() else float("nan")
    best = tr.loc[tr["val_rmse"].idxmin()]
    rows.append(dict(recipe=rid, epochs_run=int(tr["epoch"].max()), epochs_planned=FINAL_EPOCHS,
                     early_stopped=bool(tr["epoch"].max() < FINAL_EPOCHS),
                     val_rmse_pretrained=base, val_rmse_best=float(best["val_rmse"]), best_epoch=int(best["epoch"]),
                     val_rmse_final=float(tr["val_rmse"].iloc[-1]),
                     gain_vs_pretrained_m=base - float(best["val_rmse"]),
                     val_d1_at_best=float(best.get("val_d1", float("nan"))),
                     val_loss_at_best=float(best.get("val_val_loss", float("nan")))))
val_table = pd.DataFrame(rows)
print("VALIDATION (selection metric: unscaled val RMSE, m; epoch 0 = pretrained DA3METRIC-LARGE)")
print(val_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
val_table.to_csv(os.path.join(ANALYSIS_DIR, "validation_readout.csv"), index=False)
for rid in val_table.loc[val_table["val_rmse_best"] >= val_table["val_rmse_pretrained"] - 1e-3, "recipe"]:
    print(f"[health] {rid}: did NOT beat the pretrained baseline on validation — treat as 'no effect', not as a ranking.")

VALIDATION (selection metric: unscaled val RMSE, m; epoch 0 = pretrained DA3METRIC-LARGE)
                          recipe  epochs_run  epochs_planned  early_stopped  val_rmse_pretrained  val_rmse_best  best_epoch  val_rmse_final  gain_vs_pretrained_m  val_d1_at_best  val_loss_at_best
                A1_gt_pose_photo          48              50           True               3.1397         2.5421          33          2.5473                0.5976          0.9753            0.0860
A3_gt_pose_photo_smooth_temporal          42              50           True               3.1125         2.5035          27          2.5084                0.6091          0.9741            0.0882
           B1_gt_pose_rigid_flow          11              50           True               3.1397         2.8469           1         10.3165                0.2928          0.9289            0.1378
                   C1_rigid_flow          11              50           True               3.1398         2.9888           1   

## 13 · Test Evaluation (Eigen split, best-validation checkpoint, evaluated once)

Metrics are the source's `compute_all_scales` under four regimes. **Unscaled is the primary result.** Median-, mean- and affine-aligned numbers use per-image ground-truth statistics to rescale the prediction; they are diagnostics of *shape* quality and must not be used to select or headline a recipe. Pure predicted-pose SSL (group C) gives no metric-scale guarantee by itself — the unscaled columns show whether scale survived.

In [25]:
SCALE_VARIANTS = ["unscaled", "median", "mean", "affine"]
METRIC_COLS    = ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log", "log10", "delta1", "delta2", "delta3", "mean_error"]

long_rows = []
for _, r in final_df.iterrows():
    for sv in SCALE_VARIANTS:
        long_rows.append({"recipe": r["recipe"], "alignment": sv,
                          **{m: r.get(f"{sv}_{m}", float("nan")) for m in METRIC_COLS}})
test_long = pd.DataFrame(long_rows)
test_long.to_csv(os.path.join(ANALYSIS_DIR, "test_metrics_by_alignment.csv"), index=False)

print("=" * 100)
print("PRIMARY — UNSCALED test metrics (pixel-weighted RMSE in m)")
print("=" * 100)
print(test_long[test_long["alignment"] == "unscaled"].drop(columns="alignment").to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\n" + "=" * 100)
print("DIAGNOSTIC ONLY — scale-aligned (uses GT statistics; do not select / headline on these)")
print("=" * 100)
for sv in ["median", "mean", "affine"]:
    print(f"\n[{sv}-aligned]")
    print(test_long[test_long["alignment"] == sv].drop(columns="alignment")[["recipe", "rmse", "abs_rel", "delta1", "mean_error"]]
          .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

PRIMARY — UNSCALED test metrics (pixel-weighted RMSE in m)
                          recipe    rmse    mae  abs_rel  sq_rel  rmse_log  log10  delta1  delta2  delta3  mean_error
                A1_gt_pose_photo  2.6011 1.1228   0.0584  0.2375    0.0875 0.0249  0.9746  0.9957  0.9984     -0.0739
A3_gt_pose_photo_smooth_temporal  2.5266 1.0968   0.0575  0.2224    0.0871 0.0248  0.9742  0.9956  0.9984      0.1116
           B1_gt_pose_rigid_flow  2.8117 1.4413   0.0953  0.3218    0.1212 0.0390  0.9356  0.9936  0.9980     -0.5542
                   C1_rigid_flow  2.9990 1.4145   0.0787  0.3442    0.1103 0.0327  0.9492  0.9923  0.9976     -0.4784
         C3_photo_l1_flow_smooth 11.6096 8.5275   0.5041  4.7387    0.7260 0.3096  0.0026  0.0237  0.4103      8.5241

DIAGNOSTIC ONLY — scale-aligned (uses GT statistics; do not select / headline on these)

[median-aligned]
                          recipe   rmse  abs_rel  delta1  mean_error
                A1_gt_pose_photo 2.5968   0.0549  0.9764 

## 14 · Final Comparison Table

One row per recipe, in fixed recipe order (**not** sorted by test error — the test set is for reporting, not for choosing a recipe; `val_rank` orders by the *selection* metric). `pose_source` makes explicit which recipes are **OXTS-pose SSL** (A/B: OXTS supplies camera motion only) and which are **predicted-pose pure SSL** (C). `uses_gt_depth` is `False` for all six.

In [26]:
ft = pd.DataFrame({
    "recipe":                  final_df["recipe"],
    "recipe_description":      final_df["recipe_description"],
    "best_epoch":              final_df["best_epoch"],
    "best_val_rmse":           final_df["best_val_rmse"],
    "final_val_rmse":          final_df["final_val_rmse"],
    "test_unscaled_rmse":      final_df["unscaled_rmse"],
    "test_unscaled_abs_rel":   final_df["unscaled_abs_rel"],
    "test_unscaled_delta1":    final_df["unscaled_delta1"],
    "test_unscaled_delta2":    final_df["unscaled_delta2"],
    "test_unscaled_delta3":    final_df["unscaled_delta3"],
    "test_unscaled_mean_error": final_df["unscaled_mean_error"],
    "training_time":           final_df["training_time_s"].map(lambda s: f"{s/3600:.2f} h" if pd.notna(s) else "n/a"),
    "number_of_updates":       final_df["number_of_updates"],
    "skipped_updates":         final_df["skipped_updates"],
    "best_checkpoint":         final_df["best_checkpoint"],
    "pose_source":             final_df["pose_source"],
    "uses_gt_depth":           final_df["uses_gt_depth"],
    "architecture":            final_df["architecture"],
    "learning_rate":           final_df["learning_rate"],
    "augmentation":            final_df["augmentation"],
    "optimizer":               final_df["optimizer"],
    "validated_in_ablation":   final_df["validated_in_ablation"],
})
ft.insert(3, "val_rank", ft["best_val_rmse"].rank(method="min").astype(int))
assert not ft["uses_gt_depth"].any(), "a final recipe reports uses_gt_depth=True"
ft.to_csv(os.path.join(ANALYSIS_DIR, "final_comparison.csv"), index=False)
try:
    (Path(ANALYSIS_DIR) / "final_comparison.md").write_text(ft.to_markdown(index=False, floatfmt=".4f"))
except Exception:
    pass

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); pd.set_option("display.max_colwidth", 70)
print("PERFORMANCE / SELECTION")
print(ft[["recipe", "best_epoch", "val_rank", "best_val_rmse", "final_val_rmse", "test_unscaled_rmse", "test_unscaled_abs_rel",
          "test_unscaled_delta1", "test_unscaled_delta2", "test_unscaled_delta3", "test_unscaled_mean_error",
          "training_time", "number_of_updates", "skipped_updates"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nPROVENANCE  (A/B = OXTS-pose SSL, C = predicted-pose pure SSL; none uses GT depth)")
print(ft[["recipe", "pose_source", "uses_gt_depth", "architecture", "learning_rate", "augmentation", "optimizer",
          "validated_in_ablation"]].to_string(index=False))
print("\nBEST CHECKPOINTS")
for _, r in ft.iterrows():
    print(f"  {r['recipe']:34s} {r['best_checkpoint']}")

PERFORMANCE / SELECTION
                          recipe  best_epoch  val_rank  best_val_rmse  final_val_rmse  test_unscaled_rmse  test_unscaled_abs_rel  test_unscaled_delta1  test_unscaled_delta2  test_unscaled_delta3  test_unscaled_mean_error training_time  number_of_updates  skipped_updates
                A1_gt_pose_photo          33         2         2.5421          2.5473              2.6011                 0.0584                0.9746                0.9957                0.9984                   -0.0739       46.21 h              31200                0
A3_gt_pose_photo_smooth_temporal          27         1         2.5035          2.5084              2.5266                 0.0575                0.9742                0.9956                0.9984                    0.1116       78.75 h              27300                0
           B1_gt_pose_rigid_flow           1         3         2.8469         10.3165              2.8117                 0.0953                0.9356             

## 15 · Plots / Diagnostics

Reused from the source's plotting cell (same helpers, `_save`, palette; only the titles / colour keys changed): RMSE under all four alignments, unscaled error metrics, δ-accuracy, signed bias. New but in the same style: **per-recipe training curves** (train loss, validation loss, validation RMSE with the pretrained baseline and best epoch, scheduled learning rates) and **gradient / update diagnostics** (depth & PoseNet gradient norms, skipped-update fraction, applied updates per epoch), plus one overlay of validation RMSE across recipes.

In [27]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}

def _save(fig: plt.Figure, name: str, dpi: int = 140):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    log.info("  v %s", name)

# recipe-id prefixes (replaces the ablation-group prefixes of the source palette)
GROUP_COLORS = {
    "A1_": "#e67e22", "A3_": "#d35400",      # OXTS-pose SSL (photometric family)
    "B1_": "#f1c40f",                        # OXTS-pose SSL (rigid flow)
    "C1_": "#2ecc71", "C3_": "#16a085", "C5_": "#3498db",   # predicted-pose pure SSL
}

def _grp_color(label: str) -> str:
    for prefix, col in GROUP_COLORS.items():
        if label.startswith(prefix):
            return col
    return "#7f8c8d"


# ════════════════════════════════════════════════════════════════════════════
# Plot 01 — RMSE Comparison: all 4 scale variants, all configs
# ════════════════════════════════════════════════════════════════════════════
def plot_rmse_all_scales(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        vals = [df.loc[df["config"] == c, f"{sv}_rmse"].values[0]
                if f"{sv}_rmse" in df.columns else float("nan")
                for c in configs]
        ax.bar(x + (i - 1.5) * w, vals, width=w, label=sv, color=col, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Pixel-Weighted True RMSE (m)"); ax.set_xlabel("Configuration")
    ax.set_title("RMSE Across All Scale Variants — Final SSL Fine-Tuning (unscaled = primary)")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    for prefix in ["arch_", "aug_", "optim_", "combo_", "weight_"]:
        idxs = [i for i, c in enumerate(configs) if c.startswith(prefix)]
        if idxs:
            ax.axvline(x=min(idxs) - 0.5, color="grey", linestyle="--", alpha=0.5)
    _save(fig, "01_rmse_all_scales.png")


# ════════════════════════════════════════════════════════════════════════════
# Plot 02 — AbsRel / SqRel / RMSE_log / log10 (unscaled)
# ════════════════════════════════════════════════════════════════════════════
def plot_error_metrics(df: pd.DataFrame):
    metrics = ["unscaled_abs_rel", "unscaled_sq_rel", "unscaled_rmse_log", "unscaled_log10"]
    labels  = ["AbsRel", "SqRel", "RMSE_log", "log10"]
    configs = df["config"].tolist()
    x = np.arange(len(configs))
    fig, axes = plt.subplots(2, 2, figsize=(max(14, len(configs)), 10))
    axes = axes.flatten()
    for ax, metric, lbl in zip(axes, metrics, labels):
        vals = [df.loc[df["config"] == c, metric].values[0]
                if metric in df.columns else float("nan") for c in configs]
        colors = [_grp_color(c) for c in configs]
        bars = ax.bar(x, vals, color=colors, alpha=0.85,
                       edgecolor="white", linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                              ha="right", fontsize=6)
        ax.set_ylabel(lbl + " (unscaled)"); ax.set_title(lbl)
        ax.grid(axis="y", alpha=0.4)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=5)
    fig.suptitle("Error Metrics (Unscaled) — Final SSL Fine-Tuning",
                 fontsize=13, fontweight="bold")
    _save(fig, "02_error_metrics_unscaled.png")


# ════════════════════════════════════════════════════════════════════════════
# Plot 03 — Delta accuracy
# ════════════════════════════════════════════════════════════════════════════
def plot_delta_accuracy(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.25
    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    delta_info = [
        ("unscaled_delta1", "d<1.25",   "#e74c3c"),
        ("unscaled_delta2", "d<1.25^2", "#27ae60"),
        ("unscaled_delta3", "d<1.25^3", "#2980b9"),
    ]
    for i, (col, lbl, c) in enumerate(delta_info):
        vals = [df.loc[df["config"] == cf, col].values[0]
                if col in df.columns else float("nan") for cf in configs]
        ax.bar(x + (i-1)*w, vals, width=w, label=lbl, color=c, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                          ha="right", fontsize=7)
    ax.set_ylabel("Accuracy (higher = better)"); ax.set_xlabel("Configuration")
    ax.set_title("Threshold Accuracy (Unscaled) — Final SSL Fine-Tuning")
    ax.legend(); ax.grid(axis="y", alpha=0.4)
    ax.set_ylim([0, 1.05])
    _save(fig, "03_delta_accuracy.png")


# ════════════════════════════════════════════════════════════════════════════
# Plot 04 — Signed bias
# ════════════════════════════════════════════════════════════════════════════
def plot_bias(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(14, len(configs)), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        mc = f"{sv}_mean_error"
        vals = [df.loc[df["config"] == c, mc].values[0]
                if mc in df.columns else float("nan") for c in configs]
        ax.bar(x + (i-1.5)*w, vals, width=w, label=sv, color=col, alpha=0.8)
    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                          ha="right", fontsize=7)
    ax.set_ylabel("Mean Signed Error (LiDAR - Pred), m")
    ax.set_title("Depth Bias Across Scale Variants — Final SSL Fine-Tuning")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    _save(fig, "04_bias_signed.png")

In [28]:
# ── NEW: per-recipe training curves + gradient / update diagnostics (same style as the source's `plot_learning_curves`) ──
def plot_final_training_curves(rid: str) -> None:
    h = pd.read_csv(os.path.join(RESULTS_DIR, f"history_{rid}.csv"))
    tr = h[h["epoch"] > 0]
    base = h.loc[h["epoch"] == 0, "val_rmse"]
    color = _grp_color(rid)
    best_ep = int(tr.loc[tr["val_rmse"].idxmin(), "epoch"])

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    ax = axes[0, 0]; ax.plot(tr["epoch"], tr["train_loss"], color=color, linewidth=1.6)
    ax.set_title("Train loss (recipe objective)"); ax.set_xlabel("Epoch"); ax.set_ylabel("loss")
    ax = axes[0, 1]
    if "val_val_loss" in tr:
        ax.plot(tr["epoch"], tr["val_val_loss"], color=color, linewidth=1.6)
    ax.set_title("Validation loss (same objective)"); ax.set_xlabel("Epoch"); ax.set_ylabel("loss")
    ax = axes[1, 0]; ax.plot(tr["epoch"], tr["val_rmse"], color=color, linewidth=1.6, label="val RMSE")
    if len(base):
        ax.axhline(float(base.iloc[0]), color="grey", linestyle="--", label="pretrained DA3 (epoch 0)")
    ax.axvline(best_ep, color="black", linestyle=":", label=f"best epoch {best_ep}")
    ax.set_title("Validation RMSE (unscaled, m)"); ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE (m)"); ax.legend(fontsize=8)
    ax = axes[1, 1]
    if "lr_depth_sched" in tr:
        ax.plot(tr["epoch"], tr["lr_depth_sched"], color=color, linewidth=1.6, label="depth LR (end of epoch)")
        if tr["lr_pose_sched"].notna().any():
            ax.plot(tr["epoch"], tr["lr_pose_sched"], color="#8e44ad", linewidth=1.4, linestyle="--", label="PoseNet LR (end of epoch)")
        ax.set_yscale("log"); ax.legend(fontsize=8)
    ax.set_title("Scheduled learning rate (cosine warm-up, per optimizer update)"); ax.set_xlabel("Epoch"); ax.set_ylabel("LR")
    for a in axes.flat:
        a.grid(alpha=0.4)
    fig.suptitle(f"Training curves — {rid}", fontsize=13, fontweight="bold")
    _save(fig, f"train_curves_{rid}.png")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    ax = axes[0]
    ax.plot(tr["epoch"], tr["train_diag_depth_grad_norm"], color=color, label="|g_depth|")
    if tr["train_diag_pose_grad_norm"].notna().any():
        ax.plot(tr["epoch"], tr["train_diag_pose_grad_norm"], color="#8e44ad", linestyle="--", label="|g_pose|")
    ax.set_yscale("log"); ax.set_title("Mean gradient norm per applied update (pre-clip)"); ax.legend(fontsize=8)
    axes[1].bar(tr["epoch"], 100 * tr["train_diag_skip_frac"], color="#c0392b"); axes[1].set_title("Skipped (non-finite) updates, %")
    axes[2].bar(tr["epoch"], tr["train_diag_updates_applied"], color=color); axes[2].set_title("Optimizer updates applied per epoch")
    for a in axes:
        a.set_xlabel("Epoch"); a.grid(alpha=0.4)
    fig.suptitle(f"Gradient / update diagnostics — {rid}", fontsize=12, fontweight="bold")
    _save(fig, f"train_diag_{rid}.png")


def plot_val_rmse_overlay(recipe_ids) -> None:
    fig, ax = plt.subplots(figsize=(11, 6)); base = None
    for rid in recipe_ids:
        h = pd.read_csv(os.path.join(RESULTS_DIR, f"history_{rid}.csv")); tr = h[h["epoch"] > 0]
        ax.plot(tr["epoch"], tr["val_rmse"], color=_grp_color(rid), linewidth=1.6, label=rid)
        if base is None and (h["epoch"] == 0).any():
            base = float(h.loc[h["epoch"] == 0, "val_rmse"].iloc[0])
    if base is not None:
        ax.axhline(base, color="grey", linestyle="--", label="pretrained DA3METRIC-LARGE")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation RMSE (m, unscaled)"); ax.grid(alpha=0.4); ax.legend(fontsize=8)
    ax.set_title("Validation RMSE by epoch — final recipes")
    _save(fig, "val_rmse_overlay.png")


for _rid in final_df["recipe"]:
    plot_final_training_curves(_rid)
plot_val_rmse_overlay(list(final_df["recipe"]))
plot_rmse_all_scales(final_df); plot_error_metrics(final_df); plot_delta_accuracy(final_df); plot_bias(final_df)
print(f"\nAll plots saved to {PLOT_DIR}")

07:33:35  INFO        v train_curves_A1_gt_pose_photo.png
07:33:36  INFO        v train_diag_A1_gt_pose_photo.png
07:33:36  INFO        v train_curves_A3_gt_pose_photo_smooth_temporal.png
07:33:37  INFO        v train_diag_A3_gt_pose_photo_smooth_temporal.png
07:33:37  INFO        v train_curves_B1_gt_pose_rigid_flow.png
07:33:38  INFO        v train_diag_B1_gt_pose_rigid_flow.png
07:33:38  INFO        v train_curves_C1_rigid_flow.png
07:33:39  INFO        v train_diag_C1_rigid_flow.png
07:33:40  INFO        v train_curves_C3_photo_l1_flow_smooth.png
07:33:40  INFO        v train_diag_C3_photo_l1_flow_smooth.png
07:33:40  INFO        v val_rmse_overlay.png
07:33:40  INFO        v 01_rmse_all_scales.png
07:33:41  INFO        v 02_error_metrics_unscaled.png
07:33:41  INFO        v 03_delta_accuracy.png
07:33:41  INFO        v 04_bias_signed.png



All plots saved to /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/final_self_supervised_FT/plots


## 16 · Final Results Summary

In [29]:
print("FINAL SELF-SUPERVISED FINE-TUNING — RESULTS (all numbers are read from the CSVs written by THIS run)")
print("-" * 95)
for _, r in ft.iterrows():
    kind = "OXTS-pose SSL       " if r["pose_source"].startswith("OXTS") else "predicted-pose SSL  "
    print(f"{r['recipe']:34s} [{kind}] best ep {int(r['best_epoch']):3d} | val RMSE {r['best_val_rmse']:.4f} | "
          f"TEST unscaled RMSE {r['test_unscaled_rmse']:.4f}  AbsRel {r['test_unscaled_abs_rel']:.4f}  d1 {r['test_unscaled_delta1']:.4f}  "
          f"bias {r['test_unscaled_mean_error']:+.4f}")
print("\nNo recipe used LiDAR / GT depth in any loss (uses_gt_depth=False for all rows).")
print("Reported / compared on UNSCALED metrics; aligned metrics are diagnostics only.")
print("Artifacts:", RESULTS_DIR, "|", ANALYSIS_DIR, "|", PLOT_DIR, "|", CHECKPOINT_DIR)

FINAL SELF-SUPERVISED FINE-TUNING — RESULTS (all numbers are read from the CSVs written by THIS run)
-----------------------------------------------------------------------------------------------
A1_gt_pose_photo                   [OXTS-pose SSL       ] best ep  33 | val RMSE 2.5421 | TEST unscaled RMSE 2.6011  AbsRel 0.0584  d1 0.9746  bias -0.0739
A3_gt_pose_photo_smooth_temporal   [OXTS-pose SSL       ] best ep  27 | val RMSE 2.5035 | TEST unscaled RMSE 2.5266  AbsRel 0.0575  d1 0.9742  bias +0.1116
B1_gt_pose_rigid_flow              [OXTS-pose SSL       ] best ep   1 | val RMSE 2.8469 | TEST unscaled RMSE 2.8117  AbsRel 0.0953  d1 0.9356  bias -0.5542
C1_rigid_flow                      [predicted-pose SSL  ] best ep   1 | val RMSE 2.9888 | TEST unscaled RMSE 2.9990  AbsRel 0.0787  d1 0.9492  bias -0.4784
C3_photo_l1_flow_smooth            [predicted-pose SSL  ] best ep   1 | val RMSE 11.3948 | TEST unscaled RMSE 11.6096  AbsRel 0.5041  d1 0.0026  bias +8.5241

No recipe used LiDAR

---
## Summary of this notebook  *(Deliverable 2 — configuration and provenance, no results)*

**Six recipes implemented** — each an independent fine-tuning of the same pretrained DA3METRIC-LARGE:

| Recipe | Pose source | Loss |
|---|---|---|
| `A1_gt_pose_photo` | OXTS | `1.0·GT_Pose_Photo_Reproj` |
| `A3_gt_pose_photo_smooth_temporal` | OXTS | `1.0·GT_Pose_Photo_Reproj + 1e-3·Edge_Aware_Smooth + 0.1·GT_Pose_Temporal_Warp_Consistency` |
| `B1_gt_pose_rigid_flow` | OXTS | `1.0·GT_Pose_Rigid_Flow` |
| `C1_rigid_flow` | PoseNet | `1.0·Rigid_Flow_Consistency` |
| `C3_photo_l1_flow_smooth` | PoseNet | `1.0·Photo_L1 + 0.1·Flow_Smooth` |
| `C5_photo_l1_feature_metric` | PoseNet | `1.0·Photo_L1 + 0.1·Feature_Metric` |

**Directly validated in the ablation (per the project's designation): A1** (Group 5b H), **B1** (Group 5b K), **C1** (Group 1a `rigid_flow_consistency`).
**New combinations derived from the ablation findings, *not* directly validated: A3, C3, C5.** (A2 = A1 + smoothness was evaluated in the ablation and intentionally not carried forward.)

**Fixed final configuration:** DA3METRIC-LARGE · `decoder_only` · AdamW · cosine warm-up (per optimizer update) · depth LR `1e-6` · weight decay `1e-4` · `base` augmentation · `FINAL_EPOCHS = 50` (configurable) · batch 4 × accumulation 4 = 16 · early-stop patience 10 · `MAX_TRAIN = 0` (all canonical samples) · fp32 geometry / loss maths · PoseNet LR `1e-4` with a depth-frozen warm-up (C only). No HPT, architecture, augmentation, optimizer, weight or stationary-object sweeps.

**No LiDAR depth supervision anywhere in training:** `uses_gt_depth=False` is asserted for all six recipes, `w_anchor = 0`, the recipes contain no `anchor` term (A3 does **not** inherit Group 5b J's `0.5·SILog` anchor), training loaders are built with `load_gt=False`, and the losses receive `gt=None, mask=None`. Sparse LiDAR appears only in *evaluation*: validation RMSE (checkpoint selection / early stopping) and the test metrics — plus the pre-flight pose-convention diagnostic 0a.

**OXTS pose is used only by A1 / A3 / B1**, converted to the camera frame by `T_cam = T_ic·inv(W_j)·W_i·inv(T_ic)` and mirrored on horizontal flips; C1 / C3 / C5 use the PoseNet pipeline and never see OXTS.

*Numbers reported by this notebook come only from the run itself.*